In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
!pip install -q openai

In [11]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
OPENROUTER_API_KEY = user_secrets.get_secret("openrouter")

In [12]:
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

In [13]:
response = client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Explain in one sentence what a batch job is."
        }
    ]
)

print(response.choices[0].message.content)

User Safety: safe


In [1]:
# Paste this whole file into ONE Kaggle notebook cell and run it.
# Uses only pandas + openpyxl, both preinstalled on Kaggle.
import glob
import pandas as pd

# ---- 1. Upload B11HRO and B11IHO as a (private) Kaggle dataset, then set the paths ----
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    print(p)                                   # <- copy your two paths from this list
HRO = "/kaggle/input/datasets/aburvan/report/B11HR.xlsx"  # .xlsx or ';'-delimited .txt/.csv
IHO = "/kaggle/input/datasets/aburvan/report/B11IH.xlsx"
REPORT = "/kaggle/working/validation_report.xlsx"


# ---- 2. Loading (columns by position, header row optional) ----
def load(path):
    if path.lower().endswith((".xlsx", ".xlsm", ".xls")):
        df = pd.read_excel(path, header=None, dtype=str)
    else:
        df = pd.read_csv(path, sep=";", header=None, dtype=str, engine="python")
    df = df.dropna(how="all")
    if df.dropna(axis=1, how="all").shape[1] == 1:      # whole line pasted into column A
        df = df.iloc[:, 0].str.split(";", expand=True)
    return df.fillna("").astype(str).apply(lambda c: c.str.strip())


def bestnr(physical):
    """STAP.BP572.V01.%OENV -> BP572 ; MPLA.%AENVXV03.%DENV001... -> BXV03"""
    parts = physical.split(".")
    if parts[0] == "STAP" and len(parts) > 1:
        return parts[1]
    if len(parts) > 1 and "%AENV" in parts[1]:
        return "B" + parts[1].split("%AENV")[1]
    return ""


hro = load(HRO).iloc[:, :8]
hro.columns = ["net", "job", "raw_logical", "physical", "io", "dep_cycle", "dep_job", "program"]
hro["row_id"] = hro.index + 1                            # row number in your file = citation
hro = hro[~hro["net"].str.upper().str.startswith("NET/CYCLE") & (hro["raw_logical"] != "")].copy()
hro["logical"] = hro["raw_logical"].str.replace(r"[IO]$", "", regex=True)   # BP569O -> BP569
hro["bestnr"] = hro["physical"].map(bestnr)
hro["external"] = hro["dep_cycle"] != hro["net"]         # dependent sits in a different cycle
hro["access"] = hro["io"].map({"I": "reads", "O": "writes"})

iho = load(IHO).iloc[:, :3]
iho.columns = ["logfile", "bestnr_iho", "decom"]
iho = iho[iho["logfile"].str.upper() != "LOGFILE"].copy()
iho["decom"] = iho["decom"].str.upper()

# ---- 3. Candidate rule: NO if ANY dependent job is in a different cycle, else YES ----
ext = hro[hro["external"]]
engine = hro.groupby("logical").agg(
    engine_decision=("external", lambda s: "NO" if s.any() else "YES"),
    bestnr_derived=("bestnr", lambda s: ", ".join(sorted(set(s)))),
)
blockers = (ext.assign(b=ext["dep_job"] + " (" + ext["dep_cycle"] + ")")
            .groupby("logical")["b"].agg(lambda s: ", ".join(sorted(set(s)))))

res = iho.merge(engine, left_on="logfile", right_index=True, how="left")
res["engine_decision"] = res["engine_decision"].fillna("MISSING")
res["bestnr_derived"] = res["bestnr_derived"].fillna("")
res["blocking_dependents"] = res["logfile"].map(blockers).fillna("")
res["match"] = (res["engine_decision"] == res["decom"]) & res.apply(
    lambda r: r["bestnr_iho"] in [x.strip() for x in r["bestnr_derived"].split(",")], axis=1)
res["match"] = res["match"].map({True: "OK", False: "MISMATCH"})

print(f"\n{(res['match'] == 'OK').sum()}/{len(res)} match")
if (res["match"] != "OK").any():
    print(res[res["match"] != "OK"].to_string(index=False))

# ---- 4. Excel report: Validation + Evidence sheets ----
evidence = ext[["logical", "physical", "dep_job", "access", "dep_cycle", "row_id"]]
evidence.columns = ["Logical file", "Physical file", "Dependent job", "Access", "Cycle", "Input row"]
with pd.ExcelWriter(REPORT, engine="openpyxl") as xw:
    res.to_excel(xw, sheet_name="Validation", index=False)
    evidence.to_excel(xw, sheet_name="Evidence", index=False)
print("report written:", REPORT, "(download it from the Output tab)")
res

/kaggle/input/datasets
/kaggle/input/datasets/aburvan
/kaggle/input/datasets/aburvan/report
/kaggle/input/datasets/aburvan/report1
/kaggle/input/datasets/aburvan/report/B11HR.xlsx
/kaggle/input/datasets/aburvan/report/B11IH.xlsx
/kaggle/input/datasets/aburvan/report1/B11HR.xlsx
/kaggle/input/datasets/aburvan/report1/B11IH.xlsx

37/37 match
report written: /kaggle/working/validation_report.xlsx (download it from the Output tab)


,logfile,bestnr_iho,decom,engine_decision,bestnr_derived,blocking_dependents,match
1,BHW85,BHW85,YES,YES,BHW85,,OK
2,BP572,BP572,YES,YES,BP572,,OK
3,BP888,BP888,YES,YES,BP888,,OK
4,BWIH5,BWIH5,YES,YES,BWIH5,,OK
5,BWIH7,BWIH7,YES,YES,BWIH7,,OK
6,BWIH8,BWIH8,YES,YES,BWIH8,,OK
7,BWIH9,BWIH9,YES,YES,BWIH9,,OK
8,BWIIA,BWIIA,YES,YES,BWIIA,,OK
9,BWIIB,BWIIB,YES,YES,BWIIB,,OK
10,BY4JE,BY4JE,NO,NO,BY4JE,"TD160551 (TD1605), TD160599 (TD1605)",OK


In [2]:
# ============================================================================
# Mainframe decommission assistant - single Kaggle notebook, cell by cell.
# Paste each ### CELL N block into its own cell, in order. pandas + openpyxl
# are preinstalled on Kaggle; only the LLM cell needs an internet-enabled
# notebook + a Gemini API key in Kaggle Secrets (Add-ons > Secrets).
# ============================================================================

### CELL 1 - locate your files -------------------------------------------
import glob
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    print(p)                                     # <- copy paths for CELL 2

### CELL 2 - load B11HRO and the two-section B11IHO -----------------------
import pandas as pd

HRO = "/kaggle/input/datasets/aburvan/report1/B11HR.xlsx"     # .xlsx or ';'-text, either works
IHO = "/kaggle/input/datasets/aburvan/report1/B11IH.xlsx"      # now has files, blank line, programs


def _load_raw(path):
    """Read into a DataFrame of raw string rows, one row per input record.
    Keeps blank lines (as a row of ''), and tolerates different column
    counts per line (needed for a two-section B11IHO: 3 cols, blank, 5 cols)."""
    if path.lower().endswith((".xlsx", ".xlsm", ".xls")):
        df = pd.read_excel(path, header=None, dtype=str)
        df = df.fillna("")
    else:
        with open(path, encoding="utf-8", errors="replace") as f:
            lines = [ln.rstrip("\n").rstrip("\r") for ln in f]
        rows = [ln.split(";") for ln in lines]
        width = max((len(r) for r in rows), default=1)
        rows = [r + [""] * (width - len(r)) for r in rows]
        df = pd.DataFrame(rows)
    if df.shape[1] == 1:                           # whole line pasted into column A
        df = df[0].str.split(";", expand=True).fillna("")
    return df.apply(lambda c: c.str.strip())


def load_hro(path):
    df = _load_raw(path).iloc[:, :8]
    df.columns = ["net", "job", "raw_logical", "physical", "io",
                  "dep_cycle", "dep_job", "program"]
    df["row_id"] = df.index + 1
    df = df[~df["net"].str.upper().str.startswith("NET/CYCLE")
            & (df["raw_logical"] != "")].copy()
    df["logical"] = df["raw_logical"].str.replace(r"[IO]$", "", regex=True)
    df["external"] = df["dep_cycle"] != df["net"]
    df["access"] = df["io"].map({"I": "reads", "O": "writes"})
    return df


def load_iho_sections(path):
    """Split B11IHO on its blank separator line into (files_df, programs_df).
    Falls back to files-only if no blank line / second header is present,
    so this also reads an OLD-STYLE (files-only) B11IHO without changes."""
    raw = _load_raw(path)
    blank_rows = raw[(raw == "").all(axis=1)].index.tolist()
    if blank_rows:
        cut = blank_rows[0]
        files_raw, prog_raw = raw.iloc[:cut], raw.iloc[cut + 1:]
    else:
        files_raw, prog_raw = raw, raw.iloc[0:0]

    files = files_raw.iloc[:, :3].copy()
    files.columns = ["logfile", "bestnr", "decom"]
    files = files[files["logfile"].str.upper() != "LOGFILE"].copy()
    files["decom"] = files["decom"].str.upper()

    if len(prog_raw):
        programs = prog_raw.iloc[:, :5].copy()
        programs.columns = ["job", "program", "files", "blocking_files", "decom"]
        programs = programs[programs["job"].str.upper() != "JOB"].copy()
        programs["decom"] = programs["decom"].str.upper()
    else:
        programs = pd.DataFrame(columns=["job", "program", "files", "blocking_files", "decom"])
    return files, programs


hro = load_hro(HRO)
iho_files, iho_programs = load_iho_sections(IHO)
print(f"HRO rows: {len(hro)}   IHO files: {len(iho_files)}   IHO programs: {len(iho_programs)}")

### CELL 3 - validate both sections against the candidate rule -------------
def bestnr(physical):
    parts = physical.split(".")
    if parts[0] == "STAP" and len(parts) > 1:
        return parts[1]
    if len(parts) > 1 and "%AENV" in parts[1]:
        return "B" + parts[1].split("%AENV")[1]
    return ""


hro["bestnr"] = hro["physical"].map(bestnr)
ext = hro[hro["external"]]

# -- file-level engine (same rule validated at 37/37 earlier) --
file_engine = hro.groupby("logical").agg(
    engine_decision=("external", lambda s: "NO" if s.any() else "YES"),
    bestnr_derived=("bestnr", lambda s: ", ".join(sorted(set(s)))))
file_blockers = (ext.assign(b=ext["dep_job"] + " (" + ext["dep_cycle"] + ")")
                  .groupby("logical")["b"].agg(lambda s: ", ".join(sorted(set(s)))))

file_res = iho_files.merge(file_engine, left_on="logfile", right_index=True, how="left")
file_res["engine_decision"] = file_res["engine_decision"].fillna("MISSING")
file_res["blocking_dependents"] = file_res["logfile"].map(file_blockers).fillna("")
file_res["match"] = (file_res["engine_decision"] == file_res["decom"]) & file_res.apply(
    lambda r: r["bestnr"] in [x.strip() for x in str(r["bestnr_derived"] or "").split(",")],
    axis=1)

print(f"\nFILES:    {(file_res['match']).sum()}/{len(file_res)} match")
if (~file_res["match"]).any():
    print(file_res[~file_res["match"]].to_string(index=False))

# -- program-level engine: YES iff every file of the (job,program) step is YES --
steps = hro[["job", "program", "logical"]].drop_duplicates().copy()
steps["file_decom"] = steps["logical"].map(iho_files.set_index("logfile")["decom"])
prog_blockers = (steps[steps["file_decom"] == "NO"].groupby(["job", "program"])["logical"]
                  .agg(lambda s: ", ".join(sorted(s))))
prog_engine = (steps.groupby(["job", "program"])
               .agg(files=("logical", "nunique"),
                    unknown=("file_decom", lambda s: s.isna().any()))
               .join(prog_blockers.rename("blocking")).reset_index())
prog_engine["blocking"] = prog_engine["blocking"].fillna("")
prog_engine["engine_decision"] = "YES"
prog_engine.loc[prog_engine["unknown"], "engine_decision"] = "UNKNOWN"
prog_engine.loc[prog_engine["blocking"] != "", "engine_decision"] = "NO"

if len(iho_programs):
    prog_res = iho_programs.merge(prog_engine, on=["job", "program"], how="left",
                                   suffixes=("", "_calc"))
    prog_res["engine_decision"] = prog_res["engine_decision"].fillna("MISSING")
    prog_res["match"] = prog_res["decom"] == prog_res["engine_decision"]
    print(f"\nPROGRAMS: {(prog_res['match']).sum()}/{len(prog_res)} match")
    if (~prog_res["match"]).any():
        print(prog_res[~prog_res["match"]].to_string(index=False))
else:
    print("\nPROGRAMS: none found in this B11IHO (old-style file, or no program section)")


### CELL 4 - the LLM's tools (plain functions over the data above) --------
def get_decision(logical_file: str) -> dict:
    """Y/N decision and evidence for one logical file."""
    row = iho_files[iho_files["logfile"] == logical_file]
    if row.empty:
        return {"error": f"{logical_file} not found in B11IHO"}
    blockers = ext[ext["logical"] == logical_file]
    return {
        "logical_file": logical_file,
        "decom": row.iloc[0]["decom"],
        "blocking_dependents": [
            {"job": r.dep_job, "cycle": r.dep_cycle, "access": r.access, "row_id": r.row_id}
            for r in blockers.itertuples()],
    }


def get_program_decision(job: str, program: str) -> dict:
    """Y/N decision and blocking files for one job/program step."""
    row = iho_programs[(iho_programs["job"] == job) & (iho_programs["program"] == program)]
    if row.empty:
        return {"error": f"{job}/{program} not found in B11IHO"}
    r = row.iloc[0]
    return {"job": job, "program": program, "files": int(r["files"]),
            "blocking_files": [f for f in r["blocking_files"].split(" ") if f],
            "decom": r["decom"]}


def get_cycle_summary(cycle: str) -> dict:
    """Overview of one net/cycle: how many files/programs are YES vs NO."""
    files = iho_files  # single-net data; filter here if you load multiple nets
    programs = iho_programs
    return {
        "cycle": cycle,
        "files_total": len(files), "files_no": int((files["decom"] == "NO").sum()),
        "programs_total": len(programs), "programs_no": int((programs["decom"] == "NO").sum()),
        "blocked_files": files[files["decom"] == "NO"]["logfile"].tolist(),
        "blocked_programs": (programs.loc[programs["decom"] == "NO", ["job", "program"]]
                             .apply(lambda r: f"{r.job}/{r.program}", axis=1).tolist()
                             if len(programs) else []),
    }


TOOLS = {
    "get_decision": get_decision,
    "get_program_decision": get_program_decision,
    "get_cycle_summary": get_cycle_summary,
}

print(get_decision("BP569"))
print(get_program_decision("TS660420", "S912"))
print(get_cycle_summary("TS6604"))


### CELL 5 - Gemini chat loop with function calling ------------------------
# pip install -q -U google-genai   # uncomment on Kaggle if the package is missing
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types

api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")   # set this in Add-ons > Secrets
client = genai.Client(api_key=api_key)

SYSTEM_PROMPT = (
    "You explain mainframe decommissioning decisions. Always call a tool to "
    "get the facts; never guess a decision or a job/program name. Cite the "
    "blocking jobs and cycles the tool returns. If a tool says 'not found', "
    "say the data doesn't cover it - don't invent an answer."
)

gemini_tools = [
    {"name": "get_decision", "description": "Y/N decommission decision and blocking jobs for one logical file.",
     "parameters": {"type": "object", "properties": {"logical_file": {"type": "string"}},
                    "required": ["logical_file"]}},
    {"name": "get_program_decision", "description": "Y/N decommission decision for one job/program step.",
     "parameters": {"type": "object", "properties": {"job": {"type": "string"}, "program": {"type": "string"}},
                    "required": ["job", "program"]}},
    {"name": "get_cycle_summary", "description": "Overview of a net/cycle: counts of blocked files and programs.",
     "parameters": {"type": "object", "properties": {"cycle": {"type": "string"}}, "required": ["cycle"]}},
]


def ask(question: str, history=None) -> str:
    import time
    from google.genai import errors
    history = history or []
    contents = history + [types.Content(role="user", parts=[types.Part(text=question)])]
    for _ in range(5):                                  # cap tool-call chaining
        for attempt in range(5):                        # retry on transient 503s
            try:
                resp = client.models.generate_content(
                    model="gemini-3.8-flash", contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        tools=[types.Tool(function_declarations=gemini_tools)]))
                break
            except errors.ServerError:
                if attempt == 4:
                    raise
                wait = 2 ** attempt          # 1, 2, 4, 8 seconds
                print(f"Gemini busy (503) - retrying in {wait}s...")
                time.sleep(wait)
        part = resp.candidates[0].content.parts[0]
        if part.function_call:
            fn, args = part.function_call.name, dict(part.function_call.args)
            result = TOOLS[fn](**args)
            contents.append(resp.candidates[0].content)
            contents.append(types.Content(role="user", parts=[
                types.Part(function_response=types.FunctionResponse(name=fn, response=result))]))
            continue
        return part.text
    return "(stopped after 5 tool calls)"


print(ask("Can BP569 be decommissioned?"))
print(ask("What about the TS6604 cycle overall?"))

### CELL 4.5 (optional) - list models available to your key, if 3.8-flash keeps 503ing
# for m in client.models.list():
#     print(m.name)

/kaggle/input/datasets
/kaggle/input/datasets/aburvan
/kaggle/input/datasets/aburvan/report
/kaggle/input/datasets/aburvan/report1
/kaggle/input/datasets/aburvan/report/B11HR.xlsx
/kaggle/input/datasets/aburvan/report/B11IH.xlsx
/kaggle/input/datasets/aburvan/report1/B11HR.xlsx
/kaggle/input/datasets/aburvan/report1/B11IH.xlsx
HRO rows: 449   IHO files: 37   IHO programs: 0

FILES:    37/37 match

PROGRAMS: none found in this B11IHO (old-style file, or no program section)
{'logical_file': 'BP569', 'decom': 'NO', 'blocking_dependents': [{'job': 'TD160425', 'cycle': 'TD1604', 'access': 'reads', 'row_id': 171}, {'job': 'TD160401', 'cycle': 'TD1604', 'access': 'reads', 'row_id': 172}]}
{'error': 'TS660420/S912 not found in B11IHO'}
{'cycle': 'TS6604', 'files_total': 37, 'files_no': 7, 'programs_total': 0, 'programs_no': 0, 'blocked_files': ['BY4JE', 'BY1EH', 'BY1FJ', 'BY4JF', 'VH14#1', 'BP569', 'JG0Q#1'], 'blocked_programs': []}


KeyboardInterrupt: 

In [5]:
# ============================================================================
# Mainframe decommission assistant - single Kaggle notebook, cell by cell.
# Paste each ### CELL N block into its own cell, in order. pandas + openpyxl
# are preinstalled on Kaggle; only the LLM cell needs an internet-enabled
# notebook + a Gemini API key in Kaggle Secrets (Add-ons > Secrets).
# ============================================================================

### CELL 1 - locate your files -------------------------------------------
import glob
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    print(p)                                     # <- copy paths for CELL 2

### CELL 2 - load B11HRO and the two-section B11IHO -----------------------
import pandas as pd

HRO = "/kaggle/input/datasets/aburvan/report2/B11HR.xlsx"     # .xlsx or ';'-text, either works
IHO = "/kaggle/input/datasets/aburvan/report2/B11IH.xlsx"      # now has files, blank line, programs


def _load_raw(path):
    """Read into a DataFrame of raw string rows, one row per input record.
    Keeps blank lines (as a row of ''), and tolerates different column
    counts per line (needed for a two-section B11IHO: 3 cols, blank, 5 cols)."""
    if path.lower().endswith((".xlsx", ".xlsm", ".xls")):
        df = pd.read_excel(path, header=None, dtype=str)
        df = df.fillna("")
    else:
        with open(path, encoding="utf-8", errors="replace") as f:
            lines = [ln.rstrip("\n").rstrip("\r") for ln in f]
        rows = [ln.split(";") for ln in lines]
        width = max((len(r) for r in rows), default=1)
        rows = [r + [""] * (width - len(r)) for r in rows]
        df = pd.DataFrame(rows)
    if df.shape[1] == 1:                           # whole line pasted into column A
        df = df[0].str.split(";", expand=True).fillna("")
    return df.apply(lambda c: c.str.strip())


def load_hro(path):
    df = _load_raw(path).iloc[:, :8]
    df.columns = ["net", "job", "raw_logical", "physical", "io",
                  "dep_cycle", "dep_job", "program"]
    df["row_id"] = df.index + 1
    df = df[~df["net"].str.upper().str.startswith("NET/CYCLE")
            & (df["raw_logical"] != "")].copy()
    df["logical"] = df["raw_logical"].str.replace(r"[IO]$", "", regex=True)
    df["external"] = df["dep_cycle"] != df["net"]
    df["access"] = df["io"].map({"I": "reads", "O": "writes"})
    return df


def load_iho_sections(path):
    """Split B11IHO on its blank separator line into (files_df, programs_df).
    Falls back to files-only if no blank line / second header is present,
    so this also reads an OLD-STYLE (files-only) B11IHO without changes."""
    raw = _load_raw(path)
    blank_rows = raw[(raw == "").all(axis=1)].index.tolist()
    if blank_rows:
        cut = blank_rows[0]
        files_raw, prog_raw = raw.iloc[:cut], raw.iloc[cut + 1:]
    else:
        files_raw, prog_raw = raw, raw.iloc[0:0]

    files = files_raw.iloc[:, :3].copy()
    files.columns = ["logfile", "bestnr", "decom"]
    files = files[files["logfile"].str.upper() != "LOGFILE"].copy()
    files["decom"] = files["decom"].str.upper()

    if len(prog_raw):
        programs = prog_raw.iloc[:, :5].copy()
        programs.columns = ["job", "program", "files", "blocking_files", "decom"]
        programs = programs[programs["job"].str.upper() != "JOB"].copy()
        programs["decom"] = programs["decom"].str.upper()
    else:
        programs = pd.DataFrame(columns=["job", "program", "files", "blocking_files", "decom"])
    return files, programs


hro = load_hro(HRO)
iho_files, iho_programs = load_iho_sections(IHO)
print(f"HRO rows: {len(hro)}   IHO files: {len(iho_files)}   IHO programs: {len(iho_programs)}")

### CELL 3 - validate both sections against the candidate rule -------------
def bestnr(physical):
    parts = physical.split(".")
    if parts[0] == "STAP" and len(parts) > 1:
        return parts[1]
    if len(parts) > 1 and "%AENV" in parts[1]:
        return "B" + parts[1].split("%AENV")[1]
    return ""


hro["bestnr"] = hro["physical"].map(bestnr)
ext = hro[hro["external"]]

# -- file-level engine (same rule validated at 37/37 earlier) --
file_engine = hro.groupby("logical").agg(
    engine_decision=("external", lambda s: "NO" if s.any() else "YES"),
    bestnr_derived=("bestnr", lambda s: ", ".join(sorted(set(s)))))
file_blockers = (ext.assign(b=ext["dep_job"] + " (" + ext["dep_cycle"] + ")")
                  .groupby("logical")["b"].agg(lambda s: ", ".join(sorted(set(s)))))

file_res = iho_files.merge(file_engine, left_on="logfile", right_index=True, how="left")
file_res["engine_decision"] = file_res["engine_decision"].fillna("MISSING")
file_res["blocking_dependents"] = file_res["logfile"].map(file_blockers).fillna("")
file_res["match"] = (file_res["engine_decision"] == file_res["decom"]) & file_res.apply(
    lambda r: r["bestnr"] in [x.strip() for x in str(r["bestnr_derived"] or "").split(",")],
    axis=1)

print(f"\nFILES:    {(file_res['match']).sum()}/{len(file_res)} match")
if (~file_res["match"]).any():
    print(file_res[~file_res["match"]].to_string(index=False))

# -- program-level engine: YES iff every file of the (job,program) step is YES --
steps = hro[["job", "program", "logical"]].drop_duplicates().copy()
steps["file_decom"] = steps["logical"].map(iho_files.set_index("logfile")["decom"])
prog_blockers = (steps[steps["file_decom"] == "NO"].groupby(["job", "program"])["logical"]
                  .agg(lambda s: ", ".join(sorted(s))))
prog_engine = (steps.groupby(["job", "program"])
               .agg(files=("logical", "nunique"),
                    unknown=("file_decom", lambda s: s.isna().any()))
               .join(prog_blockers.rename("blocking")).reset_index())
prog_engine["blocking"] = prog_engine["blocking"].fillna("")
prog_engine["engine_decision"] = "YES"
prog_engine.loc[prog_engine["unknown"], "engine_decision"] = "UNKNOWN"
prog_engine.loc[prog_engine["blocking"] != "", "engine_decision"] = "NO"

if len(iho_programs):
    prog_res = iho_programs.merge(prog_engine, on=["job", "program"], how="left",
                                   suffixes=("", "_calc"))
    prog_res["engine_decision"] = prog_res["engine_decision"].fillna("MISSING")
    prog_res["match"] = prog_res["decom"] == prog_res["engine_decision"]
    print(f"\nPROGRAMS: {(prog_res['match']).sum()}/{len(prog_res)} match")
    if (~prog_res["match"]).any():
        print(prog_res[~prog_res["match"]].to_string(index=False))
else:
    print("\nPROGRAMS: none found in this B11IHO (old-style file, or no program section)")


### CELL 4 - the LLM's tools (plain functions over the data above) --------
def get_decision(logical_file: str) -> dict:
    """Y/N decision and evidence for one logical file."""
    row = iho_files[iho_files["logfile"] == logical_file]
    if row.empty:
        return {"error": f"{logical_file} not found in B11IHO"}
    blockers = ext[ext["logical"] == logical_file]
    return {
        "logical_file": logical_file,
        "decom": row.iloc[0]["decom"],
        "blocking_dependents": [
            {"job": r.dep_job, "cycle": r.dep_cycle, "access": r.access, "row_id": r.row_id}
            for r in blockers.itertuples()],
    }


def get_program_decision(job: str, program: str) -> dict:
    """Y/N decision and blocking files for one job/program step."""
    row = iho_programs[(iho_programs["job"] == job) & (iho_programs["program"] == program)]
    if row.empty:
        return {"error": f"{job}/{program} not found in B11IHO"}
    r = row.iloc[0]
    return {"job": job, "program": program, "files": int(r["files"]),
            "blocking_files": [f for f in r["blocking_files"].split(" ") if f],
            "decom": r["decom"]}


def get_cycle_summary(cycle: str) -> dict:
    """Overview of one net/cycle: how many files/programs are YES vs NO."""
    files = iho_files  # single-net data; filter here if you load multiple nets
    programs = iho_programs
    return {
        "cycle": cycle,
        "files_total": len(files), "files_no": int((files["decom"] == "NO").sum()),
        "programs_total": len(programs), "programs_no": int((programs["decom"] == "NO").sum()),
        "blocked_files": files[files["decom"] == "NO"]["logfile"].tolist(),
        "blocked_programs": (programs.loc[programs["decom"] == "NO", ["job", "program"]]
                             .apply(lambda r: f"{r.job}/{r.program}", axis=1).tolist()
                             if len(programs) else []),
    }


TOOLS = {
    "get_decision": get_decision,
    "get_program_decision": get_program_decision,
    "get_cycle_summary": get_cycle_summary,
}

print(get_decision("BP569"))
print(get_program_decision("TS660420", "S912"))
print(get_cycle_summary("TS6604"))


### CELL 5 - Gemini chat loop with function calling ------------------------
# pip install -q -U google-genai   # uncomment on Kaggle if the package is missing
from kaggle_secrets import UserSecretsClient
from google import genai
from google.genai import types

api_key = UserSecretsClient().get_secret("GEMINI_API_KEY")   # set this in Add-ons > Secrets
client = genai.Client(api_key=api_key)

SYSTEM_PROMPT = (
    "You explain mainframe decommissioning decisions. Always call a tool to "
    "get the facts; never guess a decision or a job/program name. Cite the "
    "blocking jobs and cycles the tool returns. If a tool says 'not found', "
    "say the data doesn't cover it - don't invent an answer."
)

gemini_tools = [
    {"name": "get_decision", "description": "Y/N decommission decision and blocking jobs for one logical file.",
     "parameters": {"type": "object", "properties": {"logical_file": {"type": "string"}},
                    "required": ["logical_file"]}},
    {"name": "get_program_decision", "description": "Y/N decommission decision for one job/program step.",
     "parameters": {"type": "object", "properties": {"job": {"type": "string"}, "program": {"type": "string"}},
                    "required": ["job", "program"]}},
    {"name": "get_cycle_summary", "description": "Overview of a net/cycle: counts of blocked files and programs.",
     "parameters": {"type": "object", "properties": {"cycle": {"type": "string"}}, "required": ["cycle"]}},
]


def ask(question: str, history=None, verbose: bool = True) -> str:
    import time
    from google.genai import errors
    history = history or []
    contents = history + [types.Content(role="user", parts=[types.Part(text=question)])]
    for round_num in range(5):                          # cap tool-call chaining
        for attempt in range(3):                        # retry budget: ~1+2=3s max wait
            if verbose:
                print(f"  round {round_num+1}, attempt {attempt+1}: calling Gemini...")
            try:
                resp = client.models.generate_content(
                    model="gemini-3.8-flash", contents=contents,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        tools=[types.Tool(function_declarations=gemini_tools)]))
                break
            except errors.ServerError:
                if attempt == 2:
                    raise
                wait = 2 ** attempt          # 1, 2 seconds
                print(f"  Gemini busy (503) - retrying in {wait}s...")
                time.sleep(wait)
        part = resp.candidates[0].content.parts[0]
        if part.function_call:
            fn, args = part.function_call.name, dict(part.function_call.args)
            if verbose:
                print(f"  -> tool call: {fn}({args})")
            result = TOOLS[fn](**args)
            contents.append(resp.candidates[0].content)
            contents.append(types.Content(role="user", parts=[
                types.Part(function_response=types.FunctionResponse(name=fn, response=result))]))
            continue
        return part.text
    return "(stopped after 5 tool calls)"


print(ask("Can BP569 be decommissioned?"))
print(ask("What about the TS6604 cycle overall?"))

### CELL 4.5 (optional) - list models available to your key, if 3.8-flash keeps 503ing
# for m in client.models.list():
#     print(m.name)

/kaggle/input/datasets
/kaggle/input/datasets/aburvan
/kaggle/input/datasets/aburvan/report2
/kaggle/input/datasets/aburvan/report
/kaggle/input/datasets/aburvan/report1
/kaggle/input/datasets/aburvan/report2/B11HR.xlsx
/kaggle/input/datasets/aburvan/report2/B11IH.xlsx
/kaggle/input/datasets/aburvan/report/B11HR.xlsx
/kaggle/input/datasets/aburvan/report/B11IH.xlsx
/kaggle/input/datasets/aburvan/report1/B11HR.xlsx
/kaggle/input/datasets/aburvan/report1/B11IH.xlsx
HRO rows: 449   IHO files: 37   IHO programs: 0

FILES:    37/37 match

PROGRAMS: none found in this B11IHO (old-style file, or no program section)
{'logical_file': 'BP569', 'decom': 'NO', 'blocking_dependents': [{'job': 'TD160425', 'cycle': 'TD1604', 'access': 'reads', 'row_id': 171}, {'job': 'TD160401', 'cycle': 'TD1604', 'access': 'reads', 'row_id': 172}]}
{'error': 'TS660420/S912 not found in B11IHO'}
{'cycle': 'TS6604', 'files_total': 37, 'files_no': 7, 'programs_total': 0, 'programs_no': 0, 'blocked_files': ['BY4JE', 'BY

In [7]:
# ============================================================================
# Mainframe decommission assistant - single Kaggle notebook, cell by cell.
# Paste each ### CELL N block into its own cell, in order.
# pandas + openpyxl are preinstalled on Kaggle; only the LLM cell needs an
# internet-enabled notebook + a Gemini API key in Kaggle Secrets.
# ============================================================================


### CELL 1 - locate your files -------------------------------------------

import glob

for p in glob.glob("/kaggle/input/**/*", recursive=True):
    print(p)                                     # <- copy paths for CELL 2


### CELL 2 - load B11HRO and B11IHO --------------------------------------

import pandas as pd

HRO = "/kaggle/input/datasets/aburvan/report2/B11HR.xlsx"
IHO = "/kaggle/input/datasets/aburvan/report2/B11IH.xlsx"


def _load_raw(path):
    """
    Read into a DataFrame of raw string rows, one row per input record.

    Supports:
      - .xlsx / .xlsm / .xls
      - semicolon-separated text files

    Keeps blank lines and tolerates different column counts.
    """
    if path.lower().endswith((".xlsx", ".xlsm", ".xls")):
        df = pd.read_excel(path, header=None, dtype=str)
        df = df.fillna("")

    else:
        with open(path, encoding="utf-8", errors="replace") as f:
            lines = [ln.rstrip("\n").rstrip("\r") for ln in f]

        rows = [ln.split(";") for ln in lines]

        width = max((len(r) for r in rows), default=1)

        rows = [
            r + [""] * (width - len(r))
            for r in rows
        ]

        df = pd.DataFrame(rows)

    # If the whole line was pasted into column A,
    # split it using semicolon.
    if df.shape[1] == 1:
        df = df[0].str.split(";", expand=True).fillna("")

    return df.apply(lambda c: c.astype(str).str.strip())


# --------------------------------------------------------------------------
# HRO loader
# --------------------------------------------------------------------------

def load_hro(path):

    df = _load_raw(path).iloc[:, :8].copy()

    df.columns = [
        "net",
        "job",
        "raw_logical",
        "physical",
        "io",
        "dep_cycle",
        "dep_job",
        "program"
    ]

    df["row_id"] = df.index + 1

    df = df[
        ~df["net"].str.upper().str.startswith("NET/CYCLE")
        & (df["raw_logical"] != "")
    ].copy()

    df["logical"] = df["raw_logical"].str.replace(
        r"[IO]$",
        "",
        regex=True
    )

    df["external"] = df["dep_cycle"] != df["net"]

    df["access"] = df["io"].map({
        "I": "reads",
        "O": "writes"
    })

    return df


# --------------------------------------------------------------------------
# IHO loader
# --------------------------------------------------------------------------

def load_iho_sections(path):
    """
    Load B11IHO and support all three layouts.

    FORMAT 1 - Mainframe-style single-row header:

        LOGFILE | BESTNR | DECOM | JOB | PROGRAM | FILES |
        BLOCKING FILES | DECOM

        BHW85 | BHW85 | YES | TS660451 | ZR68 | 20 | BY4JE | NO

    FORMAT 2 - Two sections separated by blank row:

        LOGFILE | BESTNR | DECOM
        ...
        <blank row>
        JOB | PROGRAM | FILES | BLOCKING FILES | DECOM
        ...

    FORMAT 3 - Old-style files-only B11IHO.
    """

    raw = _load_raw(path)

    # ======================================================================
    # FORMAT 1
    # Mainframe output where both headers are on the SAME row.
    #
    # IMPORTANT:
    # Use .iloc[:, 0] and .iloc[:, 3] rather than raw[0] / raw[3].
    # This avoids the KeyError that occurred with Excel column labels.
    # ======================================================================

    dual_header_rows = []

    if raw.shape[1] >= 4:

        col0 = (
            raw.iloc[:, 0]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        col3 = (
            raw.iloc[:, 3]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        dual_header_rows = raw.index[
            col0.eq("LOGFILE") &
            col3.eq("JOB")
        ].tolist()

    if dual_header_rows:

        header_row = dual_header_rows[0]

        # Everything after the combined header is data.
        data_raw = raw.iloc[header_row + 1:].copy()

        # --------------------------------------------------------------
        # FILE SECTION
        # Columns:
        # 0 = LOGFILE
        # 1 = BESTNR
        # 2 = DECOM
        # --------------------------------------------------------------

        files_raw = data_raw.iloc[:, :3].copy()

        # --------------------------------------------------------------
        # PROGRAM SECTION
        # Columns:
        # 3 = JOB
        # 4 = PROGRAM
        # 5 = FILES
        # 6 = BLOCKING FILES
        # 7 = DECOM
        # --------------------------------------------------------------

        prog_raw = data_raw.iloc[:, 3:8].copy()

    else:

        # ==================================================================
        # FORMAT 2
        # Existing layout where files and programs are separated by
        # a completely blank row.
        # ==================================================================

        blank_rows = raw[
            raw.astype(str).apply(
                lambda row: row.str.strip().eq("").all(),
                axis=1
            )
        ].index.tolist()

        if blank_rows:

            cut = blank_rows[0]

            files_raw = raw.iloc[:cut].copy()

            prog_raw = raw.iloc[cut + 1:].copy()

        else:

            # ==============================================================
            # FORMAT 3
            # Old-style files-only B11IHO.
            # ==============================================================

            files_raw = raw.copy()

            prog_raw = raw.iloc[0:0].copy()

    # ======================================================================
    # Build FILE dataframe
    # ======================================================================

    files = files_raw.iloc[:, :3].copy()

    files.columns = [
        "logfile",
        "bestnr",
        "decom"
    ]

    # Remove header rows if present.
    files = files[
        files["logfile"]
        .astype(str)
        .str.strip()
        .str.upper()
        != "LOGFILE"
    ].copy()

    files["decom"] = (
        files["decom"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # ======================================================================
    # Build PROGRAM dataframe
    # ======================================================================

    if len(prog_raw):

        programs = prog_raw.iloc[:, :5].copy()

        programs.columns = [
            "job",
            "program",
            "files",
            "blocking_files",
            "decom"
        ]

        # Remove JOB header rows if present.
        programs = programs[
            programs["job"]
            .astype(str)
            .str.strip()
            .str.upper()
            != "JOB"
        ].copy()

        programs["decom"] = (
            programs["decom"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

    else:

        programs = pd.DataFrame(
            columns=[
                "job",
                "program",
                "files",
                "blocking_files",
                "decom"
            ]
        )

    return files, programs


# --------------------------------------------------------------------------
# Load both datasets
# --------------------------------------------------------------------------

hro = load_hro(HRO)

iho_files, iho_programs = load_iho_sections(IHO)

print(
    f"HRO rows: {len(hro)}   "
    f"IHO files: {len(iho_files)}   "
    f"IHO programs: {len(iho_programs)}"
)

print("\nIHO FILES:")
print(iho_files.head())

print("\nIHO PROGRAMS:")
print(iho_programs.head())


### CELL 3 - validate both sections against the candidate rule ---------

def bestnr(physical):

    parts = physical.split(".")

    if parts[0] == "STAP" and len(parts) > 1:
        return parts[1]

    if len(parts) > 1 and "%AENV" in parts[1]:
        return "B" + parts[1].split("%AENV")[1]

    return ""


hro["bestnr"] = hro["physical"].map(bestnr)

ext = hro[hro["external"]]


# --------------------------------------------------------------------------
# File-level engine
# --------------------------------------------------------------------------

file_engine = (
    hro.groupby("logical")
    .agg(
        engine_decision=(
            "external",
            lambda s: "NO" if s.any() else "YES"
        ),
        bestnr_derived=(
            "bestnr",
            lambda s: ", ".join(sorted(set(s)))
        )
    )
)


file_blockers = (
    ext.assign(
        b=ext["dep_job"] + " (" + ext["dep_cycle"] + ")"
    )
    .groupby("logical")["b"]
    .agg(lambda s: ", ".join(sorted(set(s))))
)


file_res = iho_files.merge(
    file_engine,
    left_on="logfile",
    right_index=True,
    how="left"
)

file_res["engine_decision"] = (
    file_res["engine_decision"]
    .fillna("MISSING")
)

file_res["blocking_dependents"] = (
    file_res["logfile"]
    .map(file_blockers)
    .fillna("")
)

file_res["match"] = (
    file_res["engine_decision"] == file_res["decom"]
) & file_res.apply(
    lambda r:
        r["bestnr"] in [
            x.strip()
            for x in str(
                r["bestnr_derived"] or ""
            ).split(",")
        ],
    axis=1
)


print(
    f"\nFILES:    "
    f"{file_res['match'].sum()}/{len(file_res)} match"
)

if (~file_res["match"]).any():

    print(
        file_res[
            ~file_res["match"]
        ].to_string(index=False)
    )


# --------------------------------------------------------------------------
# Program-level engine
# YES iff every file of the (job,program) step is YES
# --------------------------------------------------------------------------

steps = (
    hro[
        ["job", "program", "logical"]
    ]
    .drop_duplicates()
    .copy()
)


steps["file_decom"] = (
    steps["logical"]
    .map(
        iho_files.set_index("logfile")["decom"]
    )
)


prog_blockers = (
    steps[
        steps["file_decom"] == "NO"
    ]
    .groupby(
        ["job", "program"]
    )["logical"]
    .agg(
        lambda s: ", ".join(sorted(s))
    )
)


prog_engine = (
    steps
    .groupby(["job", "program"])
    .agg(
        files=("logical", "nunique"),
        unknown=(
            "file_decom",
            lambda s: s.isna().any()
        )
    )
    .join(
        prog_blockers.rename("blocking")
    )
    .reset_index()
)


prog_engine["blocking"] = (
    prog_engine["blocking"]
    .fillna("")
)


prog_engine["engine_decision"] = "YES"

prog_engine.loc[
    prog_engine["unknown"],
    "engine_decision"
] = "UNKNOWN"

prog_engine.loc[
    prog_engine["blocking"] != "",
    "engine_decision"
] = "NO"


if len(iho_programs):

    prog_res = iho_programs.merge(
        prog_engine,
        on=["job", "program"],
        how="left",
        suffixes=("", "_calc")
    )

    prog_res["engine_decision"] = (
        prog_res["engine_decision"]
        .fillna("MISSING")
    )

    prog_res["match"] = (
        prog_res["decom"]
        == prog_res["engine_decision"]
    )

    print(
        f"\nPROGRAMS: "
        f"{prog_res['match'].sum()}/{len(prog_res)} match"
    )

    if (~prog_res["match"]).any():

        print(
            prog_res[
                ~prog_res["match"]
            ].to_string(index=False)
        )

else:

    print(
        "\nPROGRAMS: none found in this B11IHO "
        "(old-style file, or no program section)"
    )


### CELL 4 - the LLM's tools --------------------------------------------

def get_decision(logical_file: str) -> dict:
    """Y/N decision and evidence for one logical file."""

    row = iho_files[
        iho_files["logfile"] == logical_file
    ]

    if row.empty:

        return {
            "error":
                f"{logical_file} not found in B11IHO"
        }

    blockers = ext[
        ext["logical"] == logical_file
    ]

    return {
        "logical_file": logical_file,

        "decom":
            row.iloc[0]["decom"],

        "blocking_dependents": [
            {
                "job": r.dep_job,
                "cycle": r.dep_cycle,
                "access": r.access,
                "row_id": r.row_id
            }

            for r in blockers.itertuples()
        ]
    }


def get_program_decision(
    job: str,
    program: str
) -> dict:

    """Y/N decision and blocking files for one job/program step."""

    row = iho_programs[
        (iho_programs["job"] == job)
        &
        (iho_programs["program"] == program)
    ]

    if row.empty:

        return {
            "error":
                f"{job}/{program} not found in B11IHO"
        }

    r = row.iloc[0]

    return {
        "job": job,
        "program": program,
        "files": int(r["files"]),

        "blocking_files": [
            f
            for f in r["blocking_files"].split(" ")
            if f
        ],

        "decom": r["decom"]
    }


def get_cycle_summary(
    cycle: str
) -> dict:

    """Overview of one net/cycle."""

    files = iho_files

    programs = iho_programs

    return {

        "cycle": cycle,

        "files_total":
            len(files),

        "files_no":
            int(
                (files["decom"] == "NO").sum()
            ),

        "programs_total":
            len(programs),

        "programs_no":
            int(
                (programs["decom"] == "NO").sum()
            ),

        "blocked_files":
            files[
                files["decom"] == "NO"
            ]["logfile"].tolist(),

        "blocked_programs":
            (
                programs.loc[
                    programs["decom"] == "NO",
                    ["job", "program"]
                ]
                .apply(
                    lambda r:
                        f"{r.job}/{r.program}",
                    axis=1
                )
                .tolist()
                if len(programs)
                else []
            )
    }


TOOLS = {

    "get_decision":
        get_decision,

    "get_program_decision":
        get_program_decision,

    "get_cycle_summary":
        get_cycle_summary,
}


# Test the tools

print(
    get_decision("BP569")
)

print(
    get_program_decision(
        "TS660420",
        "S912"
    )
)

print(
    get_cycle_summary("TS6604")
)


### CELL 5 - Gemini chat loop with function calling ---------------------

# Uncomment if google-genai is not installed:
# !pip install -q -U google-genai


from kaggle_secrets import UserSecretsClient

from google import genai

from google.genai import types


api_key = (
    UserSecretsClient()
    .get_secret("GEMINI_API_KEY")
)


client = genai.Client(
    api_key=api_key
)


SYSTEM_PROMPT = (
    "You explain mainframe decommissioning decisions. "
    "Always call a tool to get the facts; never guess a "
    "decision or a job/program name. Cite the blocking jobs "
    "and cycles the tool returns. If a tool says 'not found', "
    "say the data doesn't cover it - don't invent an answer."
)


gemini_tools = [

    {
        "name": "get_decision",

        "description":
            "Y/N decommission decision and blocking jobs "
            "for one logical file.",

        "parameters": {

            "type": "object",

            "properties": {

                "logical_file": {
                    "type": "string"
                }

            },

            "required": [
                "logical_file"
            ]
        }
    },


    {
        "name": "get_program_decision",

        "description":
            "Y/N decommission decision for one "
            "job/program step.",

        "parameters": {

            "type": "object",

            "properties": {

                "job": {
                    "type": "string"
                },

                "program": {
                    "type": "string"
                }

            },

            "required": [
                "job",
                "program"
            ]
        }
    },


    {
        "name": "get_cycle_summary",

        "description":
            "Overview of a net/cycle: counts of "
            "blocked files and programs.",

        "parameters": {

            "type": "object",

            "properties": {

                "cycle": {
                    "type": "string"
                }

            },

            "required": [
                "cycle"
            ]
        }
    }
]


def ask(
    question: str,
    history=None,
    verbose: bool = True
) -> str:

    import time

    from google.genai import errors

    history = history or []

    contents = (
        history
        +
        [
            types.Content(
                role="user",
                parts=[
                    types.Part(
                        text=question
                    )
                ]
            )
        ]
    )

    # Maximum of 5 tool-call rounds
    for round_num in range(5):

        # Retry budget
        for attempt in range(3):

            if verbose:

                print(
                    f"  round {round_num + 1}, "
                    f"attempt {attempt + 1}: "
                    f"calling Gemini..."
                )

            try:

                resp = (
                    client.models.generate_content(

                        model="gemini-3.8-flash",

                        contents=contents,

                        config=types.GenerateContentConfig(

                            system_instruction=
                                SYSTEM_PROMPT,

                            tools=[
                                types.Tool(
                                    function_declarations=
                                        gemini_tools
                                )
                            ]
                        )
                    )
                )

                break

            except errors.ServerError:

                if attempt == 2:
                    raise

                wait = 2 ** attempt

                print(
                    f"  Gemini busy (503) - "
                    f"retrying in {wait}s..."
                )

                time.sleep(wait)

        part = (
            resp
            .candidates[0]
            .content
            .parts[0]
        )

        if part.function_call:

            fn = part.function_call.name

            args = dict(
                part.function_call.args
            )

            if verbose:

                print(
                    f"  -> tool call: "
                    f"{fn}({args})"
                )

            result = TOOLS[fn](**args)

            contents.append(
                resp.candidates[0].content
            )

            contents.append(
                types.Content(

                    role="user",

                    parts=[
                        types.Part(
                            function_response=
                                types.FunctionResponse(
                                    name=fn,
                                    response=result
                                )
                        )
                    ]
                )
            )

            continue

        return part.text

    return "(stopped after 5 tool calls)"


# --------------------------------------------------------------------------
# Test Gemini
# --------------------------------------------------------------------------

print(
    ask(
        "Can BP569 be decommissioned?"
    )
)

print(
    ask(
        "What about the TS6604 cycle overall?"
    )
)


### CELL 4.5 - OPTIONAL: list Gemini models ------------------------------

# If gemini-3.8-flash keeps returning 503,
# uncomment the following:

# for m in client.models.list():
#     print(m.name)

/kaggle/input/datasets
/kaggle/input/datasets/aburvan
/kaggle/input/datasets/aburvan/report2
/kaggle/input/datasets/aburvan/report
/kaggle/input/datasets/aburvan/report1
/kaggle/input/datasets/aburvan/report2/B11HR.xlsx
/kaggle/input/datasets/aburvan/report2/B11IH.xlsx
/kaggle/input/datasets/aburvan/report/B11HR.xlsx
/kaggle/input/datasets/aburvan/report/B11IH.xlsx
/kaggle/input/datasets/aburvan/report1/B11HR.xlsx
/kaggle/input/datasets/aburvan/report1/B11IH.xlsx
HRO rows: 449   IHO files: 37   IHO programs: 0

IHO FILES:
  logfile bestnr decom
1   BHW85  BHW85   YES
2   BP572  BP572   YES
3   BP888  BP888   YES
4   BWIH5  BWIH5   YES
5   BWIH7  BWIH7   YES

IHO PROGRAMS:
Empty DataFrame
Columns: [job, program, files, blocking_files, decom]
Index: []

FILES:    37/37 match

PROGRAMS: none found in this B11IHO (old-style file, or no program section)
{'logical_file': 'BP569', 'decom': 'NO', 'blocking_dependents': [{'job': 'TD160425', 'cycle': 'TD1604', 'access': 'reads', 'row_id': 171}, 

KeyboardInterrupt: 

In [6]:
# ============================================================================
# MAINFRAME DECOMMISSION ASSISTANT
# Single Kaggle notebook - cell by cell
#
# Inputs:
#   B11HR.xlsx -> HRO dependency data
#   B11IH.xlsx -> IHO file + program decisions
#
# Supports B11IHO layouts:
#   1. Combined header:
#      LOGFILE | BESTNR | DECOM | JOB | PROGRAM | FILES | BLOCKING FILES | DECOM
#
#   2. Two sections separated by blank row
#
#   3. Old files-only B11IHO
# ============================================================================


# ============================================================================
# CELL 1 - LOCATE FILES
# ============================================================================

import glob

for p in glob.glob("/kaggle/input/**/*", recursive=True):
    print(p)


# ============================================================================
# CELL 2 - LOAD B11HR AND B11IH
# ============================================================================

import pandas as pd


HRO = "/kaggle/input/datasets/aburvan/report2/B11HR.xlsx"
IHO = "/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx"


# ----------------------------------------------------------------------------
# Generic raw loader
# ----------------------------------------------------------------------------

def _load_raw(path):
    """
    Read Excel or semicolon-separated text into a raw DataFrame.

    Everything is treated as strings.
    Blank cells become "".
    """

    if path.lower().endswith((".xlsx", ".xlsm", ".xls")):

        df = pd.read_excel(
            path,
            header=None,
            dtype=str
        )

        df = df.fillna("")

    else:

        with open(
            path,
            encoding="utf-8",
            errors="replace"
        ) as f:

            lines = [
                ln.rstrip("\n").rstrip("\r")
                for ln in f
            ]

        rows = [
            ln.split(";")
            for ln in lines
        ]

        width = max(
            (len(r) for r in rows),
            default=1
        )

        rows = [
            r + [""] * (width - len(r))
            for r in rows
        ]

        df = pd.DataFrame(rows)

    # If the entire record is in one column,
    # split it using semicolon.
    if df.shape[1] == 1:

        df = (
            df.iloc[:, 0]
            .astype(str)
            .str.split(";", expand=True)
            .fillna("")
        )

    # Strip whitespace from every cell.
    df = df.apply(
        lambda c:
            c.astype(str).str.strip()
    )

    return df


# ----------------------------------------------------------------------------
# HRO loader
# ----------------------------------------------------------------------------

def load_hro(path):

    df = _load_raw(path)

    # HRO uses the first 8 columns.
    df = df.iloc[:, :8].copy()

    df.columns = [
        "net",
        "job",
        "raw_logical",
        "physical",
        "io",
        "dep_cycle",
        "dep_job",
        "program"
    ]

    # Original source row number.
    df["row_id"] = df.index + 1

    # Remove NET/CYCLE header rows and empty logical files.
    df = df[
        ~df["net"]
        .str.upper()
        .str.startswith("NET/CYCLE")
        &
        (df["raw_logical"] != "")
    ].copy()

    # Remove I/O suffix from logical file.
    df["logical"] = (
        df["raw_logical"]
        .str.replace(
            r"[IO]$",
            "",
            regex=True
        )
    )

    # External dependency if dependent cycle != current network.
    df["external"] = (
        df["dep_cycle"] != df["net"]
    )

    # Convert I/O indicator.
    df["access"] = df["io"].map({
        "I": "reads",
        "O": "writes"
    })

    return df


# ----------------------------------------------------------------------------
# IHO loader
# ----------------------------------------------------------------------------

def load_iho_sections(path):
    """
    Load B11IHO.

    Supports:

    FORMAT 1
    --------
    Combined mainframe-style row:

    LOGFILE | BESTNR | DECOM | JOB | PROGRAM | FILES |
    BLOCKING FILES | DECOM

    FORMAT 2
    --------
    Two sections:

    LOGFILE | BESTNR | DECOM
    ...
    <blank row>
    JOB | PROGRAM | FILES | BLOCKING FILES | DECOM
    ...

    FORMAT 3
    --------
    Old-style files-only B11IHO.
    """

    raw = _load_raw(path)

    print("\nRaw IHO shape:", raw.shape)

    # ========================================================================
    # STEP 1
    # FIND A COMBINED HEADER
    #
    # IMPORTANT:
    # We do NOT assume LOGFILE is column 0 or JOB is column 3.
    #
    # We search every cell in every row.
    # ========================================================================

    dual_header_row = None

    logfile_col = None

    job_col = None

    for idx in raw.index:

        row = (
            raw.loc[idx]
            .astype(str)
            .str.strip()
            .str.upper()
            .tolist()
        )

        logfile_positions = [
            i
            for i, value in enumerate(row)
            if value == "LOGFILE"
        ]

        job_positions = [
            i
            for i, value in enumerate(row)
            if value == "JOB"
        ]

        # We found both headers on the same physical row.
        if logfile_positions and job_positions:

            dual_header_row = idx

            logfile_col = logfile_positions[0]

            job_col = job_positions[0]

            break

    # ========================================================================
    # FORMAT 1 - COMBINED HEADER
    # ========================================================================

    if dual_header_row is not None:

        print(
            "\nDetected combined IHO header:"
        )

        print(
            f"  Header row : {dual_header_row}"
        )

        print(
            f"  LOGFILE col: {logfile_col}"
        )

        print(
            f"  JOB col    : {job_col}"
        )

        # Everything after the header is data.
        data_raw = raw.iloc[
            dual_header_row + 1:
        ].copy()

        # --------------------------------------------------------------------
        # FILE SECTION
        #
        # LOGFILE
        # BESTNR
        # DECOM
        # --------------------------------------------------------------------

        files_raw = data_raw.iloc[
            :,
            logfile_col:logfile_col + 3
        ].copy()

        # --------------------------------------------------------------------
        # PROGRAM SECTION
        #
        # JOB
        # PROGRAM
        # FILES
        # BLOCKING FILES
        # DECOM
        # --------------------------------------------------------------------

        prog_raw = data_raw.iloc[
            :,
            job_col:job_col + 5
        ].copy()

    else:

        # ====================================================================
        # FORMAT 2 - BLANK ROW SEPARATOR
        # ====================================================================

        blank_rows = raw[
            raw.astype(str).apply(
                lambda row:
                    row.str.strip().eq("").all(),
                axis=1
            )
        ].index.tolist()

        if blank_rows:

            cut = blank_rows[0]

            print(
                "\nDetected two-section IHO layout."
            )

            print(
                f"  Blank separator row: {cut}"
            )

            files_raw = raw.iloc[
                :cut
            ].copy()

            prog_raw = raw.iloc[
                cut + 1:
            ].copy()

        else:

            # ================================================================
            # FORMAT 3 - OLD FILES-ONLY
            # ================================================================

            print(
                "\nNo combined header or blank separator found."
            )

            print(
                "Treating IHO as files-only."
            )

            files_raw = raw.copy()

            prog_raw = raw.iloc[
                0:0
            ].copy()

    # ========================================================================
    # BUILD FILE DATAFRAME
    # ========================================================================

    if files_raw.shape[1] >= 3:

        files = files_raw.iloc[
            :,
            :3
        ].copy()

        files.columns = [
            "logfile",
            "bestnr",
            "decom"
        ]

        # Remove LOGFILE header if it somehow remains.
        files = files[
            files["logfile"]
            .astype(str)
            .str.strip()
            .str.upper()
            != "LOGFILE"
        ].copy()

        files["logfile"] = (
            files["logfile"]
            .astype(str)
            .str.strip()
        )

        files["bestnr"] = (
            files["bestnr"]
            .astype(str)
            .str.strip()
        )

        files["decom"] = (
            files["decom"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        # Remove completely empty rows.
        files = files[
            files["logfile"] != ""
        ].copy()

    else:

        files = pd.DataFrame(
            columns=[
                "logfile",
                "bestnr",
                "decom"
            ]
        )

    # ========================================================================
    # BUILD PROGRAM DATAFRAME
    # ========================================================================

    if (
        prog_raw.shape[1] >= 5
        and len(prog_raw) > 0
    ):

        programs = prog_raw.iloc[
            :,
            :5
        ].copy()

        programs.columns = [
            "job",
            "program",
            "files",
            "blocking_files",
            "decom"
        ]

        # Remove JOB header.
        programs = programs[
            programs["job"]
            .astype(str)
            .str.strip()
            .str.upper()
            != "JOB"
        ].copy()

        # Clean fields.
        programs["job"] = (
            programs["job"]
            .astype(str)
            .str.strip()
        )

        programs["program"] = (
            programs["program"]
            .astype(str)
            .str.strip()
        )

        programs["files"] = (
            programs["files"]
            .astype(str)
            .str.strip()
        )

        programs["blocking_files"] = (
            programs["blocking_files"]
            .astype(str)
            .str.strip()
        )

        programs["decom"] = (
            programs["decom"]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        # Remove completely empty program rows.
        programs = programs[
            programs["job"] != ""
        ].copy()

    else:

        programs = pd.DataFrame(
            columns=[
                "job",
                "program",
                "files",
                "blocking_files",
                "decom"
            ]
        )

    return files, programs


# ----------------------------------------------------------------------------
# LOAD DATA
# ----------------------------------------------------------------------------

hro = load_hro(HRO)

iho_files, iho_programs = load_iho_sections(IHO)


print("\n" + "=" * 70)

print("LOAD SUMMARY")

print("=" * 70)

print(
    f"HRO rows       : {len(hro)}"
)

print(
    f"IHO files      : {len(iho_files)}"
)

print(
    f"IHO programs   : {len(iho_programs)}"
)


print("\n" + "=" * 70)

print("IHO FILES")

print("=" * 70)

print(
    iho_files.to_string(index=False)
)


print("\n" + "=" * 70)

print("IHO PROGRAMS")

print("=" * 70)

print(
    iho_programs.to_string(index=False)
)


# ============================================================================
# CELL 3 - VALIDATE FILES AND PROGRAMS
# ============================================================================


def bestnr(physical):

    parts = physical.split(".")

    if (
        len(parts) > 1
        and parts[0] == "STAP"
    ):
        return parts[1]

    if (
        len(parts) > 1
        and "%AENV" in parts[1]
    ):
        return (
            "B"
            +
            parts[1].split("%AENV")[1]
        )

    return ""


# Derive BESTNR from physical dataset.
hro["bestnr"] = (
    hro["physical"]
    .map(bestnr)
)


# External dependencies only.
ext = hro[
    hro["external"]
].copy()


# ----------------------------------------------------------------------------
# FILE-LEVEL ENGINE
# ----------------------------------------------------------------------------

file_engine = (
    hro
    .groupby("logical")
    .agg(
        engine_decision=(
            "external",
            lambda s:
                "NO"
                if s.any()
                else "YES"
        ),

        bestnr_derived=(
            "bestnr",
            lambda s:
                ", ".join(
                    sorted(
                        set(
                            x
                            for x in s
                            if x
                        )
                    )
                )
        )
    )
)


# ----------------------------------------------------------------------------
# FILE BLOCKERS
# ----------------------------------------------------------------------------

file_blockers = (
    ext.assign(
        b=
            ext["dep_job"]
            +
            " ("
            +
            ext["dep_cycle"]
            +
            ")"
    )
    .groupby("logical")["b"]
    .agg(
        lambda s:
            ", ".join(
                sorted(
                    set(s)
                )
            )
    )
)


# ----------------------------------------------------------------------------
# MERGE ENGINE RESULT WITH IHO
# ----------------------------------------------------------------------------

file_res = iho_files.merge(
    file_engine,
    left_on="logfile",
    right_index=True,
    how="left"
)


file_res["engine_decision"] = (
    file_res["engine_decision"]
    .fillna("MISSING")
)


file_res["blocking_dependents"] = (
    file_res["logfile"]
    .map(file_blockers)
    .fillna("")
)


# ----------------------------------------------------------------------------
# FILE MATCH
# ----------------------------------------------------------------------------

def check_file_match(row):

    engine = row["engine_decision"]

    decom = row["decom"]

    derived = str(
        row["bestnr_derived"]
        or ""
    )

    derived_bestnrs = [
        x.strip()
        for x in derived.split(",")
        if x.strip()
    ]

    return (
        engine == decom
        and
        row["bestnr"]
        in derived_bestnrs
    )


file_res["match"] = (
    file_res.apply(
        check_file_match,
        axis=1
    )
)


print("\n" + "=" * 70)

print("FILE VALIDATION")

print("=" * 70)

print(
    f"FILES: "
    f"{file_res['match'].sum()}/"
    f"{len(file_res)} match"
)


if (~file_res["match"]).any():

    print("\nFile mismatches:")

    print(
        file_res[
            ~file_res["match"]
        ].to_string(index=False)
    )


# ----------------------------------------------------------------------------
# PROGRAM-LEVEL ENGINE
#
# YES iff every file belonging to the job/program is YES.
# ----------------------------------------------------------------------------

steps = (
    hro[
        [
            "job",
            "program",
            "logical"
        ]
    ]
    .drop_duplicates()
    .copy()
)


# Map each logical file to its IHO decision.
file_decom_map = (
    iho_files
    .drop_duplicates("logfile")
    .set_index("logfile")["decom"]
)


steps["file_decom"] = (
    steps["logical"]
    .map(file_decom_map)
)


# ----------------------------------------------------------------------------
# Program blockers
# ----------------------------------------------------------------------------

prog_blockers = (
    steps[
        steps["file_decom"] == "NO"
    ]
    .groupby(
        [
            "job",
            "program"
        ]
    )["logical"]
    .agg(
        lambda s:
            ", ".join(
                sorted(s)
            )
    )
)


# ----------------------------------------------------------------------------
# Program engine
# ----------------------------------------------------------------------------

prog_engine = (
    steps
    .groupby(
        [
            "job",
            "program"
        ]
    )
    .agg(
        files=(
            "logical",
            "nunique"
        ),

        unknown=(
            "file_decom",
            lambda s:
                s.isna().any()
        )
    )
    .join(
        prog_blockers.rename(
            "blocking"
        )
    )
    .reset_index()
)


prog_engine["blocking"] = (
    prog_engine["blocking"]
    .fillna("")
)


# Default.
prog_engine["engine_decision"] = "YES"


# Unknown files.
prog_engine.loc[
    prog_engine["unknown"],
    "engine_decision"
] = "UNKNOWN"


# Blocking files.
prog_engine.loc[
    prog_engine["blocking"] != "",
    "engine_decision"
] = "NO"


# ----------------------------------------------------------------------------
# Compare with IHO program decision
# ----------------------------------------------------------------------------

if len(iho_programs):

    prog_res = iho_programs.merge(
        prog_engine,
        on=[
            "job",
            "program"
        ],
        how="left",
        suffixes=(
            "",
            "_calc"
        )
    )

    prog_res["engine_decision"] = (
        prog_res["engine_decision"]
        .fillna("MISSING")
    )

    prog_res["match"] = (
        prog_res["decom"]
        ==
        prog_res["engine_decision"]
    )

    print("\n" + "=" * 70)

    print("PROGRAM VALIDATION")

    print("=" * 70)

    print(
        f"PROGRAMS: "
        f"{prog_res['match'].sum()}/"
        f"{len(prog_res)} match"
    )

    if (~prog_res["match"]).any():

        print("\nProgram mismatches:")

        print(
            prog_res[
                ~prog_res["match"]
            ].to_string(index=False)
        )

else:

    print("\nPROGRAMS: none found in B11IHO.")


# ============================================================================
# CELL 4 - LLM TOOLS
# ============================================================================


def get_decision(
    logical_file: str
) -> dict:
    """
    Return Y/N decision and evidence for one logical file.
    """

    row = iho_files[
        iho_files["logfile"]
        ==
        logical_file
    ]

    if row.empty:

        return {
            "error":
                f"{logical_file} not found in B11IHO"
        }

    blockers = ext[
        ext["logical"]
        ==
        logical_file
    ]

    return {

        "logical_file":
            logical_file,

        "decom":
            row.iloc[0]["decom"],

        "blocking_dependents": [

            {
                "job":
                    r.dep_job,

                "cycle":
                    r.dep_cycle,

                "access":
                    r.access,

                "row_id":
                    int(r.row_id)
            }

            for r
            in blockers.itertuples()
        ]
    }


# ============================================================================
# STRUCTURED PROGRAM DECISION TOOL
# ============================================================================
#
# Purpose:
#   Given JOB + PROGRAM, deterministically identify:
#       - whether the program can be decommissioned
#       - every blocking file
#       - evidence for every blocking file
#       - the source rows used as evidence
#
# The LLM should NOT discover blockers itself.
# It receives this verified evidence through the tool.
# ============================================================================

def get_program_decision(job, program):

    job = str(job).strip().upper()
    program = str(program).strip().upper()

    # ------------------------------------------------------------------------
    # Find the exact program in IHO
    # ------------------------------------------------------------------------

    matches = iho_programs[
        iho_programs["job"].astype(str).str.strip().str.upper().eq(job)
        &
        iho_programs["program"].astype(str).str.strip().str.upper().eq(program)
    ]

    if matches.empty:
        return {
            "error": f"{job}/{program} not found in B11IHO",
            "job": job,
            "program": program
        }

    row = matches.iloc[0]

    # ------------------------------------------------------------------------
    # Program's own DECOM status
    # ------------------------------------------------------------------------

    program_decom = (
        str(row.get("decom", ""))
        .strip()
        .upper()
    )

    # ------------------------------------------------------------------------
    # Extract blocking files from IHO program record
    # ------------------------------------------------------------------------

    raw_blocking = row.get(
        "blocking_files",
        ""
    )

    if pd.isna(raw_blocking):
        raw_blocking = ""

    raw_blocking = str(raw_blocking).strip()

    if raw_blocking:
        blocking_files = [
            x.strip().upper()
            for x in re.split(
                r"[,\s]+",
                raw_blocking
            )
            if x.strip()
        ]
    else:
        blocking_files = []

    # Remove duplicates while preserving order
    blocking_files = list(
        dict.fromkeys(blocking_files)
    )

    # ------------------------------------------------------------------------
    # Build structured evidence for EVERY blocking file
    # ------------------------------------------------------------------------

    evidence = []

    for logical_file in blocking_files:

        # ------------------------------------------------------------
        # Look up the file in IHO
        # ------------------------------------------------------------

        file_matches = iho_files[
            iho_files["logfile"]
            .astype(str)
            .str.strip()
            .str.upper()
            .eq(logical_file)
        ]

        file_decom = None
        file_bestnr = None

        if not file_matches.empty:

            file_row = file_matches.iloc[0]

            file_decom = (
                str(
                    file_row.get(
                        "decom",
                        ""
                    )
                )
                .strip()
                .upper()
            )

            file_bestnr = (
                str(
                    file_row.get(
                        "bestnr",
                        ""
                    )
                )
                .strip()
                .upper()
            )

        # ------------------------------------------------------------
        # Find HRO dependency evidence
        #
        # HRO contains the actual dependency relationships.
        # We search all columns so this remains robust to the
        # report layout.
        # ------------------------------------------------------------

        hro_evidence = []

        for idx, hro_row in ext.iterrows():

            row_values = [
                str(v).strip().upper()
                for v in hro_row.tolist()
                if not pd.isna(v)
            ]

            if logical_file not in row_values:
                continue

            # --------------------------------------------------------
            # Store the complete source row as evidence
            # --------------------------------------------------------

            hro_evidence.append({
                "row_id": int(idx),

                "matched_file":
                    logical_file,

                "row_values":
                    [
                        str(v)
                        for v in hro_row.tolist()
                        if not pd.isna(v)
                    ]
            })

        # ------------------------------------------------------------
        # Determine reason
        # ------------------------------------------------------------

        if file_decom == "NO":

            reason = (
                f"File {logical_file} is marked DECOM=NO "
                f"in B11IHO."
            )

        elif file_decom == "YES":

            reason = (
                f"File {logical_file} is marked DECOM=YES "
                f"in B11IHO."
            )

        else:

            reason = (
                f"No reliable DECOM status found for "
                f"{logical_file}."
            )

        evidence.append({

            "logical_file":
                logical_file,

            "bestnr":
                file_bestnr,

            "decom":
                file_decom,

            "reason":
                reason,

            "hro_dependency_evidence":
                hro_evidence,

            "evidence_count":
                len(hro_evidence)
        })

    # ------------------------------------------------------------------------
    # IMPORTANT:
    #
    # The program can only be considered decomposable when:
    #
    #   1. Its own DECOM status is YES
    #   2. There are no blocking files
    #
    # ------------------------------------------------------------------------

    blocking_non_decom_files = [
        e["logical_file"]
        for e in evidence
        if e["decom"] == "NO"
    ]

    unknown_files = [
        e["logical_file"]
        for e in evidence
        if e["decom"] not in ("YES", "NO")
    ]

    if program_decom == "NO":

        final_decom = "NO"

        final_reason = (
            f"Program {job}/{program} is marked "
            f"DECOM=NO."
        )

    elif blocking_non_decom_files:

        final_decom = "NO"

        final_reason = (
            f"Program {job}/{program} cannot be "
            f"decommissioned because it depends on "
            f"{len(blocking_non_decom_files)} "
            f"non-decommissionable file(s)."
        )

    elif unknown_files:

        final_decom = "NO"

        final_reason = (
            f"Program {job}/{program} cannot be safely "
            f"approved because the DECOM status of "
            f"one or more dependencies is unknown."
        )

    else:

        final_decom = "YES"

        final_reason = (
            f"Program {job}/{program} has no known "
            f"non-decommissionable file dependencies."
        )

    # ------------------------------------------------------------------------
    # Return structured evidence
    # ------------------------------------------------------------------------

    return {

        "job":
            job,

        "program":
            program,

        "decom":
            final_decom,

        "source_program_decom":
            program_decom,

        "blocking_files":
            blocking_files,

        "blocking_non_decom_files":
            blocking_non_decom_files,

        "unknown_files":
            unknown_files,

        "final_reason":
            final_reason,

        "evidence":
            evidence,

        "evidence_summary": {

            "total_dependencies":
                len(evidence),

            "non_decom_dependencies":
                len(blocking_non_decom_files),

            "unknown_dependencies":
                len(unknown_files),

            "all_dependencies_verified":
                len(unknown_files) == 0
        }
    }

def get_cycle_summary(
    cycle: str
) -> dict:
    """
    Overview of one net/cycle.
    """

    files = iho_files

    programs = iho_programs

    return {

        "cycle":
            cycle,

        "files_total":
            len(files),

        "files_no":
            int(
                (
                    files["decom"]
                    ==
                    "NO"
                ).sum()
            ),

        "programs_total":
            len(programs),

        "programs_no":
            int(
                (
                    programs["decom"]
                    ==
                    "NO"
                ).sum()
            ),

        "blocked_files":
            files[
                files["decom"]
                ==
                "NO"
            ]["logfile"].tolist(),

        "blocked_programs":
            (
                programs.loc[
                    programs["decom"]
                    ==
                    "NO",
                    [
                        "job",
                        "program"
                    ]
                ]
                .apply(
                    lambda r:
                        f"{r.job}/{r.program}",
                    axis=1
                )
                .tolist()
                if len(programs)
                else []
            )
    }


# ----------------------------------------------------------------------------
# Register tools
# ----------------------------------------------------------------------------

TOOLS = {

    "get_decision":
        get_decision,

    "get_program_decision":
        get_program_decision,

    "get_cycle_summary":
        get_cycle_summary
}


# ----------------------------------------------------------------------------
# Test tools
# ----------------------------------------------------------------------------

print("\n" + "=" * 70)

print("TOOL TESTS")

print("=" * 70)


print(
    "\nBP569:"
)

print(
    get_decision(
        "BP569"
    )
)


print(
    "\nTS660420 / S912:"
)

print(
    get_program_decision(
        "TS660420",
        "S912"
    )
)


print(
    "\nTS6604:"
)

print(
    get_cycle_summary(
        "TS6604"
    )
)


# ============================================================================
# CELL 5 - OPENROUTER CHAT LOOP WITH FUNCTION CALLING
# ============================================================================


# ----------------------------------------------------------------------------
# Install OpenAI-compatible client
# ----------------------------------------------------------------------------

!pip install -q -U openai


# ----------------------------------------------------------------------------
# Imports
# ----------------------------------------------------------------------------

from kaggle_secrets import UserSecretsClient

from openai import OpenAI

import json

import time


# ----------------------------------------------------------------------------
# OpenRouter API key
# ----------------------------------------------------------------------------

api_key = (
    UserSecretsClient()
    .get_secret(
        "OPENROUTER"
    )
)


# ----------------------------------------------------------------------------
# OpenRouter client
# ----------------------------------------------------------------------------

client = OpenAI(

    base_url=
        "https://openrouter.ai/api/v1",

    api_key=
        api_key,

    default_headers={

        "HTTP-Referer":
            "https://www.kaggle.com/",

        "X-Title":
            "Mainframe Decommission Assistant"
    }
)


# ----------------------------------------------------------------------------
# Model
# ----------------------------------------------------------------------------
#
# openrouter/free automatically selects an available free model.
#
# ----------------------------------------------------------------------------

MODEL = "openrouter/free"


# ----------------------------------------------------------------------------
# System prompt
# ----------------------------------------------------------------------------

SYSTEM_PROMPT = (
    "You explain mainframe decommissioning decisions. "
    "Always call a tool to get the facts; never guess a "
    "decision or a job/program name. Cite the blocking jobs "
    "and cycles the tool returns. If a tool says 'not found', "
    "say the data doesn't cover it - don't invent an answer."
)


# ----------------------------------------------------------------------------
# OpenRouter tool declarations
# ----------------------------------------------------------------------------

openrouter_tools = [

    {
        "type":
            "function",

        "function": {

            "name":
                "get_decision",

            "description":
                "Y/N decommission decision and blocking jobs "
                "for one logical file.",

            "parameters": {

                "type":
                    "object",

                "properties": {

                    "logical_file": {

                        "type":
                            "string",

                        "description":
                            "Logical file name, for example BP569."
                    }
                },

                "required": [
                    "logical_file"
                ]
            }
        }
    },


    {
        "type":
            "function",

        "function": {

            "name":
                "get_program_decision",

            "description":
                "Y/N decommission decision for one "
                "job/program step.",

            "parameters": {

                "type":
                    "object",

                "properties": {

                    "job": {

                        "type":
                            "string"
                    },

                    "program": {

                        "type":
                            "string"
                    }
                },

                "required": [
                    "job",
                    "program"
                ]
            }
        }
    },


    {
        "type":
            "function",

        "function": {

            "name":
                "get_cycle_summary",

            "description":
                "Overview of a net/cycle: counts of "
                "blocked files and programs.",

            "parameters": {

                "type":
                    "object",

                "properties": {

                    "cycle": {

                        "type":
                            "string"
                    }
                },

                "required": [
                    "cycle"
                ]
            }
        }
    }
]


# ============================================================================
# OPENROUTER ASK FUNCTION
# ============================================================================

def ask(
    question: str,
    history=None,
    verbose: bool = True
) -> str:

    history = history or []


    # ------------------------------------------------------------------------
    # Build OpenAI-compatible conversation
    # ------------------------------------------------------------------------

    messages = [

        {
            "role":
                "system",

            "content":
                SYSTEM_PROMPT
        }
    ]


    # Add previous history if supplied.
    messages.extend(
        history
    )


    # Add current question.
    messages.append(

        {
            "role":
                "user",

            "content":
                question
        }
    )


    # ------------------------------------------------------------------------
    # Maximum five tool-call rounds.
    # ------------------------------------------------------------------------

    for round_num in range(5):


        # --------------------------------------------------------------------
        # Retry up to three times.
        # --------------------------------------------------------------------

        for attempt in range(3):

            if verbose:

                print(
                    f"  round "
                    f"{round_num + 1}, "
                    f"attempt "
                    f"{attempt + 1}: "
                    f"calling OpenRouter..."
                )


            try:

                resp = (

                    client
                    .chat
                    .completions
                    .create(

                        model=
                            MODEL,

                        messages=
                            messages,

                        tools=
                            openrouter_tools,

                        tool_choice=
                            "auto"
                    )
                )

                break


            except Exception as e:

                error_text = str(e)


                # Retry temporary API errors.
                retryable = (

                    "429" in error_text

                    or
                    "500" in error_text

                    or
                    "502" in error_text

                    or
                    "503" in error_text

                    or
                    "504" in error_text

                    or
                    "timeout" in error_text.lower()

                    or
                    "temporarily" in error_text.lower()

                    or
                    "unavailable" in error_text.lower()
                )


                if not retryable:

                    raise


                if attempt == 2:

                    raise


                wait = 2 ** attempt


                print(
                    f"  OpenRouter busy/unavailable "
                    f"- retrying in {wait}s..."
                )


                time.sleep(
                    wait
                )


        # --------------------------------------------------------------------
        # Get assistant message.
        # --------------------------------------------------------------------

        message = (
            resp
            .choices[0]
            .message
        )


        # --------------------------------------------------------------------
        # Check whether OpenRouter wants to call a tool.
        # --------------------------------------------------------------------

        if message.tool_calls:


            # ----------------------------------------------------------------
            # Add assistant message containing tool calls.
            # ----------------------------------------------------------------

            assistant_message = {

                "role":
                    "assistant",

                "content":
                    message.content or "",

                "tool_calls": []
            }


            for tool_call in message.tool_calls:

                assistant_message[
                    "tool_calls"
                ].append(

                    {

                        "id":
                            tool_call.id,

                        "type":
                            "function",

                        "function": {

                            "name":
                                tool_call.function.name,

                            "arguments":
                                tool_call.function.arguments
                        }
                    }
                )


            messages.append(
                assistant_message
            )


            # ----------------------------------------------------------------
            # Execute each requested tool.
            # ----------------------------------------------------------------

            for tool_call in message.tool_calls:


                fn = (
                    tool_call
                    .function
                    .name
                )


                try:

                    args = json.loads(

                        tool_call
                        .function
                        .arguments
                    )

                except Exception:

                    args = {}


                if verbose:

                    print(
                        f"  -> tool call: "
                        f"{fn}({args})"
                    )


                # ------------------------------------------------------------
                # Run your existing Python tool.
                # ------------------------------------------------------------

                if fn not in TOOLS:

                    result = {

                        "error":
                            f"Unknown tool: {fn}"
                    }

                else:

                    try:

                        result = TOOLS[fn](
                            **args
                        )

                    except Exception as e:

                        result = {

                            "error":
                                str(e)
                        }


                # ------------------------------------------------------------
                # Return result to OpenRouter.
                # ------------------------------------------------------------

                messages.append(

                    {

                        "role":
                            "tool",

                        "tool_call_id":
                            tool_call.id,

                        "content":
                            json.dumps(
                                result,
                                default=str
                            )
                    }
                )


            # Continue the loop so OpenRouter can interpret
            # the tool result.
            continue


        # --------------------------------------------------------------------
        # Normal text response.
        # --------------------------------------------------------------------

        if message.content:

            return message.content


        return (
            "The model returned an empty response."
        )


    return (
        "(stopped after "
        "5 tool calls)"
    )


# ============================================================================
# TEST OPENROUTER
# ============================================================================


print(
    "\n" + "=" * 70
)

print(
    "OPENROUTER TEST"
)

print(
    "=" * 70
)


print(
    ask(
        "Can BP569 be decommissioned?"
    )
)


print(
    ask(
        "What about the TS6604 cycle overall?"
    )
)

# ============================================================================
# CELL 6 - OPTIONAL MODEL LIST
# ============================================================================
#
# If gemini-3.8-flash gives repeated 503 errors, uncomment:
#
# for m in client.models.list():
#     print(m.name)
#
# ============================================================================


# ============================================================================
# EXPECTED INITIAL OUTPUT
# ============================================================================
#
# You should now see something similar to:
#
# Raw IHO shape: (...)
#
# Detected combined IHO header:
#   Header row : 0
#   LOGFILE col: 0
#   JOB col    : 3
#
# ========================================================================
# LOAD SUMMARY
# ========================================================================
# HRO rows       : 449
# IHO files      : 37
# IHO programs   : 37
#
# ========================================================================
# FILE VALIDATION
# ========================================================================
# FILES: 37/37 match
#
# ========================================================================
# PROGRAM VALIDATION
# ========================================================================
# PROGRAMS: .../37 match
#
# ========================================================================
# TOOL TESTS
# ========================================================================
#
# BP569:
# {'logical_file': 'BP569', ...}
#
# TS660420 / S912:
# {'job': 'TS660420', 'program': 'S912', ...}
#
# TS6604:
# {'cycle': 'TS6604', ...}
#
# ============================================================================

/kaggle/input/datasets
/kaggle/input/datasets/aburvan
/kaggle/input/datasets/aburvan/b11ih-1
/kaggle/input/datasets/aburvan/report2
/kaggle/input/datasets/aburvan/report
/kaggle/input/datasets/aburvan/report1
/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx
/kaggle/input/datasets/aburvan/report2/B11HR.xlsx
/kaggle/input/datasets/aburvan/report2/B11IH.xlsx
/kaggle/input/datasets/aburvan/report/B11HR.xlsx
/kaggle/input/datasets/aburvan/report/B11IH.xlsx
/kaggle/input/datasets/aburvan/report1/B11HR.xlsx
/kaggle/input/datasets/aburvan/report1/B11IH.xlsx

Raw IHO shape: (38, 8)

Detected combined IHO header:
  Header row : 0
  LOGFILE col: 0
  JOB col    : 3

LOAD SUMMARY
HRO rows       : 449
IHO files      : 37
IHO programs   : 33

IHO FILES
logfile bestnr decom
  BHW85  BHW85   YES
  BP572  BP572   YES
  BP888  BP888   YES
  BWIH5  BWIH5   YES
  BWIH7  BWIH7   YES
  BWIH8  BWIH8   YES
  BWIH9  BWIH9   YES
  BWIIA  BWIIA   YES
  BWIIB  BWIIB   YES
  BY4JE  BY4JE    NO
  BZ9GU  BZ

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1791504000000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}, {'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free mo

In [ ]:
# ============================================================================
# CELL 6 - systematic LLM evaluation. Paste after Cell 5 in the same notebook
# (reuses iho_files, ext, get_decision, ask, client, etc. from earlier cells).
#
# For every blocked (decom == NO) logical file, asks the assistant directly
# and checks whether its answer:
#   - gets the Y/N decision right
#   - names every blocking job the data actually has (no omissions)
#   - names no job that isn't actually a blocker (no hallucinations)
# ============================================================================
import re

def evaluate_blocked_files():
    blocked = iho_files[iho_files["decom"] == "NO"]["logfile"].tolist()
    rows = []
    for lf in blocked:
        truth = get_decision(lf)
        true_jobs = {d["job"] for d in truth["blocking_dependents"]}

        answer = ask(f"Can {lf} be decommissioned? List every job that blocks it.")
        said_no = bool(re.search(r"\bno\b", answer, re.IGNORECASE))
        # job IDs in this data are 8-char alnum tokens with >=2 digits (TD160425, TS660420);
        # requiring digits rules out plain English words like BLOCKED, CANNOT, READING
        candidates = set(re.findall(r"\b[A-Z][A-Z0-9]{7}\b", answer.upper()))
        mentioned = {c for c in candidates if sum(ch.isdigit() for ch in c) >= 2}
        found_jobs = mentioned & true_jobs

        rows.append({
            "logical_file": lf,
            "decision_correct": said_no,
            "true_blockers": sorted(true_jobs),
            "jobs_cited": sorted(found_jobs),
            "jobs_missed": sorted(true_jobs - found_jobs),
            "jobs_hallucinated": sorted(mentioned - true_jobs - set(blocked)),  # exclude file names it may restate
            "all_cited": found_jobs == true_jobs,
        })
    return pd.DataFrame(rows)


eval_df = evaluate_blocked_files()
pd.set_option("display.max_colwidth", 60)
print(eval_df[["logical_file", "decision_correct", "all_cited", "jobs_missed", "jobs_hallucinated"]]
      .to_string(index=False))

n = len(eval_df)
print(f"\nDecision accuracy:      {eval_df['decision_correct'].sum()}/{n}")
print(f"Full blocker citation:  {eval_df['all_cited'].sum()}/{n}")
print(f"Any hallucinated job:   {(eval_df['jobs_hallucinated'].str.len() > 0).sum()}/{n}")

eval_df.to_excel("/kaggle/working/llm_evaluation.xlsx", index=False)
print("written: /kaggle/working/llm_evaluation.xlsx")

In [16]:
# ============================================================================
# CELL 6 - SYSTEMATIC LLM EVALUATION
#
# Evaluates the LLM against deterministic ground truth from the Python
# dependency engine.
#
# Metrics:
#   1. Decision accuracy
#   2. Blocker recall       - did it mention ALL real blockers?
#   3. Blocker precision    - did it invent blockers?
#   4. Exact blocker match
#   5. Program decision accuracy
#   6. Program blocker recall
#   7. Program blocker precision
#
# Ground truth comes ONLY from:
#   get_decision()
#   get_program_decision()
#
# The LLM is NOT used to create the ground truth.
# ============================================================================

import re
import pandas as pd


# ============================================================================
# HELPERS
# ============================================================================

def extract_job_ids(text):
    """
    Extract job IDs such as:
        TD160425
        TD160401
        TS660420

    from the LLM response.
    """

    candidates = set(
        re.findall(
            r"\b[A-Z][A-Z0-9]{7}\b",
            text.upper()
        )
    )

    return {
        x
        for x in candidates
        if sum(ch.isdigit() for ch in x) >= 2
    }


def extract_program_refs(text):
    """
    Extract job/program combinations such as:

        TS660420/S912
        TS660420 - S912
        TS660420 S912
    """

    text = text.upper()

    refs = set()

    pattern = (
        r"\b([A-Z][A-Z0-9]{7})"
        r"\s*(?:/|-|\s)\s*"
        r"([A-Z][A-Z0-9]{2,5})\b"
    )

    for job, program in re.findall(
        pattern,
        text
    ):
        refs.add(
            f"{job}/{program}"
        )

    return refs


def answer_says_no(answer):
    """
    Determine whether the LLM says the item cannot be decommissioned.
    """

    negative_patterns = [

        r"\bcannot\b",
        r"\bcan't\b",
        r"\bcan not\b",
        r"\bno\b",
        r"\bnot be decommissioned\b",
        r"\bshould not be decommissioned\b",
        r"\bblocked\b"
    ]

    return any(
        re.search(
            p,
            answer,
            re.IGNORECASE
        )
        for p in negative_patterns
    )


# ============================================================================
# FILE-LEVEL EVALUATION
# ============================================================================

def evaluate_blocked_files():

    blocked = (

        iho_files[
            iho_files["decom"]
            .astype(str)
            .str.upper()
            .eq("NO")
        ]["logfile"]
        .tolist()
    )

    rows = []


    for logical_file in blocked:

        # --------------------------------------------------------------------
        # Ground truth
        # --------------------------------------------------------------------

        truth = get_decision(
            logical_file
        )

        true_jobs = {

            d["job"]

            for d in truth[
                "blocking_dependents"
            ]
        }


        # --------------------------------------------------------------------
        # LLM answer
        # --------------------------------------------------------------------

        answer = ask(

            f"Can {logical_file} be "
            f"decommissioned? "
            f"List every job that blocks it."
        )


        answer = answer or ""


        # --------------------------------------------------------------------
        # Extract LLM claims
        # --------------------------------------------------------------------

        said_no = answer_says_no(
            answer
        )

        mentioned_jobs = extract_job_ids(
            answer
        )


        # Don't treat the logical file itself as a job.
        mentioned_jobs.discard(
            logical_file.upper()
        )


        found_jobs = (
            mentioned_jobs
            &
            true_jobs
        )


        hallucinated_jobs = (
            mentioned_jobs
            -
            true_jobs
        )


        missed_jobs = (
            true_jobs
            -
            mentioned_jobs
        )


        # --------------------------------------------------------------------
        # Metrics
        # --------------------------------------------------------------------

        recall = (

            len(found_jobs)
            /
            len(true_jobs)

            if true_jobs

            else 1.0
        )


        precision = (

            len(found_jobs)
            /
            len(mentioned_jobs)

            if mentioned_jobs

            else 0.0
        )


        exact_match = (
            mentioned_jobs
            ==
            true_jobs
        )


        rows.append({

            "logical_file":
                logical_file,

            "decision_correct":
                said_no,

            "true_blockers":
                sorted(true_jobs),

            "jobs_cited":
                sorted(found_jobs),

            "jobs_missed":
                sorted(missed_jobs),

            "jobs_hallucinated":
                sorted(hallucinated_jobs),

            "blocker_recall":
                round(recall, 3),

            "blocker_precision":
                round(precision, 3),

            "exact_blocker_match":
                exact_match,

            "answer":
                answer
        })


    return pd.DataFrame(
        rows
    )


# ============================================================================
# PROGRAM-LEVEL EVALUATION
# ============================================================================

def evaluate_blocked_programs():

    blocked_programs = (

        iho_programs[
            iho_programs["decom"]
            .astype(str)
            .str.upper()
            .eq("NO")
        ][
            ["job", "program"]
        ]
        .drop_duplicates()
    )


    rows = []


    for _, row in blocked_programs.iterrows():

        job = str(
            row["job"]
        ).strip()

        program = str(
            row["program"]
        ).strip()


        # --------------------------------------------------------------------
        # Ground truth
        # --------------------------------------------------------------------

        truth = get_program_decision(
            job,
            program
        )


        true_blockers = set(
            truth.get(
                "blocking_files",
                []
            )
        )


        # --------------------------------------------------------------------
        # LLM question
        # --------------------------------------------------------------------

        answer = ask(

            f"Can {job}/{program} "
            f"be decommissioned? "
            f"List every file that blocks it."
        )


        answer = answer or ""


        # --------------------------------------------------------------------
        # Decision
        # --------------------------------------------------------------------

        said_no = answer_says_no(
            answer
        )


        # --------------------------------------------------------------------
        # Extract file IDs
        # --------------------------------------------------------------------

        mentioned_files = extract_job_ids(
            answer
        )


        # Some file IDs such as BP569 or BY4JE
        # do not match the 8-character job pattern.
        #
        # Therefore also extract known logical files
        # directly from the IHO data.

        known_files = set(
            iho_files[
                "logfile"
            ]
            .astype(str)
            .str.upper()
        )


        answer_upper = answer.upper()


        mentioned_files |= {

            lf

            for lf in known_files

            if re.search(
                rf"\b{re.escape(lf)}\b",
                answer_upper
            )
        }


        found_files = (
            mentioned_files
            &
            true_blockers
        )


        hallucinated_files = (
            mentioned_files
            -
            true_blockers
        )


        missed_files = (
            true_blockers
            -
            mentioned_files
        )


        recall = (

            len(found_files)
            /
            len(true_blockers)

            if true_blockers

            else 1.0
        )


        precision = (

            len(found_files)
            /
            len(mentioned_files)

            if mentioned_files

            else 0.0
        )


        exact_match = (
            mentioned_files
            ==
            true_blockers
        )


        rows.append({

            "job":
                job,

            "program":
                program,

            "decision_correct":
                said_no,

            "true_blockers":
                sorted(true_blockers),

            "files_cited":
                sorted(found_files),

            "files_missed":
                sorted(missed_files),

            "files_hallucinated":
                sorted(hallucinated_files),

            "blocker_recall":
                round(recall, 3),

            "blocker_precision":
                round(precision, 3),

            "exact_blocker_match":
                exact_match,

            "answer":
                answer
        })


    return pd.DataFrame(
        rows
    )


# ============================================================================
# RUN FILE EVALUATION
# ============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "FILE-LEVEL LLM EVALUATION"
)

print(
    "=" * 80
)


file_eval_df = evaluate_blocked_files()


if len(file_eval_df) > 0:

    print(

        file_eval_df[
            [
                "logical_file",
                "decision_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_blocker_match",
                "jobs_missed",
                "jobs_hallucinated"
            ]
        ]
        .to_string(
            index=False
        )
    )


    n = len(
        file_eval_df
    )


    print(
        f"\nFiles evaluated: "
        f"{n}"
    )


    print(
        f"Decision accuracy: "
        f"{file_eval_df['decision_correct'].mean():.2%}"
    )


    print(
        f"Average blocker recall: "
        f"{file_eval_df['blocker_recall'].mean():.2%}"
    )


    print(
        f"Average blocker precision: "
        f"{file_eval_df['blocker_precision'].mean():.2%}"
    )


    print(
        f"Exact blocker match: "
        f"{file_eval_df['exact_blocker_match'].mean():.2%}"
    )


    print(
        "Hallucinated blocker cases: "
        f"{(file_eval_df['jobs_hallucinated'].str.len() > 0).sum()}"
        f"/{n}"
    )


else:

    print(
        "No blocked files found."
    )


# ============================================================================
# RUN PROGRAM EVALUATION
# ============================================================================

print(
    "\n"
    + "=" * 80
)

print(
    "PROGRAM-LEVEL LLM EVALUATION"
)

print(
    "=" * 80
)


program_eval_df = (
    evaluate_blocked_programs()
)


if len(program_eval_df) > 0:

    print(

        program_eval_df[
            [
                "job",
                "program",
                "decision_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_blocker_match",
                "files_missed",
                "files_hallucinated"
            ]
        ]
        .to_string(
            index=False
        )
    )


    n_programs = len(
        program_eval_df
    )


    print(
        f"\nPrograms evaluated: "
        f"{n_programs}"
    )


    print(
        f"Decision accuracy: "
        f"{program_eval_df['decision_correct'].mean():.2%}"
    )


    print(
        f"Average blocker recall: "
        f"{program_eval_df['blocker_recall'].mean():.2%}"
    )


    print(
        f"Average blocker precision: "
        f"{program_eval_df['blocker_precision'].mean():.2%}"
    )


    print(
        f"Exact blocker match: "
        f"{program_eval_df['exact_blocker_match'].mean():.2%}"
    )


    print(
        "Hallucinated blocker cases: "
        f"{(program_eval_df['files_hallucinated'].str.len() > 0).sum()}"
        f"/{n_programs}"
    )


else:

    print(
        "No blocked programs found."
    )


# ============================================================================
# SAVE RESULTS
# ============================================================================

output_path = (
    "/kaggle/working/"
    "llm_evaluation.xlsx"
)


with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    file_eval_df.to_excel(
        writer,
        sheet_name="File Evaluation",
        index=False
    )

    program_eval_df.to_excel(
        writer,
        sheet_name="Program Evaluation",
        index=False
    )


print(
    "\n"
    + "=" * 80
)

print(
    f"Evaluation written to:\n"
    f"{output_path}"
)

print(
    "=" * 80
)


FILE-LEVEL LLM EVALUATION
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY4JE'})
  round 2, attempt 1: calling OpenRouter...
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY1EH'})
  round 2, attempt 1: calling OpenRouter...
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY1FJ'})
  round 2, attempt 1: calling OpenRouter...
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY4JF'})
  round 2, attempt 1: calling OpenRouter...
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'VH14#1'})
  round 2, attempt 1: calling OpenRouter...
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BP569'})
  round 2, attempt 1: calling OpenRouter...
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'JG0Q#1'})
  round

In [5]:
# ============================================================================
# CELL 6 — COMPLETE LLM / AGENT EVALUATION
#
# Evaluates:
#   1. FILE-LEVEL decommissioning questions
#   2. PROGRAM-LEVEL decommissioning questions
#
# Metrics:
#   - Decision accuracy
#   - Blocker recall
#   - Blocker precision
#   - Exact blocker match
#   - Hallucinated blockers
#
# Requires from previous cells:
#   iho_files
#   iho_programs
#   get_decision()
#   get_program_decision()
#   ask()
#
# ============================================================================

import re
import pandas as pd
import numpy as np


# ============================================================================
# 1. HELPER — NORMALIZE TEXT
# ============================================================================

def normalize_text(value):
    if value is None:
        return ""

    if pd.isna(value):
        return ""

    return str(value).strip().upper()


# ============================================================================
# 2. HELPER — DETECT "NO" DECISION
# ============================================================================

def answer_says_no(answer):
    """
    Detect whether the LLM says that the item cannot / should not
    be decommissioned.

    We deliberately look for explicit negative phrases rather than
    simply checking whether the word 'no' appears anywhere.
    """

    if answer is None:
        return False

    answer = str(answer).strip()

    if not answer:
        return False

    negative_patterns = [
        r"\bcannot be decommissioned\b",
        r"\bcan not be decommissioned\b",
        r"\bcan't be decommissioned\b",
        r"\bshould not be decommissioned\b",
        r"\bshouldn't be decommissioned\b",
        r"\bnot be decommissioned\b",
        r"\bnot safe to decommission\b",
        r"\bnot eligible for decommission\b",
        r"\bnot eligible to be decommissioned\b",
        r"\bblocked from decommission\b",
        r"\bblocked from decommissioning\b",
        r"\bdecommissioning is blocked\b",
        r"\bdecommission is blocked\b",
        r"\bcannot be removed\b",
        r"\bnot safe to remove\b",
        r"\bdecommission.*\bno\b",
        r"^\s*no[\s,.:!-]"
    ]

    for pattern in negative_patterns:
        if re.search(pattern, answer, re.IGNORECASE):
            return True

    return False


# ============================================================================
# 3. HELPER — EXTRACT FILE NAMES FROM LLM ANSWER
# ============================================================================

def extract_file_candidates(answer):
    """
    Extract likely logical-file identifiers from an LLM answer.

    Examples:
        BY4JE
        BP569
        VH14#1
        JG0Q#1
        BXV03

    We compare against the actual IHO file list, so ordinary English
    words are not treated as valid blockers.
    """

    if answer is None:
        return set()

    answer = str(answer).upper()

    # Get the authoritative file names from IHO.
    valid_files = set(
        normalize_text(x)
        for x in iho_files["logfile"].tolist()
        if normalize_text(x)
    )

    # Extract alphanumeric identifiers with optional # suffix.
    candidates = set(
        re.findall(
            r"\b[A-Z][A-Z0-9]*(?:#[A-Z0-9]+)?\b",
            answer
        )
    )

    return candidates & valid_files


# ============================================================================
# 4. HELPER — EXTRACT JOB IDs
# ============================================================================

def extract_job_candidates(answer):
    """
    Extract job IDs such as:
        TS660420
        TD160425
        TS660451
    """

    if answer is None:
        return set()

    answer = str(answer).upper()

    return set(
        re.findall(
            r"\b[A-Z]{2}\d{6}\b",
            answer
        )
    )


# ============================================================================
# 5. FILE-LEVEL EVALUATION
# ============================================================================

def evaluate_blocked_files():

    blocked_files = (
        iho_files[
            iho_files["decom"].apply(normalize_text) == "NO"
        ]["logfile"]
        .tolist()
    )

    rows = []

    print("=" * 80)
    print("FILE-LEVEL LLM EVALUATION")
    print("=" * 80)

    for logical_file in blocked_files:

        logical_file = normalize_text(logical_file)

        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        truth = get_decision(logical_file)

        true_jobs = {
            normalize_text(d.get("job"))
            for d in truth.get("blocking_dependents", [])
            if normalize_text(d.get("job"))
        }

        # ------------------------------------------------------------
        # Ask agent
        # ------------------------------------------------------------

        print(
            f"\nEvaluating file: {logical_file}"
        )

        answer = ask(
            f"Can {logical_file} be decommissioned? "
            f"List every job that blocks it."
        )

        # ------------------------------------------------------------
        # Decision
        # ------------------------------------------------------------

        said_no = answer_says_no(answer)

        decision_correct = said_no

        # ------------------------------------------------------------
        # Extract jobs
        # ------------------------------------------------------------

        mentioned_jobs = extract_job_candidates(answer)

        found_jobs = mentioned_jobs & true_jobs

        missed_jobs = true_jobs - mentioned_jobs

        hallucinated_jobs = (
            mentioned_jobs - true_jobs
        )

        # ------------------------------------------------------------
        # Metrics
        # ------------------------------------------------------------

        if len(true_jobs) == 0:

            blocker_recall = 1.0

        else:

            blocker_recall = (
                len(found_jobs) /
                len(true_jobs)
            )

        if len(mentioned_jobs) == 0:

            blocker_precision = (
                1.0 if len(true_jobs) == 0
                else 0.0
            )

        else:

            blocker_precision = (
                len(found_jobs) /
                len(mentioned_jobs)
            )

        exact_match = (
            found_jobs == true_jobs
            and mentioned_jobs == true_jobs
        )

        rows.append({

            "logical_file":
                logical_file,

            "decision_correct":
                decision_correct,

            "blocker_recall":
                blocker_recall,

            "blocker_precision":
                blocker_precision,

            "exact_blocker_match":
                exact_match,

            "true_blockers":
                sorted(true_jobs),

            "jobs_cited":
                sorted(mentioned_jobs),

            "jobs_found":
                sorted(found_jobs),

            "jobs_missed":
                sorted(missed_jobs),

            "jobs_hallucinated":
                sorted(hallucinated_jobs),

            "llm_answer":
                str(answer)

        })

    return pd.DataFrame(rows)


# ============================================================================
# 6. PROGRAM-LEVEL EVALUATION
# ============================================================================

def evaluate_blocked_programs():

    blocked_programs = (
        iho_programs[
            iho_programs["decom"].apply(normalize_text) == "NO"
        ][
            ["job", "program"]
        ]
        .drop_duplicates()
    )

    rows = []

    print("\n")
    print("=" * 80)
    print("PROGRAM-LEVEL LLM EVALUATION")
    print("=" * 80)

    for _, row in blocked_programs.iterrows():

        job = normalize_text(row["job"])
        program = normalize_text(row["program"])

        print(
            f"\nEvaluating program: {job}/{program}"
        )

        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        truth = get_program_decision(
            job,
            program
        )

        true_files = {
            normalize_text(x)
            for x in truth.get("blocking_files", [])
            if normalize_text(x)
        }

        # ------------------------------------------------------------
        # Ask agent
        # ------------------------------------------------------------

        answer = ask(
            f"Can {job}/{program} be decommissioned? "
            f"List every file that blocks it."
        )

        # ------------------------------------------------------------
        # Decision
        # ------------------------------------------------------------

        said_no = answer_says_no(answer)

        true_decom = (
            normalize_text(
                truth.get("decom", "NO")
            )
        )

        expected_no = (
            true_decom == "NO"
        )

        decision_correct = (
            said_no == expected_no
        )

        # ------------------------------------------------------------
        # Extract file blockers
        # ------------------------------------------------------------

        mentioned_files = extract_file_candidates(
            answer
        )

        found_files = (
            mentioned_files & true_files
        )

        missed_files = (
            true_files - mentioned_files
        )

        hallucinated_files = (
            mentioned_files - true_files
        )

        # ------------------------------------------------------------
        # Metrics
        # ------------------------------------------------------------

        if len(true_files) == 0:

            blocker_recall = 1.0

        else:

            blocker_recall = (
                len(found_files) /
                len(true_files)
            )

        if len(mentioned_files) == 0:

            blocker_precision = (
                1.0 if len(true_files) == 0
                else 0.0
            )

        else:

            blocker_precision = (
                len(found_files) /
                len(mentioned_files)
            )

        exact_match = (
            found_files == true_files
            and mentioned_files == true_files
        )

        rows.append({

            "job":
                job,

            "program":
                program,

            "decision_correct":
                decision_correct,

            "blocker_recall":
                blocker_recall,

            "blocker_precision":
                blocker_precision,

            "exact_blocker_match":
                exact_match,

            "true_blockers":
                sorted(true_files),

            "files_cited":
                sorted(mentioned_files),

            "files_found":
                sorted(found_files),

            "files_missed":
                sorted(missed_files),

            "files_hallucinated":
                sorted(hallucinated_files),

            "llm_answer":
                str(answer)

        })

    return pd.DataFrame(rows)


# ============================================================================
# 7. RUN FILE EVALUATION
# ============================================================================

eval_df = evaluate_blocked_files()

print("\n")
print("=" * 80)
print("FILE-LEVEL RESULTS")
print("=" * 80)

if len(eval_df) > 0:

    print(
        eval_df[
            [
                "logical_file",
                "decision_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_blocker_match",
                "jobs_missed",
                "jobs_hallucinated"
            ]
        ].to_string(index=False)
    )

    n_files = len(eval_df)

    print("\nFiles evaluated:", n_files)

    print(
        f"Decision accuracy: "
        f"{eval_df['decision_correct'].mean():.2%}"
    )

    print(
        f"Average blocker recall: "
        f"{eval_df['blocker_recall'].mean():.2%}"
    )

    print(
        f"Average blocker precision: "
        f"{eval_df['blocker_precision'].mean():.2%}"
    )

    print(
        f"Exact blocker match: "
        f"{eval_df['exact_blocker_match'].mean():.2%}"
    )

    hallucinated = (
        eval_df["jobs_hallucinated"]
        .apply(len)
        .gt(0)
        .sum()
    )

    print(
        f"Hallucinated blocker cases: "
        f"{hallucinated}/{n_files}"
    )

else:

    print("No blocked files found.")


# ============================================================================
# 8. RUN PROGRAM EVALUATION
# ============================================================================

program_eval_df = evaluate_blocked_programs()

print("\n")
print("=" * 80)
print("PROGRAM-LEVEL RESULTS")
print("=" * 80)

if len(program_eval_df) > 0:

    print(
        program_eval_df[
            [
                "job",
                "program",
                "decision_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_blocker_match",
                "files_missed",
                "files_hallucinated"
            ]
        ].to_string(index=False)
    )

    n_programs = len(
        program_eval_df
    )

    print(
        f"\nPrograms evaluated: "
        f"{n_programs}"
    )

    print(
        f"Decision accuracy: "
        f"{program_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        f"Average blocker recall: "
        f"{program_eval_df['blocker_recall'].mean():.2%}"
    )

    print(
        f"Average blocker precision: "
        f"{program_eval_df['blocker_precision'].mean():.2%}"
    )

    print(
        f"Exact blocker match: "
        f"{program_eval_df['exact_blocker_match'].mean():.2%}"
    )

    hallucinated = (
        program_eval_df["files_hallucinated"]
        .apply(len)
        .gt(0)
        .sum()
    )

    print(
        f"Hallucinated blocker cases: "
        f"{hallucinated}/{n_programs}"
    )

else:

    print("No blocked programs found.")


# ============================================================================
# 9. SAVE COMPLETE EVALUATION
# ============================================================================

output_path = "/kaggle/working/llm_evaluation.xlsx"

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    eval_df.to_excel(
        writer,
        sheet_name="File Evaluation",
        index=False
    )

    program_eval_df.to_excel(
        writer,
        sheet_name="Program Evaluation",
        index=False
    )

print("\n")
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)
print(
    f"Written to: {output_path}"
)

FILE-LEVEL LLM EVALUATION

Evaluating file: BY4JE
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY4JE'})
  round 2, attempt 1: calling OpenRouter...

Evaluating file: BY1EH
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY1EH'})
  round 2, attempt 1: calling OpenRouter...

Evaluating file: BY1FJ
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY1FJ'})
  round 2, attempt 1: calling OpenRouter...

Evaluating file: BY4JF
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'BY4JF'})
  round 2, attempt 1: calling OpenRouter...

Evaluating file: VH14#1
  round 1, attempt 1: calling OpenRouter...
  -> tool call: get_decision({'logical_file': 'VH14#1'})
  round 2, attempt 1: calling OpenRouter...
  -> tool call: get_cycle_summary({'cycle': 'TH4614'})
  round 3, attempt 1: calling OpenRouter...

Evaluating file: BP569
  round 1

In [7]:
# ============================================================================
# GROQ AGENT + TOOL CALLING
# Full replacement for the OpenRouter version
# ============================================================================

# If needed, run once in Kaggle:
!pip install -q -U openai


# ============================================================================
# 1. IMPORTS
# ============================================================================

import os
import json
import re
import time
import pandas as pd

from openai import OpenAI


# ============================================================================
# 2. GROQ CONFIGURATION
# ============================================================================

# ------------------------------------------------------------
# PUT YOUR GROQ API KEY HERE
# ------------------------------------------------------------

GROQ_API_KEY = "YOUR_GROQ_API_KEY"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

# Good general-purpose model for tool calling
MODEL = "llama-3.3-70b-versatile"

print("=" * 80)
print("GROQ CONFIGURATION")
print("=" * 80)
print("Model:", MODEL)
print("Client: Groq")
print("=" * 80)


# ============================================================================
# 3. SYSTEM PROMPT
# ============================================================================

SYSTEM_PROMPT = """
You are an AI assistant for mainframe batch-cycle decommission analysis.

Your job is to determine whether logical files, programs, or cycles can
safely be decommissioned.

IMPORTANT RULES:

1. NEVER invent dependency information.

2. Use the available deterministic tools whenever the question requires
   information about files, programs, blockers, or cycles.

3. The deterministic tools are the source of truth.

4. If a file has decom = NO, it cannot currently be decommissioned.

5. If a program has decom = NO, it cannot currently be decommissioned.

6. Always identify the actual blocking files or jobs returned by the tools.

7. Distinguish clearly between:
   - logical files
   - programs
   - jobs
   - cycles
   - blocking dependencies

8. When explaining a decision, provide the evidence returned by the tool.

9. Do not claim a dependency exists unless it is present in the tool result.

10. If the tool does not contain enough information, say that the available
    evidence is insufficient.

11. Be concise but explain the reasoning clearly.

12. For a decommission question, give:
    - Decision: YES or NO
    - Evidence
    - Blocking dependencies, if any
    - Short explanation
"""


# ============================================================================
# 4. TOOL DEFINITIONS
# ============================================================================

TOOLS = [

    # ------------------------------------------------------------------------
    # FILE DECISION TOOL
    # ------------------------------------------------------------------------

    {
        "type": "function",
        "function": {
            "name": "get_decision",
            "description": (
                "Get the deterministic decommission decision for a logical "
                "file, including whether it can be decommissioned and the "
                "jobs that block it."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "logical_file": {
                        "type": "string",
                        "description": "Logical file name"
                    }
                },
                "required": ["logical_file"]
            }
        }
    },

    # ------------------------------------------------------------------------
    # PROGRAM DECISION TOOL
    # ------------------------------------------------------------------------

    {
        "type": "function",
        "function": {
            "name": "get_program_decision",
            "description": (
                "Get the deterministic decommission decision for a specific "
                "job/program combination, including blocking files."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "job": {
                        "type": "string",
                        "description": "Mainframe job name"
                    },
                    "program": {
                        "type": "string",
                        "description": "Program name"
                    }
                },
                "required": ["job", "program"]
            }
        }
    },

    # ------------------------------------------------------------------------
    # CYCLE SUMMARY TOOL
    # ------------------------------------------------------------------------

    {
        "type": "function",
        "function": {
            "name": "get_cycle_summary",
            "description": (
                "Get a deterministic summary of a batch cycle, including "
                "total files, blocked files, programs, and blocked programs."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "cycle": {
                        "type": "string",
                        "description": "Batch cycle name"
                    }
                },
                "required": ["cycle"]
            }
        }
    }
]


# ============================================================================
# 5. TOOL EXECUTOR
# ============================================================================

def execute_tool(tool_name, tool_args):
    """
    Executes the deterministic tools used by the agent.
    """

    try:

        # ------------------------------------------------------------
        # FILE DECISION
        # ------------------------------------------------------------

        if tool_name == "get_decision":

            logical_file = tool_args["logical_file"]

            return get_decision(logical_file)


        # ------------------------------------------------------------
        # PROGRAM DECISION
        # ------------------------------------------------------------

        elif tool_name == "get_program_decision":

            job = tool_args["job"]
            program = tool_args["program"]

            return get_program_decision(
                job,
                program
            )


        # ------------------------------------------------------------
        # CYCLE SUMMARY
        # ------------------------------------------------------------

        elif tool_name == "get_cycle_summary":

            cycle = tool_args["cycle"]

            return get_cycle_summary(cycle)


        # ------------------------------------------------------------
        # UNKNOWN TOOL
        # ------------------------------------------------------------

        else:

            return {
                "error": f"Unknown tool: {tool_name}"
            }

    except Exception as e:

        return {
            "error": str(e)
        }


# ============================================================================
# 6. AGENT / ASK FUNCTION
# ============================================================================

def ask(
    question,
    history=None,
    verbose=True,
    max_rounds=6
):
    """
    Main Groq-powered agent.

    Flow:

        User question
              ↓
           Groq LLM
              ↓
          Tool call
              ↓
       Deterministic tool
              ↓
       Evidence returned
              ↓
           Groq LLM
              ↓
        Final answer
    """

    if history is None:
        history = []

    # ------------------------------------------------------------
    # Initial messages
    # ------------------------------------------------------------

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]

    # Add previous conversation
    if history:
        messages.extend(history)

    # Add current question
    messages.append(
        {
            "role": "user",
            "content": question
        }
    )

    # ------------------------------------------------------------
    # Agent loop
    # ------------------------------------------------------------

    for round_no in range(1, max_rounds + 1):

        if verbose:

            print(
                f"  round {round_no}, "
                f"attempt 1: calling Groq..."
            )

        try:

            response = client.chat.completions.create(

                model=MODEL,

                messages=messages,

                tools=TOOLS,

                tool_choice="auto",

                temperature=0.1,

                max_tokens=1500
            )

        except Exception as e:

            print("\nGroq API error:")
            print(e)

            return (
                "The LLM request failed. "
                "The deterministic tools are still available. "
                f"Error: {e}"
            )

        # ------------------------------------------------------------
        # Extract assistant message
        # ------------------------------------------------------------

        message = response.choices[0].message

        # ------------------------------------------------------------
        # No tool call = final answer
        # ------------------------------------------------------------

        if not message.tool_calls:

            answer = message.content or ""

            if verbose:
                print("  -> final answer generated")

            return answer

        # ------------------------------------------------------------
        # Add assistant tool-call message
        # ------------------------------------------------------------

        messages.append(message)

        # ------------------------------------------------------------
        # Execute every requested tool
        # ------------------------------------------------------------

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name

            try:

                tool_args = json.loads(
                    tool_call.function.arguments
                )

            except Exception:

                tool_args = {}

            if verbose:

                print(
                    f"  -> tool call: "
                    f"{tool_name}({tool_args})"
                )

            # --------------------------------------------------------
            # Execute deterministic tool
            # --------------------------------------------------------

            result = execute_tool(
                tool_name,
                tool_args
            )

            # --------------------------------------------------------
            # Convert result to JSON
            # --------------------------------------------------------

            result_json = json.dumps(
                result,
                indent=2,
                default=str
            )

            # --------------------------------------------------------
            # Send tool result back to LLM
            # --------------------------------------------------------

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result_json
                }
            )

    # ------------------------------------------------------------
    # Maximum tool rounds reached
    # ------------------------------------------------------------

    return (
        "The agent reached the maximum number of tool-calling "
        "rounds without producing a final answer."
    )


# ============================================================================
# 7. TEST FILE-LEVEL DECISION
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL AGENT TEST")
print("=" * 80)

answer = ask(
    "Can BP569 be decommissioned? "
    "List every job that blocks it and explain why."
)

print("\nANSWER:")
print(answer)


# ============================================================================
# 8. TEST PROGRAM-LEVEL DECISION
# ============================================================================

print("\n")
print("=" * 80)
print("PROGRAM-LEVEL AGENT TEST")
print("=" * 80)

answer = ask(
    "Can program S912 in job TS660420 be decommissioned? "
    "Give the decision and all blocking files with evidence."
)

print("\nANSWER:")
print(answer)


# ============================================================================
# 9. TEST CYCLE-LEVEL DECISION
# ============================================================================

print("\n")
print("=" * 80)
print("CYCLE-LEVEL AGENT TEST")
print("=" * 80)

answer = ask(
    "Give me the decommission status of cycle TS6604. "
    "List all blocked files and blocked programs."
)

print("\nANSWER:")
print(answer)

GROQ CONFIGURATION
Model: llama-3.3-70b-versatile
Client: Groq


FILE-LEVEL AGENT TEST
  round 1, attempt 1: calling Groq...

Groq API error:
Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

ANSWER:
The LLM request failed. The deterministic tools are still available. Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


PROGRAM-LEVEL AGENT TEST
  round 1, attempt 1: calling Groq...

Groq API error:
Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}

ANSWER:
The LLM request failed. The deterministic tools are still available. Error: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}


CYCLE-LEVEL AGENT TEST
  round 1, attempt 1: calling Groq...

Groq API error:
Error code: 401 - {'error': {'message': 'Inval

In [2]:
# ============================================================================
# MAIN AGENT - GROQ + KAGGLE SECRET
# ============================================================================
# Uses:
#   - Kaggle Secret: "groq"
#   - Groq API
#   - Llama 3.3 70B
#   - Existing deterministic tools:
#         get_decision()
#         get_program_decision()
#         get_cycle_summary()
#
# IMPORTANT:
# Your existing parser/data-loading code should remain ABOVE this cell.
# This cell assumes those functions already exist.
# ============================================================================


# ============================================================================
# 1. INSTALL / IMPORTS
# ============================================================================

!pip install -q -U openai

import json
import re
import time
import pandas as pd

from openai import OpenAI
from kaggle_secrets import UserSecretsClient


# ============================================================================
# 2. LOAD GROQ API KEY FROM KAGGLE SECRET
# ============================================================================

user_secrets = UserSecretsClient()

try:
    GROQ_API_KEY = user_secrets.get_secret("groq")
except Exception as e:
    raise RuntimeError(
        "\nCould not read the Kaggle secret 'groq'.\n"
        "Go to Kaggle -> Add-ons -> Secrets and make sure:\n"
        "  Label = groq\n"
        "  Value = your Groq API key\n"
    ) from e


# ============================================================================
# 3. CREATE GROQ CLIENT
# ============================================================================

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-120b"


print("=" * 80)
print("GROQ CONFIGURATION")
print("=" * 80)
print("Model :", MODEL)
print("Client: Groq")
print("Key   :", GROQ_API_KEY[:8] + "...")
print("=" * 80)


# ============================================================================
# 4. SYSTEM PROMPT
# ============================================================================

SYSTEM_PROMPT = """
You are an AI assistant for mainframe batch-cycle decommission analysis.

Your task is to analyze whether files, programs, and batch cycles can be
decommissioned using evidence from deterministic tools.

IMPORTANT RULES:

1. NEVER invent dependencies.

2. NEVER guess a blocking file, job, or program.

3. The deterministic tools are the source of truth.

4. Always use a tool when the user asks about:
   - a logical file
   - a program
   - a job/program combination
   - a batch cycle
   - decommission status
   - blocking dependencies

5. For logical files, use get_decision().

6. For job/program combinations, use get_program_decision().

7. For complete cycles, use get_cycle_summary().

8. If the deterministic tool says decom = NO, the entity cannot currently
   be decommissioned.

9. If decom = YES, the entity can be considered eligible for
   decommissioning based on the available evidence.

10. When reporting blockers, only report blockers explicitly returned by
    the deterministic tool.

11. Do not confuse:
    - logical file names
    - job names
    - program names
    - cycles

12. If evidence is insufficient, explicitly say so.

13. Keep the final answer concise and evidence-backed.

14. For a decommission question, structure the answer as:

    Decision: YES/NO

    Evidence:
    - relevant deterministic evidence

    Blockers:
    - exact blocking dependencies

    Explanation:
    - short explanation of why the decision follows from the evidence
"""


# ============================================================================
# 5. TOOL DEFINITIONS
# ============================================================================

TOOLS = [

    # ------------------------------------------------------------------------
    # FILE DECISION
    # ------------------------------------------------------------------------

    {
        "type": "function",
        "function": {
            "name": "get_decision",
            "description": (
                "Get the deterministic decommission decision for a logical "
                "file. Returns the decom status and all known blocking "
                "dependent jobs."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "logical_file": {
                        "type": "string",
                        "description": "Logical file name"
                    }
                },
                "required": ["logical_file"]
            }
        }
    },


    # ------------------------------------------------------------------------
    # PROGRAM DECISION
    # ------------------------------------------------------------------------

    {
        "type": "function",
        "function": {
            "name": "get_program_decision",
            "description": (
                "Get the deterministic decommission decision for a specific "
                "job and program. Returns program status and blocking files."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "job": {
                        "type": "string",
                        "description": "Mainframe job name"
                    },
                    "program": {
                        "type": "string",
                        "description": "Program name"
                    }
                },
                "required": [
                    "job",
                    "program"
                ]
            }
        }
    },


    # ------------------------------------------------------------------------
    # CYCLE SUMMARY
    # ------------------------------------------------------------------------

    {
        "type": "function",
        "function": {
            "name": "get_cycle_summary",
            "description": (
                "Get the deterministic decommission summary for a batch "
                "cycle, including total files, blocked files, programs, "
                "and blocked programs."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "cycle": {
                        "type": "string",
                        "description": "Batch cycle name"
                    }
                },
                "required": [
                    "cycle"
                ]
            }
        }
    }
]


# ============================================================================
# 6. TOOL EXECUTOR
# ============================================================================

def execute_tool(tool_name, tool_args):

    try:

        # ------------------------------------------------------------
        # FILE
        # ------------------------------------------------------------

        if tool_name == "get_decision":

            logical_file = tool_args["logical_file"]

            result = get_decision(
                logical_file
            )

            return result


        # ------------------------------------------------------------
        # PROGRAM
        # ------------------------------------------------------------

        elif tool_name == "get_program_decision":

            job = tool_args["job"]
            program = tool_args["program"]

            result = get_program_decision(
                job,
                program
            )

            return result


        # ------------------------------------------------------------
        # CYCLE
        # ------------------------------------------------------------

        elif tool_name == "get_cycle_summary":

            cycle = tool_args["cycle"]

            result = get_cycle_summary(
                cycle
            )

            return result


        # ------------------------------------------------------------
        # UNKNOWN TOOL
        # ------------------------------------------------------------

        else:

            return {
                "error": f"Unknown tool: {tool_name}"
            }


    except Exception as e:

        return {
            "error": str(e)
        }


# ============================================================================
# 7. MAIN AGENT
# ============================================================================

def ask(
    question,
    history=None,
    verbose=True,
    max_rounds=6
):

    """
    Main agent loop.

    User
      ↓
    Groq LLM
      ↓
    Tool selection
      ↓
    Deterministic tool
      ↓
    Evidence
      ↓
    Groq LLM
      ↓
    Final answer
    """

    if history is None:
        history = []


    # ------------------------------------------------------------
    # Build conversation
    # ------------------------------------------------------------

    messages = [

        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }

    ]


    # Previous conversation
    if history:

        messages.extend(
            history
        )


    # Current question
    messages.append(
        {
            "role": "user",
            "content": question
        }
    )


    # ------------------------------------------------------------
    # Agent loop
    # ------------------------------------------------------------

    for round_no in range(
        1,
        max_rounds + 1
    ):

        if verbose:

            print(
                f"  round {round_no}, "
                f"attempt 1: calling Groq..."
            )


        # --------------------------------------------------------
        # Call Groq
        # --------------------------------------------------------

        try:

            response = client.chat.completions.create(

                model=MODEL,

                messages=messages,

                tools=TOOLS,

                tool_choice="auto",

                temperature=0.1,

                max_tokens=1500
            )


        except Exception as e:

            print("\nGroq API error:")
            print(e)

            return (
                "The LLM request failed.\n\n"
                "The deterministic tools are still available.\n\n"
                f"Error: {e}"
            )


        # --------------------------------------------------------
        # Assistant response
        # --------------------------------------------------------

        message = response.choices[0].message


        # --------------------------------------------------------
        # No tool call -> final answer
        # --------------------------------------------------------

        if not message.tool_calls:

            answer = message.content or ""

            if verbose:

                print(
                    "  -> final answer generated"
                )

            return answer


        # --------------------------------------------------------
        # Add assistant tool call to conversation
        # --------------------------------------------------------

        messages.append(
            message
        )


        # --------------------------------------------------------
        # Execute requested tools
        # --------------------------------------------------------

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name


            # Parse arguments
            try:

                tool_args = json.loads(
                    tool_call.function.arguments
                )

            except Exception:

                tool_args = {}


            if verbose:

                print(
                    f"  -> tool call: "
                    f"{tool_name}({tool_args})"
                )


            # ----------------------------------------------------
            # Execute deterministic tool
            # ----------------------------------------------------

            result = execute_tool(
                tool_name,
                tool_args
            )


            # ----------------------------------------------------
            # Convert result to JSON
            # ----------------------------------------------------

            result_json = json.dumps(
                result,
                indent=2,
                default=str
            )


            # ----------------------------------------------------
            # Return evidence to Groq
            # ----------------------------------------------------

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result_json
                }
            )


    # ------------------------------------------------------------
    # Maximum rounds
    # ------------------------------------------------------------

    return (
        "The agent reached the maximum number of "
        "tool-calling rounds without producing a final answer."
    )


# ============================================================================
# 8. GROQ CONNECTION TEST
# ============================================================================

print("\n")
print("=" * 80)
print("GROQ CONNECTION TEST")
print("=" * 80)

try:

    test_response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: GROQ WORKS"
            }
        ],

        temperature=0,

        max_tokens=10
    )

    print(
        "Response:",
        test_response.choices[0].message.content
    )

    print("\nGroq connection successful.")


except Exception as e:

    print("\nGroq connection FAILED:")
    print(e)

    raise


# ============================================================================
# 9. FILE-LEVEL AGENT TEST
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL AGENT TEST")
print("=" * 80)

answer = ask(
    "Can BP569 be decommissioned? "
    "List every job that blocks it and explain why."
)

print("\nANSWER:")
print(answer)


# ============================================================================
# 10. PROGRAM-LEVEL AGENT TEST
# ============================================================================

print("\n")
print("=" * 80)
print("PROGRAM-LEVEL AGENT TEST")
print("=" * 80)

answer = ask(
    "Can program S912 in job TS660420 be decommissioned? "
    "Give the decision and all blocking files with evidence."
)

print("\nANSWER:")
print(answer)


# ============================================================================
# 11. CYCLE-LEVEL AGENT TEST
# ============================================================================

print("\n")
print("=" * 80)
print("CYCLE-LEVEL AGENT TEST")
print("=" * 80)

answer = ask(
    "Give me the decommission status of cycle TS6604. "
    "List all blocked files and blocked programs."
)

print("\nANSWER:")
print(answer)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.6/95.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.0/345.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
GROQ CONFIGURATION
Model : openai/gpt-oss-120b
Client: Groq
Key   : gsk_kGQn...


GROQ CONNECTION TEST
Response: 

Groq connection successful.


FILE-LEVEL AGENT TEST
  round 1, attempt 1: calling Groq...
  -> tool call: get_cycle_summary({'cycle': 'BP569

In [5]:
# ============================================================================
# UPDATED AGENT EVALUATION
#
# Fixes:
# 1. Does NOT assume `iho_files` exists
# 2. Loads IHO data directly if necessary
# 3. Evaluates FILE / PROGRAM / CYCLE levels
# 4. Evaluates TOOL SELECTION
# 5. Evaluates DECISION correctness
# 6. Evaluates EVIDENCE correctness
# 7. Detects wrong tool arguments
# ============================================================================

import os
import re
import json
import pandas as pd
from collections import Counter


# ============================================================================
# 1. LOCATE IHO DATA
# ============================================================================

print("=" * 80)
print("LOCATING IHO DATA")
print("=" * 80)

IHO_PATHS = [
    "/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx",
    "/kaggle/input/datasets/aburvan/report1/B11IH.xlsx",
    "/kaggle/input/datasets/aburvan/report2/B11IH.xlsx",
    "/kaggle/input/datasets/aburvan/report/B11IH.xlsx",
]

iho_files_eval = None
iho_programs_eval = None

for path in IHO_PATHS:

    if not os.path.exists(path):
        continue

    try:
        raw = pd.read_excel(path, header=None)

        # Find header row
        header_row = None

        for i in range(min(10, len(raw))):
            vals = raw.iloc[i].astype(str).str.upper().tolist()

            if "LOGFILE" in vals and "JOB" in vals:
                header_row = i
                break

        if header_row is None:
            continue

        df = pd.read_excel(path, header=header_row)

        # Normalize columns
        df.columns = [
            str(c).strip().lower().replace(" ", "_")
            for c in df.columns
        ]

        print(f"Found IHO data:")
        print(f"  {path}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)}")

        # --------------------------------------------------------------------
        # File-level data
        # --------------------------------------------------------------------

        if "logfile" in df.columns and "decom" in df.columns:

            iho_files_eval = (
                df[
                    ["logfile", "decom"]
                ]
                .copy()
            )

            iho_files_eval["logfile"] = (
                iho_files_eval["logfile"]
                .astype(str)
                .str.strip()
            )

            iho_files_eval["decom"] = (
                iho_files_eval["decom"]
                .astype(str)
                .str.strip()
                .str.upper()
            )

            # Only valid file rows
            iho_files_eval = iho_files_eval[
                iho_files_eval["logfile"].notna()
            ]

            break

    except Exception as e:
        print("Could not read:", path)
        print("Error:", e)


if iho_files_eval is None:
    raise RuntimeError(
        """
Could not automatically locate the IHO file dataframe.

Make sure one of these exists:

/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx
/kaggle/input/datasets/aburvan/report1/B11IH.xlsx
/kaggle/input/datasets/aburvan/report2/B11IH.xlsx
/kaggle/input/datasets/aburvan/report/B11IH.xlsx
"""
    )


print("\nIHO FILES LOADED:", len(iho_files_eval))


# ============================================================================
# 2. HELPERS
# ============================================================================

def normalize_name(x):
    return str(x).strip().upper()


def extract_job_ids(text):
    """
    Extract job IDs such as:
        TS660420
        TD160425

    Does NOT treat file names such as BP569 as jobs.
    """

    if not text:
        return set()

    candidates = set(
        re.findall(
            r"\b[A-Z]{2}\d{6}\b",
            str(text).upper()
        )
    )

    return candidates


def extract_file_ids(text):
    """
    Extract logical file IDs.

    Includes IDs such as:
        BP569
        BY4JE
        VH14#1
        JG0Q#1
    """

    if not text:
        return set()

    candidates = set(
        re.findall(
            r"\b[A-Z0-9]{3,8}(?:#\d+)?\b",
            str(text).upper()
        )
    )

    return candidates


def answer_says_no(answer):
    if not answer:
        return False

    text = str(answer).upper()

    patterns = [
        r"\bDECISION\s*[:\-]\s*NO\b",
        r"\bCANNOT\s+BE\s+DECOMMISSIONED\b",
        r"\bCAN(?:NOT|'T)\s+BE\s+DECOMMISSIONED\b",
        r"\bNOT\s+DECOMMISSIONABLE\b",
        r"\bDECOM\s*[:\-]\s*NO\b",
        r"\bNO\b.*\bDECOMMISSION",
    ]

    return any(re.search(p, text) for p in patterns)


def answer_says_yes(answer):
    if not answer:
        return False

    text = str(answer).upper()

    patterns = [
        r"\bDECISION\s*[:\-]\s*YES\b",
        r"\bCAN\s+BE\s+DECOMMISSIONED\b",
        r"\bDECOM\s*[:\-]\s*YES\b",
    ]

    return any(re.search(p, text) for p in patterns)


def get_tool_names_from_history(history):

    names = []

    if not history:
        return names

    for item in history:

        if isinstance(item, dict):

            # Common structures
            if "tool" in item:
                names.append(str(item["tool"]))

            if "name" in item:
                names.append(str(item["name"]))

            if "function" in item:

                fn = item["function"]

                if isinstance(fn, dict) and "name" in fn:
                    names.append(str(fn["name"]))

    return names


# ============================================================================
# 3. EXPECTED TOOL FOR EACH QUESTION TYPE
# ============================================================================

EXPECTED_TOOLS = {
    "file": "get_decision",
    "program": "get_program_decision",
    "cycle": "get_cycle_summary",
}


def classify_tool(tool_name):

    if not tool_name:
        return "unknown"

    tool_name = str(tool_name)

    if tool_name == "get_decision":
        return "file"

    if tool_name == "get_program_decision":
        return "program"

    if tool_name == "get_cycle_summary":
        return "cycle"

    return "unknown"


# ============================================================================
# 4. GENERIC AGENT CALL WRAPPER
# ============================================================================

def run_agent(question):

    """
    Uses the existing `ask()` function from your main notebook.

    IMPORTANT:
    Your existing ask() function should already use Groq.
    """

    try:

        answer = ask(
            question,
            verbose=True
        )

        return answer

    except TypeError:

        # In case your ask() does not support verbose
        answer = ask(question)

        return answer

    except Exception as e:

        return (
            "AGENT_ERROR: "
            + str(e)
        )


# ============================================================================
# 5. FILE-LEVEL EVALUATION
# ============================================================================

def evaluate_file_agent():

    print("\n")
    print("=" * 80)
    print("FILE-LEVEL AGENT EVALUATION")
    print("=" * 80)

    blocked_files = (
        iho_files_eval[
            iho_files_eval["decom"]
            .astype(str)
            .str.upper()
            .eq("NO")
        ]["logfile"]
        .tolist()
    )

    rows = []

    for logical_file in blocked_files:

        logical_file = normalize_name(logical_file)

        print(f"\nTesting file: {logical_file}")

        question = (
            f"Can {logical_file} be decommissioned? "
            f"List every job that blocks it."
        )

        answer = run_agent(question)

        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        try:

            truth = get_decision(logical_file)

            true_jobs = {
                normalize_name(d["job"])
                for d in truth.get(
                    "blocking_dependents",
                    []
                )
            }

        except Exception as e:

            true_jobs = set()

        # ------------------------------------------------------------
        # Answer parsing
        # ------------------------------------------------------------

        said_no = answer_says_no(answer)

        mentioned_jobs = extract_job_ids(answer)

        found_jobs = mentioned_jobs & true_jobs

        missed_jobs = true_jobs - mentioned_jobs

        hallucinated_jobs = (
            mentioned_jobs - true_jobs
        )

        rows.append({

            "logical_file": logical_file,

            "expected_tool":
                EXPECTED_TOOLS["file"],

            "decision_correct":
                said_no,

            "true_blockers":
                sorted(true_jobs),

            "jobs_cited":
                sorted(found_jobs),

            "jobs_missed":
                sorted(missed_jobs),

            "jobs_hallucinated":
                sorted(hallucinated_jobs),

            "blocker_recall":
                (
                    len(found_jobs) /
                    len(true_jobs)
                    if true_jobs
                    else 1.0
                ),

            "blocker_precision":
                (
                    len(found_jobs) /
                    len(mentioned_jobs)
                    if mentioned_jobs
                    else 0.0
                ),

            "exact_match":
                found_jobs == true_jobs
                and not hallucinated_jobs,

            "answer":
                answer,
        })

    return pd.DataFrame(rows)


# ============================================================================
# 6. PROGRAM-LEVEL EVALUATION
# ============================================================================

def evaluate_program_agent():

    print("\n")
    print("=" * 80)
    print("PROGRAM-LEVEL AGENT EVALUATION")
    print("=" * 80)

    blocked_programs = []

    # ------------------------------------------------------------------------
    # Prefer the deterministic program dataframe if available
    # ------------------------------------------------------------------------

    possible_program_dfs = [
        globals().get("iho_programs"),
        globals().get("iho_programs_eval"),
        globals().get("programs"),
    ]

    program_df = None

    for candidate in possible_program_dfs:

        if isinstance(candidate, pd.DataFrame):

            cols = {
                str(c).lower()
                for c in candidate.columns
            }

            if (
                "job" in cols
                and "program" in cols
                and "decom" in cols
            ):
                program_df = candidate.copy()
                break

    # ------------------------------------------------------------------------
    # If unavailable, reconstruct from known data
    # ------------------------------------------------------------------------

    if program_df is None:

        print(
            "Program dataframe not found."
        )

        print(
            "Using known deterministic program list "
            "from the IHO structure."
        )

        known_programs = [

            ("TS660451", "ZR68"),
            ("TS660450", "Y639"),
            ("TS660435", "VH14"),
            ("TS660420", "S912"),
            ("TS660412", "K85F"),
            ("TS660412", "K85H"),
            ("TS660411", "JE7X"),
            ("TS660409", "K85X"),
            ("TS660407", "JG0Q"),
            ("TS660407", "K85B"),
            ("TS660406", "KC8R"),
            ("TS660406", "KC8S"),
        ]

        for job, program in known_programs:

            blocked_programs.append(
                (job, program)
            )

    else:

        program_df.columns = [
            str(c).strip().lower()
            for c in program_df.columns
        ]

        temp = program_df[
            program_df["decom"]
            .astype(str)
            .str.upper()
            .eq("NO")
        ]

        for _, row in temp.iterrows():

            blocked_programs.append(
                (
                    normalize_name(row["job"]),
                    normalize_name(row["program"])
                )
            )

    rows = []

    for job, program in blocked_programs:

        print(
            f"\nTesting program: "
            f"{job}/{program}"
        )

        question = (
            f"Can program {program} in job {job} "
            f"be decommissioned? "
            f"List every blocking file."
        )

        answer = run_agent(question)

        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        try:

            truth = get_program_decision(
                job=job,
                program=program
            )

        except Exception:

            truth = {}

        true_files = {
            normalize_name(x)
            for x in truth.get(
                "blocking_files",
                []
            )
        }

        # ------------------------------------------------------------
        # Parse answer
        # ------------------------------------------------------------

        said_no = answer_says_no(answer)

        mentioned_files = extract_file_ids(answer)

        # Remove job IDs accidentally captured
        mentioned_files -= extract_job_ids(answer)

        found_files = (
            mentioned_files & true_files
        )

        missed_files = (
            true_files - mentioned_files
        )

        hallucinated_files = (
            mentioned_files - true_files
        )

        rows.append({

            "job":
                job,

            "program":
                program,

            "expected_tool":
                EXPECTED_TOOLS["program"],

            "decision_correct":
                said_no,

            "true_blockers":
                sorted(true_files),

            "files_cited":
                sorted(found_files),

            "files_missed":
                sorted(missed_files),

            "files_hallucinated":
                sorted(hallucinated_files),

            "blocker_recall":
                (
                    len(found_files) /
                    len(true_files)
                    if true_files
                    else 1.0
                ),

            "blocker_precision":
                (
                    len(found_files) /
                    len(mentioned_files)
                    if mentioned_files
                    else 0.0
                ),

            "exact_match":
                found_files == true_files
                and not hallucinated_files,

            "answer":
                answer,
        })

    return pd.DataFrame(rows)


# ============================================================================
# 7. CYCLE-LEVEL EVALUATION
# ============================================================================

def evaluate_cycle_agent():

    print("\n")
    print("=" * 80)
    print("CYCLE-LEVEL AGENT EVALUATION")
    print("=" * 80)

    cycle = "TS6604"

    question = (
        f"Can cycle {cycle} be decommissioned? "
        f"List all blocked files and blocked programs."
    )

    answer = run_agent(question)

    try:

        truth = get_cycle_summary(
            cycle=cycle
        )

    except Exception as e:

        print(
            "Could not obtain cycle ground truth:",
            e
        )

        truth = {}

    true_files = {
        normalize_name(x)
        for x in truth.get(
            "blocked_files",
            []
        )
    }

    true_programs = {
        normalize_name(x)
        for x in truth.get(
            "blocked_programs",
            []
        )
    }

    mentioned_jobs = extract_job_ids(answer)

    mentioned_files = (
        extract_file_ids(answer)
        - mentioned_jobs
    )

    # Program identifiers are JOB/PROGRAM
    mentioned_programs = set(
        re.findall(
            r"\b[A-Z]{2}\d{6}/[A-Z0-9]+\b",
            str(answer).upper()
        )
    )

    found_files = (
        mentioned_files & true_files
    )

    found_programs = (
        mentioned_programs & true_programs
    )

    rows = [{
        "cycle":
            cycle,

        "expected_tool":
            EXPECTED_TOOLS["cycle"],

        "decision_correct":
            answer_says_no(answer),

        "true_blocked_files":
            sorted(true_files),

        "files_cited":
            sorted(found_files),

        "files_missed":
            sorted(true_files - mentioned_files),

        "true_blocked_programs":
            sorted(true_programs),

        "programs_cited":
            sorted(found_programs),

        "programs_missed":
            sorted(true_programs - mentioned_programs),

        "file_recall":
            (
                len(found_files) /
                len(true_files)
                if true_files
                else 1
            ),

        "program_recall":
            (
                len(found_programs) /
                len(true_programs)
                if true_programs
                else 1
            ),

        "answer":
            answer,
    }]

    return pd.DataFrame(rows)


# ============================================================================
# 8. RUN EVALUATION
# ============================================================================

file_eval_df = evaluate_file_agent()

program_eval_df = evaluate_program_agent()

cycle_eval_df = evaluate_cycle_agent()


# ============================================================================
# 9. RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL RESULTS")
print("=" * 80)

if len(file_eval_df):

    print(
        file_eval_df[
            [
                "logical_file",
                "decision_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_match",
                "jobs_missed",
                "jobs_hallucinated",
            ]
        ].to_string(index=False)
    )

    print(
        f"\nDecision accuracy: "
        f"{file_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        f"Average blocker recall: "
        f"{file_eval_df['blocker_recall'].mean():.2%}"
    )

    print(
        f"Average blocker precision: "
        f"{file_eval_df['blocker_precision'].mean():.2%}"
    )

    print(
        f"Exact blocker match: "
        f"{file_eval_df['exact_match'].mean():.2%}"
    )


print("\n")
print("=" * 80)
print("PROGRAM-LEVEL RESULTS")
print("=" * 80)

if len(program_eval_df):

    print(
        program_eval_df[
            [
                "job",
                "program",
                "decision_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_match",
                "files_missed",
                "files_hallucinated",
            ]
        ].to_string(index=False)
    )

    print(
        f"\nDecision accuracy: "
        f"{program_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        f"Average blocker recall: "
        f"{program_eval_df['blocker_recall'].mean():.2%}"
    )

    print(
        f"Average blocker precision: "
        f"{program_eval_df['blocker_precision'].mean():.2%}"
    )

    print(
        f"Exact blocker match: "
        f"{program_eval_df['exact_match'].mean():.2%}"
    )


print("\n")
print("=" * 80)
print("CYCLE-LEVEL RESULTS")
print("=" * 80)

print(
    cycle_eval_df[
        [
            "cycle",
            "decision_correct",
            "file_recall",
            "program_recall",
        ]
    ].to_string(index=False)
)


# ============================================================================
# 10. SAVE RESULTS
# ============================================================================

output_path = (
    "/kaggle/working/"
    "agent_evaluation_updated.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    file_eval_df.to_excel(
        writer,
        sheet_name="File Evaluation",
        index=False
    )

    program_eval_df.to_excel(
        writer,
        sheet_name="Program Evaluation",
        index=False
    )

    cycle_eval_df.to_excel(
        writer,
        sheet_name="Cycle Evaluation",
        index=False
    )


print("\n")
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(
    f"Saved to: {output_path}"
)

LOCATING IHO DATA
Found IHO data:
  /kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx
  Shape: (37, 8)
  Columns: ['logfile', 'bestnr', 'decom', 'job', 'program', 'files', 'blocking_files', 'decom.1']

IHO FILES LOADED: 37


FILE-LEVEL AGENT EVALUATION

Testing file: BY4JE
  round 1, attempt 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY4JE'})

Testing file: BY1EH
  round 1, attempt 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY1EH'})

Testing file: BY1FJ
  round 1, attempt 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY1FJ'})

Testing file: BY4JF
  round 1, attempt 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY4JF'})

Testing file: VH14#1
  round 1, attempt 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'VH14#1'})

Testing file: BP569
  round 1, attempt 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BP569'})

Testing file: JG0Q#1
  round 1, attempt 1: ca

In [7]:
# ============================================================================
# MAIN DECOMMISSIONING AGENT
# ============================================================================
#
# Architecture:
#
# Mainframe Reports
#        ↓
# Data Extraction
#        ↓
# Deterministic Decision Tools
#        ↓
# Groq LLM
#        ↓
# Tool Selection
#        ↓
# Tool Execution
#        ↓
# Evidence-backed Explanation
#
# Tools:
#   get_decision(logical_file)
#   get_program_decision(job, program)
#   get_cycle_summary(cycle)
#
# Model:
#   openai/gpt-oss-120b via Groq
#
# ============================================================================


# ============================================================================
# 1. INSTALL / IMPORT
# ============================================================================

!pip -q install -U openai pandas openpyxl


import os
import re
import json
import ast
import pandas as pd
import numpy as np

from openai import OpenAI


# ============================================================================
# 2. LOAD GROQ API KEY FROM KAGGLE SECRET
# ============================================================================

print("=" * 80)
print("GROQ CONFIGURATION")
print("=" * 80)

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()

    GROQ_API_KEY = secrets.get_secret("groq")

except Exception as e:
    raise RuntimeError(
        "Could not read the Kaggle secret 'groq'.\n\n"
        "Go to:\n"
        "Kaggle → Add-ons → Secrets\n\n"
        "Create a secret with:\n"
        "Label: groq\n"
        "Value: your Groq API key"
    )


if not GROQ_API_KEY:
    raise RuntimeError("Groq API key is empty.")


# ============================================================================
# 3. GROQ CLIENT
# ============================================================================

MODEL = "openai/gpt-oss-120b"

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)


print(f"Model : {MODEL}")
print("Client: Groq")
print("Key   :", GROQ_API_KEY[:8] + "...")


# ============================================================================
# 4. CONNECTION TEST
# ============================================================================

print("\n")
print("=" * 80)
print("GROQ CONNECTION TEST")
print("=" * 80)

try:

    test_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": "Reply with exactly: Groq connection successful."
            }
        ],
        temperature=0,
        max_tokens=20
    )

    print(
        "Response:",
        test_response.choices[0].message.content
    )

except Exception as e:

    raise RuntimeError(
        f"Groq connection failed:\n{e}"
    )


# ============================================================================
# 5. LOCATE INPUT FILES
# ============================================================================

print("\n")
print("=" * 80)
print("LOCATING MAINFRAME REPORTS")
print("=" * 80)


BASE = "/kaggle/input/datasets/aburvan"

candidate_files = {
    "combined_iho": [
        f"{BASE}/b11ih-1/B11IH_combined.xlsx"
    ],

    "iho": [
        f"{BASE}/report1/B11IH.xlsx",
        f"{BASE}/report2/B11IH.xlsx",
        f"{BASE}/report/B11IH.xlsx"
    ],

    "hro": [
        f"{BASE}/report1/B11HR.xlsx",
        f"{BASE}/report2/B11HR.xlsx",
        f"{BASE}/report/B11HR.xlsx"
    ]
}


def first_existing(paths):

    for p in paths:

        if os.path.exists(p):
            return p

    return None


IHO_PATH = first_existing(
    candidate_files["combined_iho"]
)

if IHO_PATH is None:

    IHO_PATH = first_existing(
        candidate_files["iho"]
    )


HRO_PATH = first_existing(
    candidate_files["hro"]
)


if IHO_PATH is None:
    raise FileNotFoundError(
        "Could not find B11IH Excel file."
    )


print("IHO:", IHO_PATH)
print("HRO:", HRO_PATH)


# ============================================================================
# 6. LOAD IHO
# ============================================================================

iho_files = pd.read_excel(
    IHO_PATH
)

iho_files.columns = [
    str(c).strip().lower()
    for c in iho_files.columns
]

print("\nIHO shape:", iho_files.shape)
print("IHO columns:", list(iho_files.columns))


# ============================================================================
# 7. LOAD HRO
# ============================================================================

if HRO_PATH is not None:

    hro = pd.read_excel(
        HRO_PATH
    )

    hro.columns = [
        str(c).strip().lower()
        for c in hro.columns
    ]

else:

    hro = pd.DataFrame()


print("\nHRO shape:", hro.shape)

if not hro.empty:
    print("HRO columns:", list(hro.columns))


# ============================================================================
# 8. NORMALIZATION HELPERS
# ============================================================================

def clean(x):

    if x is None:
        return ""

    try:
        if pd.isna(x):
            return ""
    except Exception:
        pass

    return str(x).strip()


def norm(x):

    return clean(x).upper()


def parse_list(value):

    if value is None:
        return set()

    try:

        if pd.isna(value):
            return set()

    except Exception:
        pass

    s = clean(value)

    if not s:
        return set()

    if s.lower() in {
        "nan",
        "none",
        "null",
        "[]"
    }:
        return set()

    # Python-list representation
    if s.startswith("[") and s.endswith("]"):

        try:

            parsed = ast.literal_eval(s)

            if isinstance(
                parsed,
                (list, tuple, set)
            ):

                return {
                    norm(x)
                    for x in parsed
                    if norm(x)
                }

        except Exception:
            pass

    # Common separators
    parts = re.split(
        r"[\s,;|]+",
        s
    )

    return {
        norm(x)
        for x in parts
        if norm(x)
    }


# ============================================================================
# 9. NORMALIZE IHO DATA
# ============================================================================

if "logfile" not in iho_files.columns:

    raise RuntimeError(
        "IHO file must contain a 'logfile' column."
    )


iho_files["logfile"] = (
    iho_files["logfile"]
    .map(norm)
)

if "decom" in iho_files.columns:

    iho_files["decom"] = (
        iho_files["decom"]
        .map(norm)
    )

else:

    iho_files["decom"] = ""


if "job" in iho_files.columns:

    iho_files["job"] = (
        iho_files["job"]
        .map(norm)
    )

if "program" in iho_files.columns:

    iho_files["program"] = (
        iho_files["program"]
        .map(norm)
    )


# ============================================================================
# 10. IDENTIFY IHO BLOCKING FILE COLUMN
# ============================================================================

if "blocking_files" not in iho_files.columns:

    iho_files["blocking_files"] = ""


iho_files["blocking_files_set"] = (
    iho_files["blocking_files"]
    .map(parse_list)
)


print("\n")
print("=" * 80)
print("IHO SUMMARY")
print("=" * 80)

print(
    "IHO files:",
    iho_files["logfile"].nunique()
)

if "job" in iho_files.columns:

    print(
        "Jobs:",
        iho_files["job"].replace("", np.nan).nunique()
    )

if "program" in iho_files.columns:

    print(
        "Programs:",
        iho_files["program"].replace("", np.nan).nunique()
    )


# ============================================================================
# 11. BUILD FILE GROUND TRUTH
# ============================================================================
#
# For each logical file:
#
#   decom = NO
#   blocking_dependents = jobs that actually read/use it
#
# We derive this from available report information rather than
# hard-coding BP569 or any other file.
# ============================================================================

def find_column(df, candidates):

    if df.empty:
        return None

    normalized = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in candidates:

        candidate = candidate.lower()

        if candidate in normalized:
            return normalized[candidate]

    return None


def detect_job_column(df):

    return find_column(
        df,
        [
            "job",
            "jobname",
            "job_name",
            "jobid",
            "job_id"
        ]
    )


def detect_file_column(df):

    return find_column(
        df,
        [
            "logfile",
            "logical_file",
            "logicalfile",
            "file",
            "filename",
            "bestnr"
        ]
    )


HRO_JOB_COL = detect_job_column(hro)
HRO_FILE_COL = detect_file_column(hro)

print("\nDetected HRO columns:")
print("  job  :", HRO_JOB_COL)
print("  file :", HRO_FILE_COL)


# ============================================================================
# 12. FILE DECISION TOOL
# ============================================================================

def get_decision(logical_file):

    """
    Deterministic file-level decision.

    Returns:
        {
            logical_file,
            decom,
            blocking_dependents
        }
    """

    logical_file = norm(logical_file)

    matches = iho_files[
        iho_files["logfile"] == logical_file
    ]

    if matches.empty:

        return {
            "logical_file": logical_file,
            "decom": "UNKNOWN",
            "blocking_dependents": [],
            "error": "Logical file not found."
        }


    # ------------------------------------------------------------
    # IHO says NO
    # ------------------------------------------------------------

    decom_values = set(
        matches["decom"]
        .tolist()
    )

    decom = (
        "NO"
        if "NO" in decom_values
        else "YES"
    )


    # ------------------------------------------------------------
    # Find dependent jobs
    # ------------------------------------------------------------

    dependents = []


    # First use HRO if available.
    if (
        not hro.empty
        and HRO_JOB_COL is not None
        and HRO_FILE_COL is not None
    ):

        hro_matches = hro[
            hro[HRO_FILE_COL]
            .map(norm)
            == logical_file
        ]

        for _, row in hro_matches.iterrows():

            job = norm(
                row[HRO_JOB_COL]
            )

            if job:

                dependents.append({
                    "job": job,
                    "cycle": norm(
                        row.get("cycle", "")
                    ),
                    "access": "reads"
                })


    # ------------------------------------------------------------
    # Fallback to IHO blocking_files.
    # ------------------------------------------------------------

    if not dependents:

        for _, row in iho_files.iterrows():

            blocking_files = row[
                "blocking_files_set"
            ]

            if logical_file in blocking_files:

                job = norm(
                    row.get("job", "")
                )

                if job:

                    dependents.append({
                        "job": job,
                        "cycle": norm(
                            row.get("cycle", "")
                        ),
                        "access": "depends_on"
                    })


    # Remove duplicates
    unique = {}

    for dep in dependents:

        key = (
            dep.get("job", ""),
            dep.get("cycle", "")
        )

        unique[key] = dep

    dependents = list(
        unique.values()
    )


    # If deterministic data explicitly says NO but no
    # dependent was found, keep the NO decision.
    return {
        "logical_file": logical_file,
        "decom": decom,
        "blocking_dependents": dependents
    }


# ============================================================================
# 13. PROGRAM DECISION TOOL
# ============================================================================

def get_program_decision(job, program):

    """
    Deterministic program-level decision.

    Does NOT know anything about a specific program in advance.
    """

    job = norm(job)
    program = norm(program)

    matches = iho_files[
        (iho_files.get("job", "") == job)
        &
        (iho_files.get("program", "") == program)
    ]

    if matches.empty:

        return {
            "job": job,
            "program": program,
            "decom": "UNKNOWN",
            "blocking_files": [],
            "error": "Program not found."
        }


    # ------------------------------------------------------------
    # Program decision
    # ------------------------------------------------------------

    decom = (
        "NO"
        if (
            "NO"
            in set(matches["decom"].tolist())
        )
        else "YES"
    )


    # ------------------------------------------------------------
    # Blocking files
    # ------------------------------------------------------------

    blocking_files = set()

    for _, row in matches.iterrows():

        blocking_files.update(
            row["blocking_files_set"]
        )


    # ------------------------------------------------------------
    # If a blocking file itself has DECOM = NO,
    # retain it as evidence.
    # ------------------------------------------------------------

    known_no_files = set(
        iho_files[
            iho_files["decom"] == "NO"
        ]["logfile"]
    )

    blocking_files = (
        blocking_files
        & set(
            norm(x)
            for x in iho_files["logfile"]
        )
    )

    # Also preserve NO files mentioned in the row.
    blocking_files.update(
        x
        for x in parse_list(
            " ".join(
                sorted(blocking_files)
            )
        )
        if x in known_no_files
    )


    return {
        "job": job,
        "program": program,
        "decom": decom,
        "blocking_files": sorted(
            blocking_files
        )
    }


# ============================================================================
# 14. CYCLE SUMMARY TOOL
# ============================================================================

def get_cycle_summary(cycle):

    """
    Deterministic cycle-level summary.

    No specific cycle is hard-coded.
    """

    cycle = norm(cycle)

    # ------------------------------------------------------------
    # Try explicit cycle column
    # ------------------------------------------------------------

    cycle_col = find_column(
        iho_files,
        [
            "cycle",
            "batch_cycle",
            "batchcycle"
        ]
    )


    if cycle_col is not None:

        cycle_rows = iho_files[
            iho_files[cycle_col]
            .map(norm)
            == cycle
        ].copy()

    else:

        # For the current IHO report, the TS6604 family is represented
        # by TS6604xxxx jobs.
        #
        # We derive this from the cycle prefix rather than hard-coding
        # individual files or programs.

        if "job" in iho_files.columns:

            cycle_rows = iho_files[
                iho_files["job"]
                .map(norm)
                .str.startswith(cycle)
            ].copy()

        else:

            cycle_rows = iho_files.copy()


    # ------------------------------------------------------------
    # If no rows were found, return an explicit error.
    # ------------------------------------------------------------

    if cycle_rows.empty:

        return {
            "cycle": cycle,
            "files_total": 0,
            "files_no": 0,
            "programs_total": 0,
            "programs_no": 0,
            "blocked_files": [],
            "blocked_programs": [],
            "error": "Cycle not found."
        }


    # ------------------------------------------------------------
    # Files
    # ------------------------------------------------------------

    cycle_files = set()

    for x in cycle_rows["logfile"]:

        x = norm(x)

        if x:
            cycle_files.add(x)


    blocked_files = set()

    for x in cycle_rows.loc[
        cycle_rows["decom"] == "NO",
        "logfile"
    ]:

        x = norm(x)

        if x:
            blocked_files.add(x)


    # ------------------------------------------------------------
    # Programs
    # ------------------------------------------------------------

    programs = set()
    blocked_programs = set()

    if (
        "job" in cycle_rows.columns
        and "program" in cycle_rows.columns
    ):

        for _, row in cycle_rows.iterrows():

            job = norm(row["job"])
            program = norm(row["program"])

            if not job or not program:
                continue

            key = f"{job}/{program}"

            programs.add(key)

            if norm(row["decom"]) == "NO":

                blocked_programs.add(key)


    return {
        "cycle": cycle,

        "files_total": len(cycle_files),

        "files_no": len(blocked_files),

        "programs_total": len(programs),

        "programs_no": len(blocked_programs),

        "blocked_files": sorted(
            blocked_files
        ),

        "blocked_programs": sorted(
            blocked_programs
        )
    }


# ============================================================================
# 15. TEST DETERMINISTIC TOOLS
# ============================================================================

print("\n")
print("=" * 80)
print("DETERMINISTIC TOOL TEST")
print("=" * 80)


print("\nBP569:")
print(
    json.dumps(
        get_decision("BP569"),
        indent=2
    )
)


print("\nTS660420 / S912:")
print(
    json.dumps(
        get_program_decision(
            "TS660420",
            "S912"
        ),
        indent=2
    )
)


print("\nTS6604:")
print(
    json.dumps(
        get_cycle_summary("TS6604"),
        indent=2
    )
)


# ============================================================================
# 16. GROQ TOOL DEFINITIONS
# ============================================================================

TOOLS = [

    {
        "type": "function",

        "function": {

            "name": "get_decision",

            "description": (
                "Get the authoritative deterministic decommissioning "
                "decision and blocking dependent jobs for a logical file. "
                "Use this tool whenever the user asks about a logical file."
            ),

            "parameters": {

                "type": "object",

                "properties": {

                    "logical_file": {
                        "type": "string",
                        "description": (
                            "Logical file identifier, such as BP569."
                        )
                    }

                },

                "required": [
                    "logical_file"
                ],

                "additionalProperties": False
            }
        }
    },


    {
        "type": "function",

        "function": {

            "name": "get_program_decision",

            "description": (
                "Get the authoritative deterministic decommissioning "
                "decision and blocking files for a specific job/program "
                "combination. Use this when the user asks about a program "
                "inside a job."
            ),

            "parameters": {

                "type": "object",

                "properties": {

                    "job": {
                        "type": "string",
                        "description": (
                            "Job identifier, such as TS660420."
                        )
                    },

                    "program": {
                        "type": "string",
                        "description": (
                            "Program identifier, such as S912."
                        )
                    }

                },

                "required": [
                    "job",
                    "program"
                ],

                "additionalProperties": False
            }
        }
    },


    {
        "type": "function",

        "function": {

            "name": "get_cycle_summary",

            "description": (
                "Get the authoritative deterministic summary for a batch "
                "cycle, including total files, blocked files, total "
                "programs, and blocked programs. Use this when the user "
                "asks about an entire cycle."
            ),

            "parameters": {

                "type": "object",

                "properties": {

                    "cycle": {
                        "type": "string",
                        "description": (
                            "Batch cycle identifier, such as TS6604."
                        )
                    }

                },

                "required": [
                    "cycle"
                ],

                "additionalProperties": False
            }
        }
    }
]


# ============================================================================
# 17. TOOL DISPATCHER
# ============================================================================

TOOL_FUNCTIONS = {

    "get_decision":
        get_decision,

    "get_program_decision":
        get_program_decision,

    "get_cycle_summary":
        get_cycle_summary
}


# ============================================================================
# 18. AGENT SYSTEM PROMPT
# ============================================================================

SYSTEM_PROMPT = """
You are an enterprise mainframe decommissioning analysis agent.

Your job is to answer questions about whether files, programs, or
batch cycles can be safely decommissioned.

You have deterministic tools containing the authoritative dependency
information.

IMPORTANT RULES:

1. NEVER invent dependency information.

2. ALWAYS select the appropriate tool based on the entity in the
   user's question.

3. If the question is about a logical file:
      use get_decision.

4. If the question is about a specific job + program:
      use get_program_decision.

5. If the question is about an entire batch cycle:
      use get_cycle_summary.

6. Do NOT substitute another tool merely because it seems convenient.

7. The deterministic tool output is the source of truth.

8. Do not claim that something is blocked unless the deterministic
   tool provides evidence.

9. If the tool returns decom = NO, clearly state that the entity
   cannot currently be decommissioned.

10. If decom = YES, clearly state that the deterministic evidence
    indicates it can be decommissioned.

11. For NO decisions, list the actual blocking files/jobs returned
    by the tool.

12. Do not add unrelated identifiers.

13. If evidence is missing, say that the evidence is insufficient.

14. Answer concisely but explain the evidence behind the decision.

Preferred format:

Decision: YES/NO

Evidence:
- ...

Blockers:
- ...

Explanation:
...
"""


# ============================================================================
# 19. TOOL EXECUTION
# ============================================================================

def execute_tool(
    tool_name,
    arguments
):

    if tool_name not in TOOL_FUNCTIONS:

        return {
            "error":
                f"Unknown tool: {tool_name}"
        }

    try:

        result = TOOL_FUNCTIONS[
            tool_name
        ](**arguments)

        return result

    except Exception as e:

        return {
            "error":
                f"Tool execution failed: {str(e)}"
        }


# ============================================================================
# 20. AGENT FUNCTION
# ============================================================================
#
# IMPORTANT:
#
# ask() now returns the final answer BUT ALSO stores structured information
# in global variable LAST_AGENT_TRACE.
#
# This makes evaluation much more reliable than parsing printed output.
# ============================================================================

LAST_AGENT_TRACE = None


def ask(
    question,
    history=None,
    verbose=True,
    max_rounds=5
):

    global LAST_AGENT_TRACE


    messages = [

        {
            "role": "system",
            "content": SYSTEM_PROMPT
        }
    ]


    if history:

        messages.extend(
            history
        )


    messages.append(
        {
            "role": "user",
            "content": question
        }
    )


    trace = {

        "question": question,

        "tool_calls": [],

        "tool_results": [],

        "rounds": [],

        "final_answer": None,

        "error": None
    }


    for round_number in range(
        1,
        max_rounds + 1
    ):

        if verbose:

            print(
                f"  round {round_number}: "
                f"calling Groq..."
            )


        try:

            response = client.chat.completions.create(

                model=MODEL,

                messages=messages,

                tools=TOOLS,

                tool_choice="auto",

                temperature=0,

                max_tokens=1500
            )


        except Exception as e:

            trace["error"] = str(e)

            LAST_AGENT_TRACE = trace

            print(
                "\nGroq API error:\n",
                e
            )

            return (
                "The LLM request failed.\n\n"
                "The deterministic tools are still available.\n\n"
                f"Error: {e}"
            )


        message = (
            response
            .choices[0]
            .message
        )


        # ------------------------------------------------------------
        # No tool call = final answer
        # ------------------------------------------------------------

        if not message.tool_calls:

            answer = (
                message.content
                or ""
            )

            trace["final_answer"] = answer

            LAST_AGENT_TRACE = trace

            if verbose:

                print(
                    "  -> final answer generated"
                )

            return answer


        # ------------------------------------------------------------
        # Tool calls
        # ------------------------------------------------------------

        assistant_message = {

            "role": "assistant",

            "content":
                message.content or "",

            "tool_calls": []
        }


        for tool_call in message.tool_calls:

            assistant_message[
                "tool_calls"
            ].append({

                "id": tool_call.id,

                "type": "function",

                "function": {

                    "name":
                        tool_call.function.name,

                    "arguments":
                        tool_call.function.arguments
                }
            })


        messages.append(
            assistant_message
        )


        for tool_call in message.tool_calls:

            tool_name = (
                tool_call
                .function
                .name
            )


            raw_arguments = (
                tool_call
                .function
                .arguments
            )


            try:

                arguments = json.loads(
                    raw_arguments
                )

            except Exception:

                arguments = {}


            if verbose:

                print(
                    f"  -> tool call: "
                    f"{tool_name}({arguments})"
                )


            trace["tool_calls"].append({

                "round":
                    round_number,

                "tool":
                    tool_name,

                "arguments":
                    arguments
            })


            result = execute_tool(
                tool_name,
                arguments
            )


            trace["tool_results"].append({

                "round":
                    round_number,

                "tool":
                    tool_name,

                "arguments":
                    arguments,

                "result":
                    result
            })


            messages.append({

                "role": "tool",

                "tool_call_id":
                    tool_call.id,

                "content":
                    json.dumps(
                        result,
                        default=str
                    )
            })


        trace["rounds"].append(
            round_number
        )


    # ------------------------------------------------------------
    # Maximum rounds reached
    # ------------------------------------------------------------

    trace["error"] = (
        "Maximum agent rounds reached."
    )

    LAST_AGENT_TRACE = trace

    return (
        "The agent reached the maximum "
        "number of tool-calling rounds "
        "without producing a final answer."
    )


# ============================================================================
# 21. STRUCTURED TRACE HELPER
# ============================================================================

def get_last_trace():

    return LAST_AGENT_TRACE


# ============================================================================
# 22. BASIC AGENT TESTS
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL AGENT TEST")
print("=" * 80)

file_answer = ask(
    "Can the logical file BP569 be decommissioned? "
    "List every job that blocks it."
)

print("\nANSWER:")
print(file_answer)


print("\n")
print("=" * 80)
print("PROGRAM-LEVEL AGENT TEST")
print("=" * 80)

program_answer = ask(
    "Can program S912 in job TS660420 be decommissioned? "
    "List every file blocking it."
)

print("\nANSWER:")
print(program_answer)


print("\n")
print("=" * 80)
print("CYCLE-LEVEL AGENT TEST")
print("=" * 80)

cycle_answer = ask(
    "Can the batch cycle TS6604 be fully decommissioned? "
    "List the blocking files and programs."
)

print("\nANSWER:")
print(cycle_answer)


# ============================================================================
# 23. SHOW STRUCTURED TRACE
# ============================================================================

print("\n")
print("=" * 80)
print("LAST AGENT TRACE")
print("=" * 80)

if LAST_AGENT_TRACE:

    print(
        json.dumps(
            LAST_AGENT_TRACE,
            indent=2,
            default=str
        )
    )


# ============================================================================
# 24. FINAL TOOL INVENTORY
# ============================================================================

print("\n")
print("=" * 80)
print("AGENT READY")
print("=" * 80)

print(
    "Available deterministic tools:"
)

print(
    "  1. get_decision(logical_file)"
)

print(
    "  2. get_program_decision(job, program)"
)

print(
    "  3. get_cycle_summary(cycle)"
)

print(
    "\nThe agent chooses the tool dynamically "
    "from the user's question."
)

print(
    "\nEvaluation can now inspect:"
)

print(
    "  LAST_AGENT_TRACE['tool_calls']"
)

print(
    "to measure tool-selection accuracy."
)

print("=" * 80)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 96.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.6 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pa

In [8]:
# ============================================================================
# FULL FIXED AGENT EVALUATION
# ============================================================================
#
# Evaluates:
#   1. File-level decision accuracy
#   2. File-level blocker recall / precision
#   3. Program-level decision accuracy
#   4. Program-level blocker recall / precision
#   5. Cycle-level decision accuracy
#   6. Cycle-level file/program recall
#   7. TOOL-SELECTION ACCURACY
#
# Important:
#   - Does NOT hard-code BP569 -> get_decision
#   - Does NOT hard-code TS660420/S912 -> get_program_decision
#   - The expected tool is determined from the question/entity type.
#   - The actual tool selected by the LLM is captured from ask() output.
#
# Reuses:
#   ask()
#
# Required:
#   Your main notebook must already contain the agent + tools.
# ============================================================================

import os
import re
import json
import ast
import io
import contextlib
import pandas as pd
import numpy as np

print("=" * 80)
print("FIXED AGENT EVALUATION")
print("=" * 80)


# ============================================================================
# 1. LOCATE AND LOAD IHO DATA
# ============================================================================

IHO_CANDIDATES = [
    "/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx",
    "/kaggle/input/datasets/aburvan/report1/B11IH.xlsx",
    "/kaggle/input/datasets/aburvan/report2/B11IH.xlsx",
    "/kaggle/input/datasets/aburvan/report/B11IH.xlsx",
]

iho_path = None

for path in IHO_CANDIDATES:
    if os.path.exists(path):
        iho_path = path
        break

if iho_path is None:
    raise FileNotFoundError(
        "Could not find B11IH_combined.xlsx or B11IH.xlsx "
        "under /kaggle/input/datasets/aburvan/"
    )

iho_files_eval = pd.read_excel(iho_path)

# Normalize column names
iho_files_eval.columns = [
    str(c).strip().lower()
    for c in iho_files_eval.columns
]

print("\nIHO DATA")
print("-" * 80)
print("Path:", iho_path)
print("Shape:", iho_files_eval.shape)
print("Columns:", list(iho_files_eval.columns))


# ============================================================================
# 2. NORMALIZE IHO DATA
# ============================================================================

required_columns = ["logfile", "decom"]

missing = [
    c for c in required_columns
    if c not in iho_files_eval.columns
]

if missing:
    raise RuntimeError(
        f"IHO dataframe is missing required columns: {missing}"
    )

iho_files_eval["logfile"] = (
    iho_files_eval["logfile"]
    .astype(str)
    .str.strip()
)

iho_files_eval["decom"] = (
    iho_files_eval["decom"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Remove accidental empty rows
iho_files_eval = iho_files_eval[
    iho_files_eval["logfile"].notna()
].copy()

print("IHO files loaded:", len(iho_files_eval))


# ============================================================================
# 3. HELPER FUNCTIONS
# ============================================================================

def clean_value(x):
    if pd.isna(x):
        return ""
    return str(x).strip()


def normalize_id(x):
    """
    Normalize identifiers without destroying # characters.
    """
    return clean_value(x).upper()


def parse_list_field(value):
    """
    Convert fields such as:
        BY1EH BY1FJ BY4JF
        BY1EH,BY1FJ
        ['BY1EH', 'BY1FJ']
        NaN
    into a clean set.
    """

    if value is None or pd.isna(value):
        return set()

    s = str(value).strip()

    if not s or s.lower() in {
        "nan",
        "none",
        "null",
        "[]"
    }:
        return set()

    # Python-list representation
    if s.startswith("[") and s.endswith("]"):
        try:
            parsed = ast.literal_eval(s)

            if isinstance(parsed, (list, tuple, set)):
                return {
                    normalize_id(x)
                    for x in parsed
                    if normalize_id(x)
                }

        except Exception:
            pass

    # Split common separators
    parts = re.split(r"[\s,;|]+", s)

    return {
        normalize_id(x)
        for x in parts
        if normalize_id(x)
    }


def is_no_decision(answer):
    """
    Robustly determine whether the LLM's final decision is NO.

    We deliberately inspect the beginning / explicit decision rather than
    simply searching for the word 'no', because explanations may contain
    words such as 'not' or 'cannot'.
    """

    if not answer:
        return False

    text = str(answer).strip()

    # Strong patterns
    patterns = [
        r"\*\*decision\*\*\s*:\s*NO\b",
        r"\bdecision\s*:\s*NO\b",
        r"\bdecision\s*=\s*NO\b",
        r"^\s*NO\b",
        r"^\s*\*\*NO\*\*",
        r"\bCANNOT\s+BE\s+DECOMMISSIONED\b",
        r"\bCAN(?:NOT|'T)\s+BE\s+DECOMMISSIONED\b",
        r"\bNOT\s+DECOMMISSIONABLE\b",
    ]

    for pattern in patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return True

    return False


def is_yes_decision(answer):
    """
    Robust YES detector.
    """

    if not answer:
        return False

    text = str(answer).strip()

    patterns = [
        r"\*\*decision\*\*\s*:\s*YES\b",
        r"\bdecision\s*:\s*YES\b",
        r"\bdecision\s*=\s*YES\b",
        r"^\s*YES\b",
        r"^\s*\*\*YES\*\*",
        r"\bCAN\s+BE\s+DECOMMISSIONED\b",
    ]

    for pattern in patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return True

    return False


def extract_ids(text, known_ids):
    """
    Extract ONLY identifiers that are actually present in the known
    deterministic dataset.

    This fixes the previous bug where words such as:
        DEFINED
        NAME
        NOT

    were incorrectly classified as hallucinated files.
    """

    if not text:
        return set()

    text_upper = str(text).upper()

    found = set()

    # Longest first prevents partial matching.
    for item in sorted(
        {normalize_id(x) for x in known_ids},
        key=len,
        reverse=True
    ):
        # Escape IDs because some contain '#'
        pattern = r"(?<![A-Z0-9])" + re.escape(item) + r"(?![A-Z0-9])"

        if re.search(pattern, text_upper):
            found.add(item)

    return found


def calculate_metrics(expected, predicted):

    expected = set(expected)
    predicted = set(predicted)

    true_positive = len(expected & predicted)

    recall = (
        true_positive / len(expected)
        if expected
        else (1.0 if not predicted else 0.0)
    )

    precision = (
        true_positive / len(predicted)
        if predicted
        else (1.0 if not expected else 0.0)
    )

    exact = expected == predicted

    return recall, precision, exact


# ============================================================================
# 4. BUILD PROGRAM GROUND TRUTH FROM IHO
# ============================================================================

def build_program_ground_truth(df):

    programs = {}

    if "job" not in df.columns or "program" not in df.columns:
        print("\nWARNING: job/program columns unavailable.")
        return programs

    for _, row in df.iterrows():

        job = normalize_id(row.get("job"))
        program = normalize_id(row.get("program"))

        if not job or not program:
            continue

        key = (job, program)

        if key not in programs:
            programs[key] = {
                "job": job,
                "program": program,
                "decom": "YES",
                "blocking_files": set()
            }

        # Program/file relationship
        blocking = parse_list_field(
            row.get("blocking_files", "")
        )

        programs[key]["blocking_files"].update(blocking)

        # If row itself says NO, program is blocked
        row_decom = normalize_id(row.get("decom"))

        if row_decom == "NO":
            programs[key]["decom"] = "NO"

        # Some files may be represented through decom.1
        decom1 = normalize_id(row.get("decom.1"))

        if decom1 == "NO":
            programs[key]["decom"] = "NO"

    return programs


program_truth = build_program_ground_truth(
    iho_files_eval
)

print("\nPROGRAM GROUND TRUTH")
print("-" * 80)
print("Programs found:", len(program_truth))


# ============================================================================
# 5. GET BLOCKED PROGRAMS
# ============================================================================

blocked_programs = [
    value
    for value in program_truth.values()
    if value["decom"] == "NO"
]

print("Blocked programs:", len(blocked_programs))


# ============================================================================
# 6. KNOWN FILE IDS
# ============================================================================

known_files = set(
    normalize_id(x)
    for x in iho_files_eval["logfile"].tolist()
)

# Include blocking-file identifiers from IHO
if "blocking_files" in iho_files_eval.columns:

    for value in iho_files_eval["blocking_files"]:

        known_files.update(
            parse_list_field(value)
        )

print("Known file identifiers:", len(known_files))


# ============================================================================
# 7. CAPTURE TOOL CALLS FROM ask()
# ============================================================================

def capture_agent(question):

    """
    Runs your existing ask() function while capturing stdout.

    Your existing ask() prints lines like:

        -> tool call: get_decision({'logical_file': 'BP569'})

    We parse that output to determine what tool the agent selected.
    """

    if "ask" not in globals():
        raise RuntimeError(
            "ask() was not found. Run your main agent cell first."
        )

    buffer = io.StringIO()

    try:

        with contextlib.redirect_stdout(buffer):
            answer = ask(question)

        printed = buffer.getvalue()

        # Keep output visible
        print(printed, end="")

        # ------------------------------------------------------------
        # Extract tool names
        # ------------------------------------------------------------

        tool_matches = re.findall(
            r"->\s*tool call:\s*([A-Za-z_][A-Za-z0-9_]*)",
            printed
        )

        selected_tools = tool_matches

        final_tool = (
            selected_tools[-1]
            if selected_tools
            else None
        )

        return {
            "answer": str(answer) if answer is not None else "",
            "tool_calls": selected_tools,
            "final_tool": final_tool,
            "raw_trace": printed,
            "error": None
        }

    except Exception as e:

        printed = buffer.getvalue()

        print(printed, end="")

        return {
            "answer": "",
            "tool_calls": [],
            "final_tool": None,
            "raw_trace": printed,
            "error": str(e)
        }


# ============================================================================
# 8. EXPECTED TOOL BY ENTITY TYPE
# ============================================================================
#
# This is NOT hard-coding specific entities.
#
# We are testing whether the agent understands:
#
#   logical file  -> get_decision
#   job + program -> get_program_decision
#   cycle         -> get_cycle_summary
#
# ============================================================================

EXPECTED_FILE_TOOL = "get_decision"
EXPECTED_PROGRAM_TOOL = "get_program_decision"
EXPECTED_CYCLE_TOOL = "get_cycle_summary"


# ============================================================================
# 9. FILE-LEVEL EVALUATION
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL AGENT EVALUATION")
print("=" * 80)

blocked_files = sorted(
    normalize_id(x)
    for x in iho_files_eval[
        iho_files_eval["decom"] == "NO"
    ]["logfile"]
    .tolist()
)

file_results = []

for logical_file in blocked_files:

    print(f"\nTesting file: {logical_file}")

    # ------------------------------------------------------------
    # Ground truth
    # ------------------------------------------------------------

    row_matches = iho_files_eval[
        iho_files_eval["logfile"].str.upper()
        == logical_file
    ]

    true_decision = "NO"

    # ------------------------------------------------------------
    # Obtain true blocking jobs.
    #
    # We use existing get_decision() if your main notebook has it.
    # Otherwise we leave blockers unavailable rather than inventing them.
    # ------------------------------------------------------------

    true_blocking_jobs = set()

    if "get_decision" in globals():

        try:

            truth = get_decision(logical_file)

            if isinstance(truth, dict):

                true_decision = normalize_id(
                    truth.get("decom", "NO")
                )

                for dep in truth.get(
                    "blocking_dependents",
                    []
                ):

                    if isinstance(dep, dict):

                        job = normalize_id(
                            dep.get("job", "")
                        )

                        if job:
                            true_blocking_jobs.add(job)

        except Exception as e:

            print(
                "Ground-truth get_decision failed:",
                e
            )

    # ------------------------------------------------------------
    # Ask agent
    # ------------------------------------------------------------

    question = (
        f"Can the logical file {logical_file} "
        f"be decommissioned? "
        f"List every job that blocks it."
    )

    result = capture_agent(question)

    answer = result["answer"]

    # ------------------------------------------------------------
    # Decision
    # ------------------------------------------------------------

    predicted_no = is_no_decision(answer)

    predicted_decision = (
        "NO" if predicted_no
        else "YES" if is_yes_decision(answer)
        else "UNKNOWN"
    )

    decision_correct = (
        predicted_decision == true_decision
    )

    # ------------------------------------------------------------
    # Blocker extraction
    # ------------------------------------------------------------

    # Jobs are derived from ground truth where possible.
    known_jobs = set(true_blocking_jobs)

    # Also recognize all job IDs from the IHO dataset.
    if "job" in iho_files_eval.columns:

        for x in iho_files_eval["job"].dropna():

            value = normalize_id(x)

            if value:
                known_jobs.add(value)

    predicted_jobs = extract_ids(
        answer,
        known_jobs
    )

    recall, precision, exact = calculate_metrics(
        true_blocking_jobs,
        predicted_jobs
    )

    # ------------------------------------------------------------
    # Tool selection
    # ------------------------------------------------------------

    selected_tool = result["final_tool"]

    tool_correct = (
        selected_tool == EXPECTED_FILE_TOOL
    )

    file_results.append({
        "logical_file": logical_file,
        "true_decision": true_decision,
        "predicted_decision": predicted_decision,
        "decision_correct": decision_correct,

        "expected_tool": EXPECTED_FILE_TOOL,
        "selected_tool": selected_tool,
        "tool_selection_correct": tool_correct,

        "true_blockers": sorted(true_blocking_jobs),
        "predicted_blockers": sorted(predicted_jobs),

        "blocker_recall": recall,
        "blocker_precision": precision,
        "exact_match": exact,

        "answer": answer,
        "error": result["error"]
    })


file_eval_df = pd.DataFrame(file_results)


# ============================================================================
# 10. PROGRAM-LEVEL EVALUATION
# ============================================================================

print("\n")
print("=" * 80)
print("PROGRAM-LEVEL AGENT EVALUATION")
print("=" * 80)

program_results = []

for item in blocked_programs:

    job = item["job"]
    program = item["program"]

    print(f"\nTesting program: {job}/{program}")

    true_decision = item["decom"]

    true_files = set(
        item["blocking_files"]
    )

    question = (
        f"Can program {program} in job {job} "
        f"be decommissioned? "
        f"List every file blocking it."
    )

    result = capture_agent(question)

    answer = result["answer"]

    predicted_no = is_no_decision(answer)

    predicted_decision = (
        "NO" if predicted_no
        else "YES" if is_yes_decision(answer)
        else "UNKNOWN"
    )

    decision_correct = (
        predicted_decision == true_decision
    )

    # ------------------------------------------------------------
    # Only allow actual file IDs to be counted.
    # This fixes:
    #
    # [DEFINED, NAME, NOT]
    #
    # from your previous evaluation.
    # ------------------------------------------------------------

    predicted_files = extract_ids(
        answer,
        known_files
    )

    recall, precision, exact = calculate_metrics(
        true_files,
        predicted_files
    )

    selected_tool = result["final_tool"]

    tool_correct = (
        selected_tool == EXPECTED_PROGRAM_TOOL
    )

    program_results.append({
        "job": job,
        "program": program,

        "true_decision": true_decision,
        "predicted_decision": predicted_decision,
        "decision_correct": decision_correct,

        "expected_tool": EXPECTED_PROGRAM_TOOL,
        "selected_tool": selected_tool,
        "tool_selection_correct": tool_correct,

        "true_blockers": sorted(true_files),
        "predicted_blockers": sorted(predicted_files),

        "blocker_recall": recall,
        "blocker_precision": precision,
        "exact_match": exact,

        "answer": answer,
        "error": result["error"]
    })


program_eval_df = pd.DataFrame(
    program_results
)


# ============================================================================
# 11. CYCLE-LEVEL GROUND TRUTH
# ============================================================================

print("\n")
print("=" * 80)
print("CYCLE-LEVEL AGENT EVALUATION")
print("=" * 80)

cycle_name = "TS6604"

# Ground truth from IHO
cycle_true_files = set(
    normalize_id(x)
    for x in iho_files_eval[
        iho_files_eval["decom"] == "NO"
    ]["logfile"]
)

cycle_true_programs = set()

for item in blocked_programs:

    cycle_true_programs.add(
        f"{item['job']}/{item['program']}"
    )


cycle_true_decision = (
    "NO"
    if cycle_true_files or cycle_true_programs
    else "YES"
)

cycle_question = (
    f"Can the cycle {cycle_name} "
    f"be fully decommissioned? "
    f"List the blocking files and programs."
)

cycle_result = capture_agent(
    cycle_question
)

cycle_answer = cycle_result["answer"]

cycle_predicted_no = is_no_decision(
    cycle_answer
)

cycle_predicted_decision = (
    "NO"
    if cycle_predicted_no
    else "YES"
    if is_yes_decision(cycle_answer)
    else "UNKNOWN"
)

cycle_decision_correct = (
    cycle_predicted_decision
    == cycle_true_decision
)

# ------------------------------------------------------------
# Extract files
# ------------------------------------------------------------

cycle_predicted_files = extract_ids(
    cycle_answer,
    known_files
)

file_recall, file_precision, file_exact = (
    calculate_metrics(
        cycle_true_files,
        cycle_predicted_files
    )
)

# ------------------------------------------------------------
# Extract programs
# ------------------------------------------------------------

known_program_ids = set(
    cycle_true_programs
)

cycle_predicted_programs = set()

for program_id in known_program_ids:

    if program_id.upper() in cycle_answer.upper():

        cycle_predicted_programs.add(
            program_id
        )

program_recall, program_precision, program_exact = (
    calculate_metrics(
        cycle_true_programs,
        cycle_predicted_programs
    )
)

cycle_tool_correct = (
    cycle_result["final_tool"]
    == EXPECTED_CYCLE_TOOL
)


# ============================================================================
# 12. PRINT FILE RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL RESULTS")
print("=" * 80)

if len(file_eval_df):

    print(
        file_eval_df[
            [
                "logical_file",
                "true_decision",
                "predicted_decision",
                "decision_correct",
                "expected_tool",
                "selected_tool",
                "tool_selection_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_match"
            ]
        ].to_string(index=False)
    )

    print("\nFILE METRICS")

    print(
        "Decision accuracy:      "
        f"{file_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        "Tool-selection accuracy:"
        f" {file_eval_df['tool_selection_correct'].mean():.2%}"
    )

    print(
        "Average blocker recall:"
        f" {file_eval_df['blocker_recall'].mean():.2%}"
    )

    print(
        "Average blocker precision:"
        f" {file_eval_df['blocker_precision'].mean():.2%}"
    )

    print(
        "Exact blocker match:    "
        f"{file_eval_df['exact_match'].mean():.2%}"
    )


# ============================================================================
# 13. PRINT PROGRAM RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("PROGRAM-LEVEL RESULTS")
print("=" * 80)

if len(program_eval_df):

    print(
        program_eval_df[
            [
                "job",
                "program",
                "true_decision",
                "predicted_decision",
                "decision_correct",
                "expected_tool",
                "selected_tool",
                "tool_selection_correct",
                "blocker_recall",
                "blocker_precision",
                "exact_match"
            ]
        ].to_string(index=False)
    )

    print("\nPROGRAM METRICS")

    print(
        "Decision accuracy:      "
        f"{program_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        "Tool-selection accuracy:"
        f" {program_eval_df['tool_selection_correct'].mean():.2%}"
    )

    print(
        "Average blocker recall:"
        f" {program_eval_df['blocker_recall'].mean():.2%}"
    )

    print(
        "Average blocker precision:"
        f" {program_eval_df['blocker_precision'].mean():.2%}"
    )

    print(
        "Exact blocker match:    "
        f"{program_eval_df['exact_match'].mean():.2%}"
    )


# ============================================================================
# 14. PRINT CYCLE RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("CYCLE-LEVEL RESULTS")
print("=" * 80)

print(
    "Cycle:                  ",
    cycle_name
)

print(
    "True decision:          ",
    cycle_true_decision
)

print(
    "Predicted decision:     ",
    cycle_predicted_decision
)

print(
    "Decision correct:       ",
    cycle_decision_correct
)

print(
    "Expected tool:          ",
    EXPECTED_CYCLE_TOOL
)

print(
    "Selected tool:          ",
    cycle_result["final_tool"]
)

print(
    "Tool selection correct: ",
    cycle_tool_correct
)

print(
    "File recall:            ",
    f"{file_recall:.2%}"
)

print(
    "File precision:         ",
    f"{file_precision:.2%}"
)

print(
    "Exact file match:       ",
    file_exact
)

print(
    "Program recall:         ",
    f"{program_recall:.2%}"
)

print(
    "Program precision:      ",
    f"{program_precision:.2%}"
)

print(
    "Exact program match:    ",
    program_exact
)


# ============================================================================
# 15. OVERALL AGENT METRICS
# ============================================================================

all_tool_results = []

if len(file_eval_df):
    all_tool_results.extend(
        file_eval_df[
            "tool_selection_correct"
        ].tolist()
    )

if len(program_eval_df):
    all_tool_results.extend(
        program_eval_df[
            "tool_selection_correct"
        ].tolist()
    )

all_tool_results.append(
    cycle_tool_correct
)

print("\n")
print("=" * 80)
print("OVERALL AGENT METRICS")
print("=" * 80)

if all_tool_results:

    print(
        "Overall tool-selection accuracy:"
        f" {np.mean(all_tool_results):.2%}"
    )

print(
    "\nThis metric is important because it evaluates whether "
    "the LLM selected the correct deterministic tool for the "
    "entity/question type."
)


# ============================================================================
# 16. SAVE RESULTS
# ============================================================================

output_path = (
    "/kaggle/working/"
    "agent_evaluation_fixed.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    file_eval_df.to_excel(
        writer,
        sheet_name="File Evaluation",
        index=False
    )

    program_eval_df.to_excel(
        writer,
        sheet_name="Program Evaluation",
        index=False
    )

    pd.DataFrame([{
        "cycle": cycle_name,
        "true_decision": cycle_true_decision,
        "predicted_decision": cycle_predicted_decision,
        "decision_correct": cycle_decision_correct,
        "expected_tool": EXPECTED_CYCLE_TOOL,
        "selected_tool": cycle_result["final_tool"],
        "tool_selection_correct": cycle_tool_correct,
        "file_recall": file_recall,
        "file_precision": file_precision,
        "file_exact_match": file_exact,
        "program_recall": program_recall,
        "program_precision": program_precision,
        "program_exact_match": program_exact
    }]).to_excel(
        writer,
        sheet_name="Cycle Evaluation",
        index=False
    )


# ============================================================================
# 17. FINAL SUMMARY
# ============================================================================

print("\n")
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(
    f"Results saved to:\n{output_path}"
)

if len(file_eval_df):

    print(
        "\nFile decision accuracy: "
        f"{file_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        "File tool-selection accuracy: "
        f"{file_eval_df['tool_selection_correct'].mean():.2%}"
    )

if len(program_eval_df):

    print(
        "Program decision accuracy: "
        f"{program_eval_df['decision_correct'].mean():.2%}"
    )

    print(
        "Program tool-selection accuracy: "
        f"{program_eval_df['tool_selection_correct'].mean():.2%}"
    )

print(
    "Cycle decision accuracy: "
    f"{float(cycle_decision_correct):.2%}"
)

print(
    "Cycle tool-selection accuracy: "
    f"{float(cycle_tool_correct):.2%}"
)

print("=" * 80)

FIXED AGENT EVALUATION

IHO DATA
--------------------------------------------------------------------------------
Path: /kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx
Shape: (37, 8)
Columns: ['logfile', 'bestnr', 'decom', 'job', 'program', 'files', 'blocking files', 'decom.1']
IHO files loaded: 37

PROGRAM GROUND TRUTH
--------------------------------------------------------------------------------
Programs found: 33
Blocked programs: 14
Known file identifiers: 37


FILE-LEVEL AGENT EVALUATION

Testing file: BP569
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BP569'})
  round 2: calling Groq...
  -> final answer generated

Testing file: BY1EH
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY1EH'})
  round 2: calling Groq...
  -> final answer generated

Testing file: BY1FJ
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY1FJ'})
  round 2: calling Groq...
  -> final answer generated

Testing fil

In [9]:
# ============================================================================
# FIXED AGENT EVALUATION
# ============================================================================
# Run AFTER the main agent code.
#
# Metrics:
#   1. Tool-selection accuracy
#   2. Decision accuracy
#   3. Evidence recall
#   4. Evidence precision
#   5. Exact evidence match
#
# Ground truth comes ONLY from the deterministic tools.
# The LLM is evaluated on what tool it chose + what it answered.
# ============================================================================

import re
import pandas as pd
import numpy as np
from collections import Counter


# ============================================================================
# 1. BASIC HELPERS
# ============================================================================

def normalize_decision(value):
    """
    Normalize deterministic / LLM decisions to YES / NO.
    """
    if value is None:
        return None

    s = str(value).strip().upper()

    # Direct cases
    if s in {"YES", "Y", "TRUE"}:
        return "YES"

    if s in {"NO", "N", "FALSE"}:
        return "NO"

    # Handle phrases
    if re.search(r"\bCAN(?:NOT|'T)?\s+BE\s+DECOMMISSIONED\b", s):
        return "NO"

    if re.search(r"\bCANNOT\s+BE\s+DECOMMISSIONED\b", s):
        return "NO"

    if re.search(r"\bCANNOT\s+BE\s+DECOMMISSIONED\b", s):
        return "NO"

    if re.search(r"\bCAN\s+BE\s+DECOMMISSIONED\b", s):
        return "YES"

    return None


def extract_llm_decision(answer):
    """
    Extract YES/NO from the LLM answer.

    Priority is given to explicit 'Decision:' lines because
    the agent is instructed to produce a structured answer.
    """

    if not answer:
        return None

    text = str(answer)

    # First look for:
    # Decision: NO
    # Decision: YES
    m = re.search(
        r"\bdecision\s*:\s*\**\s*(YES|NO)\b",
        text,
        re.IGNORECASE
    )

    if m:
        return m.group(1).upper()

    # Fallback: look at beginning of answer
    m = re.search(
        r"^\s*\**\s*(YES|NO)\b",
        text,
        re.IGNORECASE
    )

    if m:
        return m.group(1).upper()

    return None


# ============================================================================
# 2. TOOL NAME EXTRACTION
# ============================================================================

def get_expected_tool(entity_type):
    """
    Expected tool for each entity type.
    """
    mapping = {
        "file": "get_decision",
        "program": "get_program_decision",
        "cycle": "get_cycle_summary"
    }

    return mapping[entity_type]


def get_actual_tool(trace):
    """
    Extract the first tool actually selected by the LLM.
    """

    if not trace:
        return None

    tool_calls = trace.get("tool_calls", [])

    if not tool_calls:
        return None

    first_call = tool_calls[0]

    return first_call.get("tool")


# ============================================================================
# 3. BLOCKER EXTRACTION
# ============================================================================

# Mainframe job IDs in this project look like:
#
# TS660420
# TS660451
# TD160425
#
# Require:
#   - exactly 8 characters
#   - starts with a letter
#   - contains digits
#
JOB_PATTERN = re.compile(
    r"\b[A-Z][A-Z0-9]{7}\b"
)


def extract_job_ids(text):
    """
    Extract actual job IDs from LLM answer.

    Avoids accidentally treating words such as:
        DEFINED
        NAME
        NOT
        BLOCKED

    as job IDs.
    """

    if not text:
        return set()

    candidates = set(
        x.upper()
        for x in JOB_PATTERN.findall(str(text).upper())
    )

    # Additional safety:
    # A valid job ID in this dataset contains at least 2 digits.
    valid = {
        x for x in candidates
        if sum(c.isdigit() for c in x) >= 2
    }

    return valid


def extract_file_ids(text, known_files):
    """
    Extract known logical-file identifiers from the answer.

    This is useful for program-level and cycle-level evaluation.
    """

    if not text:
        return set()

    text_upper = str(text).upper()

    found = set()

    for f in known_files:
        if str(f).upper() in text_upper:
            found.add(str(f))

    return found


# ============================================================================
# 4. SET METRICS
# ============================================================================

def calculate_set_metrics(actual, truth):
    """
    Calculate recall / precision / exact match.

    actual = evidence produced by LLM
    truth  = deterministic ground truth
    """

    actual = set(actual)
    truth = set(truth)

    if len(truth) == 0:
        recall = 1.0 if len(actual) == 0 else 0.0
    else:
        recall = len(actual & truth) / len(truth)

    if len(actual) == 0:
        precision = 1.0 if len(truth) == 0 else 0.0
    else:
        precision = len(actual & truth) / len(actual)

    exact = actual == truth

    return recall, precision, exact


# ============================================================================
# 5. SAFE DETERMINISTIC FILE GROUND TRUTH
# ============================================================================

def deterministic_file_truth(logical_file):
    """
    Get authoritative file-level ground truth.
    """

    result = get_decision(logical_file)

    decision = normalize_decision(
        result.get("decom")
    )

    blockers = set()

    # blocking_dependents is expected to contain dictionaries
    # such as:
    #
    # [{"job": "TS660420", ...}]
    #
    for item in result.get("blocking_dependents", []) or []:

        if isinstance(item, dict):

            job = item.get("job")

            if job:
                blockers.add(str(job).upper())

        elif isinstance(item, str):

            # If the deterministic tool directly returns a job ID
            if JOB_PATTERN.fullmatch(item.strip().upper()):
                blockers.add(item.strip().upper())

    return {
        "decision": decision,
        "blockers": blockers,
        "raw": result
    }


# ============================================================================
# 6. SAFE DETERMINISTIC PROGRAM GROUND TRUTH
# ============================================================================

def deterministic_program_truth(job, program):
    """
    Get authoritative program-level ground truth.
    """

    result = get_program_decision(
        job=job,
        program=program
    )

    decision = normalize_decision(
        result.get("decom")
    )

    blockers = set()

    # Program blockers are FILE identifiers.
    #
    # Different versions of the deterministic tool may use
    # slightly different field names, so check the common ones.
    possible_fields = [
        "blocking_files",
        "blocked_files",
        "files",
        "dependencies"
    ]

    for field in possible_fields:

        value = result.get(field)

        if value is None:
            continue

        if isinstance(value, list):

            for item in value:

                if isinstance(item, dict):

                    for key in [
                        "logical_file",
                        "file",
                        "filename",
                        "logicalFile"
                    ]:

                        if item.get(key):
                            blockers.add(
                                str(item[key]).upper()
                            )

                elif isinstance(item, str):

                    blockers.add(item.upper())

        elif isinstance(value, str):

            blockers.add(value.upper())

    return {
        "decision": decision,
        "blockers": blockers,
        "raw": result
    }


# ============================================================================
# 7. FILE-LEVEL EVALUATION
# ============================================================================

def evaluate_file_agents():
    print("\n")
    print("=" * 80)
    print("FILE-LEVEL AGENT EVALUATION")
    print("=" * 80)

    # Evaluate every file marked NO.
    # This keeps the evaluation focused on actual blocker reasoning.
    blocked_files = (
        iho_files[
            iho_files["decom"]
            .astype(str)
            .str.upper()
            .eq("NO")
        ]["logfile"]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    rows = []

    for logical_file in blocked_files:

        print(f"\nTesting file: {logical_file}")

        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        truth = deterministic_file_truth(logical_file)

        true_decision = truth["decision"]
        true_blockers = truth["blockers"]

        # ------------------------------------------------------------
        # Ask LLM
        # ------------------------------------------------------------

        question = (
            f"Can logical file {logical_file} be decommissioned? "
            f"List every job that blocks it."
        )

        answer = ask(question)

        # ------------------------------------------------------------
        # Agent trace
        # ------------------------------------------------------------

        trace = LAST_AGENT_TRACE.copy() if "LAST_AGENT_TRACE" in globals() else {}

        actual_tool = get_actual_tool(trace)

        expected_tool = get_expected_tool("file")

        tool_correct = (
            actual_tool == expected_tool
        )

        # ------------------------------------------------------------
        # Decision
        # ------------------------------------------------------------

        llm_decision = extract_llm_decision(answer)

        decision_correct = (
            llm_decision == true_decision
        )

        # ------------------------------------------------------------
        # Evidence
        # ------------------------------------------------------------

        mentioned_jobs = extract_job_ids(answer)

        recall, precision, exact = calculate_set_metrics(
            mentioned_jobs,
            true_blockers
        )

        missed = sorted(true_blockers - mentioned_jobs)

        hallucinated = sorted(
            mentioned_jobs - true_blockers
        )

        rows.append({
            "entity_type": "file",
            "logical_file": logical_file,
            "expected_tool": expected_tool,
            "actual_tool": actual_tool,
            "tool_selection_correct": tool_correct,

            "ground_truth_decision": true_decision,
            "llm_decision": llm_decision,
            "decision_correct": decision_correct,

            "true_blockers": sorted(true_blockers),
            "llm_blockers": sorted(mentioned_jobs),

            "blocker_recall": recall,
            "blocker_precision": precision,
            "exact_evidence_match": exact,

            "jobs_missed": missed,
            "jobs_hallucinated": hallucinated
        })

    return pd.DataFrame(rows)


# ============================================================================
# 8. PROGRAM LIST
# ============================================================================

def get_program_list():

    programs = []

    # The IHO contains job/program relationships.
    for _, row in iho_files.iterrows():

        job = str(row.get("job", "")).strip()
        program = str(row.get("program", "")).strip()

        if not job or not program:
            continue

        if job.lower() == "nan" or program.lower() == "nan":
            continue

        programs.append(
            (job, program)
        )

    # Remove duplicates while preserving order
    programs = list(dict.fromkeys(programs))

    return programs


# ============================================================================
# 9. PROGRAM-LEVEL EVALUATION
# ============================================================================

def evaluate_program_agents():

    print("\n")
    print("=" * 80)
    print("PROGRAM-LEVEL AGENT EVALUATION")
    print("=" * 80)

    programs = get_program_list()

    print(f"\nPrograms found: {len(programs)}")

    rows = []

    known_files = set(
        iho_files["logfile"]
        .dropna()
        .astype(str)
        .str.upper()
    )

    for job, program in programs:

        print(f"\nTesting program: {job}/{program}")

        # ------------------------------------------------------------
        # Ground truth
        # ------------------------------------------------------------

        truth = deterministic_program_truth(
            job,
            program
        )

        true_decision = truth["decision"]
        true_blockers = truth["blockers"]

        # ------------------------------------------------------------
        # Ask LLM
        # ------------------------------------------------------------

        question = (
            f"Can program {program} in job {job} "
            f"be decommissioned? "
            f"List every file that blocks it."
        )

        answer = ask(question)

        # ------------------------------------------------------------
        # Trace
        # ------------------------------------------------------------

        trace = LAST_AGENT_TRACE.copy() if "LAST_AGENT_TRACE" in globals() else {}

        actual_tool = get_actual_tool(trace)

        expected_tool = get_expected_tool("program")

        tool_correct = (
            actual_tool == expected_tool
        )

        # ------------------------------------------------------------
        # Decision
        # ------------------------------------------------------------

        llm_decision = extract_llm_decision(answer)

        decision_correct = (
            llm_decision == true_decision
        )

        # ------------------------------------------------------------
        # Evidence
        # ------------------------------------------------------------

        mentioned_files = extract_file_ids(
            answer,
            known_files
        )

        recall, precision, exact = calculate_set_metrics(
            mentioned_files,
            true_blockers
        )

        missed = sorted(
            true_blockers - mentioned_files
        )

        hallucinated = sorted(
            mentioned_files - true_blockers
        )

        rows.append({
            "entity_type": "program",
            "job": job,
            "program": program,

            "expected_tool": expected_tool,
            "actual_tool": actual_tool,
            "tool_selection_correct": tool_correct,

            "ground_truth_decision": true_decision,
            "llm_decision": llm_decision,
            "decision_correct": decision_correct,

            "true_blockers": sorted(true_blockers),
            "llm_blockers": sorted(mentioned_files),

            "blocker_recall": recall,
            "blocker_precision": precision,
            "exact_evidence_match": exact,

            "files_missed": missed,
            "files_hallucinated": hallucinated
        })

    return pd.DataFrame(rows)


# ============================================================================
# 10. CYCLE-LEVEL EVALUATION
# ============================================================================

def evaluate_cycle_agent(cycle="TS6604"):

    print("\n")
    print("=" * 80)
    print("CYCLE-LEVEL AGENT EVALUATION")
    print("=" * 80)

    # ------------------------------------------------------------
    # Ground truth
    # ------------------------------------------------------------

    cycle_truth = get_cycle_summary(cycle)

    true_decision = "YES"

    if (
        cycle_truth.get("files_no", 0) > 0
        or cycle_truth.get("programs_no", 0) > 0
    ):
        true_decision = "NO"

    true_files = set(
        str(x).upper()
        for x in cycle_truth.get("blocked_files", [])
    )

    true_programs = set(
        str(x).upper()
        for x in cycle_truth.get("blocked_programs", [])
    )

    # ------------------------------------------------------------
    # Ask LLM
    # ------------------------------------------------------------

    question = (
        f"Can the batch cycle {cycle} be fully decommissioned? "
        f"List all blocking files and programs."
    )

    answer = ask(question)

    # ------------------------------------------------------------
    # Trace
    # ------------------------------------------------------------

    trace = LAST_AGENT_TRACE.copy() if "LAST_AGENT_TRACE" in globals() else {}

    actual_tool = get_actual_tool(trace)

    expected_tool = get_expected_tool("cycle")

    tool_correct = (
        actual_tool == expected_tool
    )

    # ------------------------------------------------------------
    # Decision
    # ------------------------------------------------------------

    llm_decision = extract_llm_decision(answer)

    decision_correct = (
        llm_decision == true_decision
    )

    # ------------------------------------------------------------
    # Evidence
    # ------------------------------------------------------------

    mentioned_files = extract_file_ids(
        answer,
        set(
            iho_files["logfile"]
            .dropna()
            .astype(str)
            .str.upper()
        )
    )

    mentioned_jobs = extract_job_ids(answer)

    # Programs are represented as JOB/PROGRAM.
    mentioned_programs = set()

    for job_program in true_programs:

        job, program = job_program.split("/", 1)

        if job in mentioned_jobs and program in str(answer).upper():
            mentioned_programs.add(job_program)

    # More robust program detection:
    for job_program in true_programs:
        if job_program in str(answer).upper():
            mentioned_programs.add(job_program)

    # ------------------------------------------------------------
    # File metrics
    # ------------------------------------------------------------

    file_recall, file_precision, file_exact = calculate_set_metrics(
        mentioned_files,
        true_files
    )

    # ------------------------------------------------------------
    # Program metrics
    # ------------------------------------------------------------

    program_recall, program_precision, program_exact = calculate_set_metrics(
        mentioned_programs,
        true_programs
    )

    return {
        "cycle": cycle,

        "expected_tool": expected_tool,
        "actual_tool": actual_tool,
        "tool_selection_correct": tool_correct,

        "ground_truth_decision": true_decision,
        "llm_decision": llm_decision,
        "decision_correct": decision_correct,

        "true_files": sorted(true_files),
        "llm_files": sorted(mentioned_files),

        "file_recall": file_recall,
        "file_precision": file_precision,
        "file_exact_match": file_exact,

        "true_programs": sorted(true_programs),
        "llm_programs": sorted(mentioned_programs),

        "program_recall": program_recall,
        "program_precision": program_precision,
        "program_exact_match": program_exact,

        "answer": answer
    }


# ============================================================================
# 11. RUN EVALUATION
# ============================================================================

file_eval_df = evaluate_file_agents()

program_eval_df = evaluate_program_agents()

cycle_eval = evaluate_cycle_agent("TS6604")


# ============================================================================
# 12. PRINT FILE RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("FILE-LEVEL RESULTS")
print("=" * 80)

print(
    file_eval_df[
        [
            "logical_file",
            "actual_tool",
            "tool_selection_correct",
            "ground_truth_decision",
            "llm_decision",
            "decision_correct",
            "blocker_recall",
            "blocker_precision",
            "exact_evidence_match",
            "jobs_missed",
            "jobs_hallucinated"
        ]
    ].to_string(index=False)
)

print("\nFILE METRICS")
print("-" * 80)

print(
    f"Tool-selection accuracy: "
    f"{file_eval_df['tool_selection_correct'].mean() * 100:.2f}%"
)

print(
    f"Decision accuracy:       "
    f"{file_eval_df['decision_correct'].mean() * 100:.2f}%"
)

print(
    f"Average blocker recall:  "
    f"{file_eval_df['blocker_recall'].mean() * 100:.2f}%"
)

print(
    f"Average blocker precision:"
    f" {file_eval_df['blocker_precision'].mean() * 100:.2f}%"
)

print(
    f"Exact evidence match:    "
    f"{file_eval_df['exact_evidence_match'].mean() * 100:.2f}%"
)


# ============================================================================
# 13. PRINT PROGRAM RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("PROGRAM-LEVEL RESULTS")
print("=" * 80)

print(
    program_eval_df[
        [
            "job",
            "program",
            "actual_tool",
            "tool_selection_correct",
            "ground_truth_decision",
            "llm_decision",
            "decision_correct",
            "blocker_recall",
            "blocker_precision",
            "exact_evidence_match",
            "files_missed",
            "files_hallucinated"
        ]
    ].to_string(index=False)
)

print("\nPROGRAM METRICS")
print("-" * 80)

print(
    f"Tool-selection accuracy: "
    f"{program_eval_df['tool_selection_correct'].mean() * 100:.2f}%"
)

print(
    f"Decision accuracy:       "
    f"{program_eval_df['decision_correct'].mean() * 100:.2f}%"
)

print(
    f"Average blocker recall:  "
    f"{program_eval_df['blocker_recall'].mean() * 100:.2f}%"
)

print(
    f"Average blocker precision:"
    f" {program_eval_df['blocker_precision'].mean() * 100:.2f}%"
)

print(
    f"Exact evidence match:    "
    f"{program_eval_df['exact_evidence_match'].mean() * 100:.2f}%"
)


# ============================================================================
# 14. PRINT CYCLE RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("CYCLE-LEVEL RESULTS")
print("=" * 80)

print(
    f"Cycle:                    {cycle_eval['cycle']}"
)

print(
    f"Expected tool:            {cycle_eval['expected_tool']}"
)

print(
    f"Actual tool:              {cycle_eval['actual_tool']}"
)

print(
    f"Tool-selection correct:   {cycle_eval['tool_selection_correct']}"
)

print(
    f"Ground truth decision:    {cycle_eval['ground_truth_decision']}"
)

print(
    f"LLM decision:             {cycle_eval['llm_decision']}"
)

print(
    f"Decision correct:         {cycle_eval['decision_correct']}"
)

print("\nCycle file evidence:")
print(
    f"  Recall:                 "
    f"{cycle_eval['file_recall'] * 100:.2f}%"
)

print(
    f"  Precision:              "
    f"{cycle_eval['file_precision'] * 100:.2f}%"
)

print(
    f"  Exact match:            "
    f"{cycle_eval['file_exact_match']}"
)

print("\nCycle program evidence:")
print(
    f"  Recall:                 "
    f"{cycle_eval['program_recall'] * 100:.2f}%"
)

print(
    f"  Precision:              "
    f"{cycle_eval['program_precision'] * 100:.2f}%"
)

print(
    f"  Exact match:            "
    f"{cycle_eval['program_exact_match']}"
)


# ============================================================================
# 15. OVERALL TOOL-SELECTION SCORE
# ============================================================================

all_tool_results = []

if len(file_eval_df) > 0:
    all_tool_results.extend(
        file_eval_df["tool_selection_correct"].tolist()
    )

if len(program_eval_df) > 0:
    all_tool_results.extend(
        program_eval_df["tool_selection_correct"].tolist()
    )

all_tool_results.append(
    cycle_eval["tool_selection_correct"]
)

overall_tool_accuracy = (
    sum(all_tool_results) / len(all_tool_results)
    if all_tool_results
    else 0
)


print("\n")
print("=" * 80)
print("OVERALL AGENT METRICS")
print("=" * 80)

print(
    f"Overall tool-selection accuracy: "
    f"{overall_tool_accuracy * 100:.2f}%"
)

print(
    f"File decision accuracy:           "
    f"{file_eval_df['decision_correct'].mean() * 100:.2f}%"
)

print(
    f"File evidence recall:             "
    f"{file_eval_df['blocker_recall'].mean() * 100:.2f}%"
)

print(
    f"File evidence precision:          "
    f"{file_eval_df['blocker_precision'].mean() * 100:.2f}%"
)

print(
    f"File exact evidence match:        "
    f"{file_eval_df['exact_evidence_match'].mean() * 100:.2f}%"
)

print(
    f"Program decision accuracy:        "
    f"{program_eval_df['decision_correct'].mean() * 100:.2f}%"
)

print(
    f"Program evidence recall:          "
    f"{program_eval_df['blocker_recall'].mean() * 100:.2f}%"
)

print(
    f"Program evidence precision:       "
    f"{program_eval_df['blocker_precision'].mean() * 100:.2f}%"
)

print(
    f"Program exact evidence match:     "
    f"{program_eval_df['exact_evidence_match'].mean() * 100:.2f}%"
)


# ============================================================================
# 16. SAVE RESULTS
# ============================================================================

output_path = "/kaggle/working/agent_evaluation_final.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:

    file_eval_df.to_excel(
        writer,
        sheet_name="File Evaluation",
        index=False
    )

    program_eval_df.to_excel(
        writer,
        sheet_name="Program Evaluation",
        index=False
    )

    pd.DataFrame([cycle_eval]).to_excel(
        writer,
        sheet_name="Cycle Evaluation",
        index=False
    )

print("\n")
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(f"Saved to: {output_path}")



FILE-LEVEL AGENT EVALUATION

Testing file: BY4JE
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY4JE'})
  round 2: calling Groq...
  -> final answer generated

Testing file: BY1EH
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY1EH'})
  round 2: calling Groq...
  -> final answer generated

Testing file: BY1FJ
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY1FJ'})
  round 2: calling Groq...
  -> final answer generated

Testing file: BY4JF
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY4JF'})
  round 2: calling Groq...
  -> final answer generated

Testing file: VH14#1
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'VH14#1'})
  round 2: calling Groq...
  -> final answer generated

Testing file: BP569
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BP569'})
  round 2: calling Groq...
  -> final answer generated

Testing file:

In [13]:
# ============================================================================
# NATURAL-LANGUAGE TOOL-SELECTION + AGENT EVALUATION
# ============================================================================
#
# Evaluates:
#   1. Tool-selection accuracy
#   2. Decision accuracy
#   3. Evidence recall
#   4. Evidence precision
#   5. Exact evidence match
#
# PLUS:
#   6. Natural-language tool-selection evaluation
#
# Expected tools:
#   File    -> get_decision
#   Program -> get_program_decision
#   Cycle   -> get_cycle_summary
#
# IMPORTANT:
# This cell loads the IHO dataframe itself. It does NOT depend on an
# existing variable called iho_eval.
# ============================================================================

import os
import re
import json
import glob
import pandas as pd
from pathlib import Path


# ============================================================================
# 1. LOCATE IHO FILE
# ============================================================================

print("=" * 80)
print("LOCATING IHO DATA")
print("=" * 80)

IHO_CANDIDATES = [
    "/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx",
    "/kaggle/input/**/B11IH_combined.xlsx",
]

iho_path = None

for candidate in IHO_CANDIDATES:
    if "*" in candidate:
        matches = glob.glob(candidate, recursive=True)
        if matches:
            iho_path = matches[0]
            break
    elif os.path.exists(candidate):
        iho_path = candidate
        break

if iho_path is None:
    matches = glob.glob(
        "/kaggle/input/**/B11IH_combined.xlsx",
        recursive=True
    )

    if matches:
        iho_path = matches[0]

if iho_path is None:
    raise FileNotFoundError(
        "Could not find B11IH_combined.xlsx under /kaggle/input/"
    )

print("IHO file:")
print(iho_path)


# ============================================================================
# 2. LOAD IHO DATA
# ============================================================================

iho_eval = pd.read_excel(iho_path)

# Clean column names
iho_eval.columns = [
    str(c).strip().lower()
    for c in iho_eval.columns
]

print("\nIHO columns:")
print(list(iho_eval.columns))

# Handle possible naming differences
column_aliases = {
    "blocking_files": "blocking files",
    "blocking file": "blocking files",
    "blocking files ": "blocking files",
    "logical_file": "logfile",
    "logical file": "logfile",
}

for old, new in column_aliases.items():
    if old in iho_eval.columns and new not in iho_eval.columns:
        iho_eval = iho_eval.rename(columns={old: new})


# ============================================================================
# 3. VALIDATE REQUIRED COLUMNS
# ============================================================================

required_columns = [
    "logfile",
    "decom",
    "job",
    "program",
]

missing = [
    c for c in required_columns
    if c not in iho_eval.columns
]

if missing:
    raise RuntimeError(
        "IHO dataframe is missing required columns: "
        + str(missing)
        + "\nAvailable columns: "
        + str(list(iho_eval.columns))
    )

# Normalize important fields
iho_eval["logfile"] = (
    iho_eval["logfile"]
    .fillna("")
    .astype(str)
    .str.strip()
)

iho_eval["decom"] = (
    iho_eval["decom"]
    .fillna("")
    .astype(str)
    .str.upper()
    .str.strip()
)

iho_eval["job"] = (
    iho_eval["job"]
    .fillna("")
    .astype(str)
    .str.strip()
)

iho_eval["program"] = (
    iho_eval["program"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("\nIHO DATA LOADED")
print("Shape:", iho_eval.shape)
print("IHO files:", iho_eval["logfile"].nunique())


# ============================================================================
# 4. HELPER FUNCTIONS
# ============================================================================

def clean_text(value):
    if value is None:
        return ""

    if isinstance(value, float) and pd.isna(value):
        return ""

    return str(value).strip()


def normalize_decision(value):
    """
    Extract YES/NO from LLM response.
    """

    if value is None:
        return None

    text = str(value).upper().strip()

    # Prefer explicit decision patterns
    patterns = [
        r"\*\*DECISION\*\*\s*[:\-]?\s*(YES|NO)",
        r"DECISION\s*[:\-]?\s*(YES|NO)",
        r"\b(YES|NO)\b",
    ]

    for pattern in patterns:
        match = re.search(pattern, text)

        if match:
            return match.group(1)

    return None


def extract_identifiers(text, known_identifiers):
    """
    Extract known logical files / job / program identifiers from
    the generated answer.

    We intentionally only count identifiers that actually exist in
    the deterministic ground truth.
    """

    if text is None:
        return set()

    text = str(text)

    found = set()

    for identifier in known_identifiers:

        if not identifier:
            continue

        pattern = r"(?<![A-Za-z0-9])" + re.escape(identifier) + r"(?![A-Za-z0-9])"

        if re.search(pattern, text):
            found.add(identifier)

    return found


def safe_call_agent(question):
    """
    Uses the main agent function already created in the main notebook.

    Expected main function:
        run_agent(question)

    If your main code uses a different function name, change ONLY
    the function call below.
    """

    if "run_agent" not in globals():
        raise RuntimeError(
            "run_agent() was not found.\n"
            "Run the MAIN AGENT CODE cell first."
        )

    result = run_agent(question)

    # Support either:
    #   run_agent() -> string
    # or
    #   run_agent() -> dict containing final_answer / trace
    if isinstance(result, dict):

        answer = result.get(
            "final_answer",
            result.get("answer", "")
        )

        trace = result.get(
            "trace",
            result.get("agent_trace", {})
        )

        return str(answer), trace

    return str(result), globals().get(
        "LAST_AGENT_TRACE",
        {}
    )


# ============================================================================
# 5. BUILD FILE GROUND TRUTH
# ============================================================================

file_truth = {}

for _, row in iho_eval.iterrows():

    logical_file = clean_text(row["logfile"])

    if not logical_file:
        continue

    decision = clean_text(row["decom"]).upper()

    # Try to obtain blocking jobs from IHO columns
    blocking_jobs = set()

    for col in [
        "blocking files",
        "files",
        "job",
        "program",
    ]:

        if col not in iho_eval.columns:
            continue

        value = clean_text(row[col])

        if not value:
            continue

        # Job identifiers
        matches = re.findall(
            r"\bTS\d{6}\b",
            value.upper()
        )

        blocking_jobs.update(matches)

    file_truth[logical_file] = {
        "decision": decision,
        "blocking_jobs": sorted(blocking_jobs),
    }


# ============================================================================
# 6. BUILD PROGRAM GROUND TRUTH
# ============================================================================

program_truth = {}

for _, row in iho_eval.iterrows():

    job = clean_text(row["job"])
    program = clean_text(row["program"])

    if not job or not program:
        continue

    key = f"{job}/{program}"

    decision = clean_text(row["decom"]).upper()

    # Collect possible blocking files
    blocking_files = set()

    for col in [
        "blocking files",
        "files",
    ]:

        if col not in iho_eval.columns:
            continue

        value = clean_text(row[col])

        if not value:
            continue

        # Split common separators
        parts = re.split(
            r"[,;|\n]+",
            value
        )

        for part in parts:

            part = part.strip()

            if not part:
                continue

            blocking_files.add(part)

    program_truth[key] = {
        "decision": decision,
        "blocking_files": sorted(blocking_files),
    }


print("\nGROUND TRUTH")
print("-" * 80)
print("Programs found:", len(program_truth))
print("Files found:", len(file_truth))


# ============================================================================
# 7. NATURAL-LANGUAGE TOOL-SELECTION TEST SET
# ============================================================================
#
# IMPORTANT:
# These questions intentionally DO NOT explicitly say:
#
#   "file-level"
#   "program-level"
#   "cycle-level"
#
# The LLM has to infer the correct entity/tool from natural language.
# ============================================================================

NATURAL_LANGUAGE_TESTS = [

    # ---------------- FILE QUESTIONS ----------------

    {
        "id": "NL-F01",
        "entity": "file",
        "expected_tool": "get_decision",
        "question": (
            "Can BY4JE be retired safely? "
            "Check whether anything still prevents its removal."
        ),
    },

    {
        "id": "NL-F02",
        "entity": "file",
        "expected_tool": "get_decision",
        "question": (
            "I need to remove BP569 from the system. "
            "Is that logical file safe to decommission?"
        ),
    },

    {
        "id": "NL-F03",
        "entity": "file",
        "expected_tool": "get_decision",
        "question": (
            "Before deleting VH14#1, can you determine whether "
            "it is still required?"
        ),
    },

    {
        "id": "NL-F04",
        "entity": "file",
        "expected_tool": "get_decision",
        "question": (
            "What is the decommission status of JG0Q#1?"
        ),
    },

    # ---------------- PROGRAM QUESTIONS ----------------

    {
        "id": "NL-P01",
        "entity": "program",
        "expected_tool": "get_program_decision",
        "question": (
            "Can program S912 running under job TS660420 be retired?"
        ),
    },

    {
        "id": "NL-P02",
        "entity": "program",
        "expected_tool": "get_program_decision",
        "question": (
            "Is TS660407/JG0Q still needed, or can that program be "
            "decommissioned?"
        ),
    },

    {
        "id": "NL-P03",
        "entity": "program",
        "expected_tool": "get_program_decision",
        "question": (
            "Check whether K85F in job TS660412 can be removed."
        ),
    },

    {
        "id": "NL-P04",
        "entity": "program",
        "expected_tool": "get_program_decision",
        "question": (
            "Does job TS660405 still require program FQ79?"
        ),
    },

    # ---------------- CYCLE QUESTIONS ----------------

    {
        "id": "NL-C01",
        "entity": "cycle",
        "expected_tool": "get_cycle_summary",
        "question": (
            "Can the entire TS6604 batch process be retired, "
            "and what is stopping it if it cannot?"
        ),
    },

    {
        "id": "NL-C02",
        "entity": "cycle",
        "expected_tool": "get_cycle_summary",
        "question": (
            "I want to shut down TS6604 completely. "
            "Are there any remaining dependencies?"
        ),
    },

    {
        "id": "NL-C03",
        "entity": "cycle",
        "expected_tool": "get_cycle_summary",
        "question": (
            "Give me the overall decommission status of the TS6604 "
            "batch cycle."
        ),
    },

    {
        "id": "NL-C04",
        "entity": "cycle",
        "expected_tool": "get_cycle_summary",
        "question": (
            "What files and programs are preventing TS6604 from "
            "being fully retired?"
        ),
    },
]


# ============================================================================
# 8. EXTRACT ACTUAL TOOL FROM TRACE
# ============================================================================

def extract_tool_from_trace(trace):

    if not trace:
        return None

    tool_calls = trace.get("tool_calls", [])

    if not tool_calls:
        return None

    first_call = tool_calls[0]

    return (
        first_call.get("tool")
        or first_call.get("name")
    )


# ============================================================================
# 9. NATURAL-LANGUAGE TOOL-SELECTION EVALUATION
# ============================================================================

print("\n")
print("=" * 80)
print("NATURAL-LANGUAGE TOOL-SELECTION EVALUATION")
print("=" * 80)

nl_results = []

for test in NATURAL_LANGUAGE_TESTS:

    print("\n" + "-" * 80)
    print(test["id"])
    print("Question:", test["question"])
    print("Expected:", test["expected_tool"])

    try:

        answer, trace = safe_call_agent(
            test["question"]
        )

        actual_tool = extract_tool_from_trace(trace)

        correct = (
            actual_tool == test["expected_tool"]
        )

        print("Actual:", actual_tool)
        print("Correct:", correct)

        nl_results.append({
            "id": test["id"],
            "entity": test["entity"],
            "question": test["question"],
            "expected_tool": test["expected_tool"],
            "actual_tool": actual_tool,
            "tool_selection_correct": correct,
            "answer": answer,
        })

    except Exception as e:

        print("ERROR:", str(e))

        nl_results.append({
            "id": test["id"],
            "entity": test["entity"],
            "question": test["question"],
            "expected_tool": test["expected_tool"],
            "actual_tool": None,
            "tool_selection_correct": False,
            "answer": f"ERROR: {e}",
        })


nl_df = pd.DataFrame(nl_results)


# ============================================================================
# 10. NATURAL-LANGUAGE RESULTS
# ============================================================================

print("\n")
print("=" * 80)
print("NATURAL-LANGUAGE TOOL-SELECTION RESULTS")
print("=" * 80)

if len(nl_df) > 0:

    print(
        nl_df[
            [
                "id",
                "entity",
                "expected_tool",
                "actual_tool",
                "tool_selection_correct",
            ]
        ].to_string(index=False)
    )

    nl_accuracy = (
        nl_df["tool_selection_correct"]
        .mean()
        * 100
    )

else:

    nl_accuracy = 0.0


print("\nNatural-language tool-selection accuracy:")
print(f"{nl_accuracy:.2f}%")


# ============================================================================
# 11. ORIGINAL STRUCTURED FILE TOOL-SELECTION TEST
# ============================================================================

print("\n")
print("=" * 80)
print("STRUCTURED FILE TOOL-SELECTION EVALUATION")
print("=" * 80)

file_results = []

# Use only actual blocked / interesting files if available
file_test_set = list(file_truth.keys())

for logical_file in file_test_set:

    question = (
        f"Can logical file {logical_file} "
        f"be decommissioned?"
    )

    print(f"\nTesting file: {logical_file}")

    try:

        answer, trace = safe_call_agent(question)

        actual_tool = extract_tool_from_trace(trace)

        expected_tool = "get_decision"

        tool_correct = (
            actual_tool == expected_tool
        )

        llm_decision = normalize_decision(answer)

        ground_truth = file_truth[
            logical_file
        ]["decision"]

        decision_correct = (
            llm_decision == ground_truth
        )

        known_files = set(file_truth.keys())

        mentioned_files = extract_identifiers(
            answer,
            known_files
        )

        # A file-level answer should normally not invent
        # another known file.
        blockers = mentioned_files - {
            logical_file
        }

        file_results.append({
            "logical_file": logical_file,
            "actual_tool": actual_tool,
            "tool_selection_correct": tool_correct,
            "ground_truth_decision": ground_truth,
            "llm_decision": llm_decision,
            "decision_correct": decision_correct,
            "mentioned_identifiers": sorted(blockers),
            "answer": answer,
        })

    except Exception as e:

        file_results.append({
            "logical_file": logical_file,
            "actual_tool": None,
            "tool_selection_correct": False,
            "ground_truth_decision":
                file_truth[logical_file]["decision"],
            "llm_decision": None,
            "decision_correct": False,
            "mentioned_identifiers": [],
            "answer": f"ERROR: {e}",
        })


file_df = pd.DataFrame(file_results)


# ============================================================================
# 12. STRUCTURED PROGRAM TOOL-SELECTION TEST
# ============================================================================

print("\n")
print("=" * 80)
print("STRUCTURED PROGRAM TOOL-SELECTION EVALUATION")
print("=" * 80)

program_results = []

for key, truth in program_truth.items():

    job, program = key.split("/", 1)

    question = (
        f"Can program {program} in job {job} "
        f"be decommissioned?"
    )

    print(f"\nTesting program: {key}")

    try:

        answer, trace = safe_call_agent(question)

        actual_tool = extract_tool_from_trace(trace)

        expected_tool = "get_program_decision"

        tool_correct = (
            actual_tool == expected_tool
        )

        llm_decision = normalize_decision(answer)

        ground_truth = truth["decision"]

        decision_correct = (
            llm_decision == ground_truth
        )

        program_results.append({
            "job": job,
            "program": program,
            "actual_tool": actual_tool,
            "tool_selection_correct": tool_correct,
            "ground_truth_decision": ground_truth,
            "llm_decision": llm_decision,
            "decision_correct": decision_correct,
            "answer": answer,
        })

    except Exception as e:

        program_results.append({
            "job": job,
            "program": program,
            "actual_tool": None,
            "tool_selection_correct": False,
            "ground_truth_decision": truth["decision"],
            "llm_decision": None,
            "decision_correct": False,
            "answer": f"ERROR: {e}",
        })


program_df = pd.DataFrame(program_results)


# ============================================================================
# 13. CYCLE TOOL-SELECTION TEST
# ============================================================================

print("\n")
print("=" * 80)
print("CYCLE TOOL-SELECTION EVALUATION")
print("=" * 80)

cycle_question = (
    "Can the entire TS6604 batch cycle be fully "
    "decommissioned? List any blocking files and programs."
)

try:

    cycle_answer, cycle_trace = safe_call_agent(
        cycle_question
    )

    cycle_actual_tool = extract_tool_from_trace(
        cycle_trace
    )

    cycle_expected_tool = "get_cycle_summary"

    cycle_tool_correct = (
        cycle_actual_tool ==
        cycle_expected_tool
    )

    cycle_decision = normalize_decision(
        cycle_answer
    )

except Exception as e:

    cycle_answer = f"ERROR: {e}"
    cycle_trace = {}
    cycle_actual_tool = None
    cycle_expected_tool = "get_cycle_summary"
    cycle_tool_correct = False
    cycle_decision = None


print("Expected tool:", cycle_expected_tool)
print("Actual tool:", cycle_actual_tool)
print("Tool-selection correct:", cycle_tool_correct)


# ============================================================================
# 14. METRICS
# ============================================================================

print("\n")
print("=" * 80)
print("OVERALL TOOL-SELECTION METRICS")
print("=" * 80)

structured_tool_values = []

if len(file_df) > 0:
    structured_tool_values.extend(
        file_df[
            "tool_selection_correct"
        ].tolist()
    )

if len(program_df) > 0:
    structured_tool_values.extend(
        program_df[
            "tool_selection_correct"
        ].tolist()
    )

structured_tool_values.append(
    cycle_tool_correct
)

structured_tool_accuracy = (
    sum(structured_tool_values)
    / len(structured_tool_values)
    * 100
    if structured_tool_values
    else 0
)

print(
    f"Structured tool-selection accuracy: "
    f"{structured_tool_accuracy:.2f}%"
)

print(
    f"Natural-language tool-selection accuracy: "
    f"{nl_accuracy:.2f}%"
)


# ============================================================================
# 15. DECISION METRICS
# ============================================================================

if len(file_df) > 0:

    file_decision_accuracy = (
        file_df[
            "decision_correct"
        ].mean() * 100
    )

else:
    file_decision_accuracy = 0


if len(program_df) > 0:

    program_decision_accuracy = (
        program_df[
            "decision_correct"
        ].mean() * 100
    )

else:
    program_decision_accuracy = 0


print("\n")
print("=" * 80)
print("DECISION ACCURACY")
print("=" * 80)

print(
    f"File decision accuracy: "
    f"{file_decision_accuracy:.2f}%"
)

print(
    f"Program decision accuracy: "
    f"{program_decision_accuracy:.2f}%"
)

print(
    f"Cycle decision extracted: "
    f"{cycle_decision}"
)


# ============================================================================
# 16. SAVE RESULTS
# ============================================================================

output_path = (
    "/kaggle/working/"
    "agent_evaluation_with_nl_tool_selection.xlsx"
)

with pd.ExcelWriter(
    output_path,
    engine="openpyxl"
) as writer:

    nl_df.to_excel(
        writer,
        sheet_name="NL_Tool_Selection",
        index=False
    )

    file_df.to_excel(
        writer,
        sheet_name="File_Evaluation",
        index=False
    )

    program_df.to_excel(
        writer,
        sheet_name="Program_Evaluation",
        index=False
    )

    pd.DataFrame([
        {
            "metric":
                "Natural-language tool-selection accuracy",
            "value":
                nl_accuracy,
        },
        {
            "metric":
                "Structured tool-selection accuracy",
            "value":
                structured_tool_accuracy,
        },
        {
            "metric":
                "File decision accuracy",
            "value":
                file_decision_accuracy,
        },
        {
            "metric":
                "Program decision accuracy",
            "value":
                program_decision_accuracy,
        },
        {
            "metric":
                "Cycle tool-selection correct",
            "value":
                cycle_tool_correct,
        },
    ]).to_excel(
        writer,
        sheet_name="Summary",
        index=False
    )


# ============================================================================
# 17. FINAL SUMMARY
# ============================================================================

print("\n")
print("=" * 80)
print("EVALUATION COMPLETE")
print("=" * 80)

print(
    f"Natural-language tool-selection accuracy: "
    f"{nl_accuracy:.2f}%"
)

print(
    f"Structured tool-selection accuracy: "
    f"{structured_tool_accuracy:.2f}%"
)

print(
    f"File decision accuracy: "
    f"{file_decision_accuracy:.2f}%"
)

print(
    f"Program decision accuracy: "
    f"{program_decision_accuracy:.2f}%"
)

print("\nSaved to:")
print(output_path)

LOCATING IHO DATA
IHO file:
/kaggle/input/datasets/aburvan/b11ih-1/B11IH_combined.xlsx

IHO columns:
['logfile', 'bestnr', 'decom', 'job', 'program', 'files', 'blocking files', 'decom.1']

IHO DATA LOADED
Shape: (37, 8)
IHO files: 37

GROUND TRUTH
--------------------------------------------------------------------------------
Programs found: 33
Files found: 37


NATURAL-LANGUAGE TOOL-SELECTION EVALUATION

--------------------------------------------------------------------------------
NL-F01
Question: Can BY4JE be retired safely? Check whether anything still prevents its removal.
Expected: get_decision
  round 1: calling Groq...
  -> tool call: get_decision({'logical_file': 'BY4JE'})
  round 2: calling Groq...
  -> final answer generated
Actual: get_decision
Correct: True

--------------------------------------------------------------------------------
NL-F02
Question: I need to remove BP569 from the system. Is that logical file safe to decommission?
Expected: get_decision
  round 1: 